In [ ]:
#!/usr/bin/env python3
"""
train_v4_final_fixed_less_over_scale.py
=======================================
Mục tiêu:
- Giữ DL tương thích với RL Option C.
- Giảm false positive của error_rate để hạn chế RL scale dư.
- Vẫn giữ khả năng bắt spike latency tốt.

Các điểm chỉnh chính:
1. Giảm oversampling error spike.
2. Giảm trọng số error trong regression loss.
3. Giảm pos_weight BCE của error head.
4. Giữ output model: (final_out, err_logit)
"""

import os
import copy
import random
from pathlib import Path

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import Dataset, DataLoader, WeightedRandomSampler

from sklearn.model_selection import train_test_split


# ============================================================
# 1. CONFIG
# ============================================================

SEED = 42

random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)

if torch.cuda.is_available():
    torch.cuda.manual_seed_all(SEED)
    torch.backends.cudnn.benchmark = True

OUTPUT_DIR = Path("/kaggle/working/")
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

T_WINDOW = 4
BATCH_SIZE = 32
EPOCHS = 150
LR = 1e-3
WEIGHT_DECAY = 5e-4
EARLY_STOP_PATIENCE = 30

TARGET_SERVICES = sorted([
    "adservice",
    "cartservice",
    "checkoutservice",
    "currencyservice",
    "emailservice",
    "frontend",
    "paymentservice",
    "productcatalogservice",
    "recommendationservice",
    "shippingservice",
])

NUM_NODES = len(TARGET_SERVICES)

DATASET_SESSIONS = [
    {
        "name": "normal",
        "node": "/kaggle/input/datasets/kudo123a/dataset-v8/data_normal/nodes_data.csv",
        "edge": "/kaggle/input/datasets/kudo123a/dataset-v8/data_normal/edges_data.csv",
    },
    {
        "name": "anomaly_main",
        "node": "/kaggle/input/datasets/kudo123a/dataset-v8/data_anomaly/nodes_data.csv",
        "edge": "/kaggle/input/datasets/kudo123a/dataset-v8/data_anomaly/edges_data.csv",
    },
    {
        "name": "anomaly_edge",
        "node": "/kaggle/input/datasets/kudo123a/dataset-v8/data_deep_anomaly/nodes_data.csv",
        "edge": "/kaggle/input/datasets/kudo123a/dataset-v8/data_deep_anomaly/edges_data.csv",
    },
    {
        "name": "final_cascade",
        "node": "/kaggle/input/datasets/kudo123a/dataset-v8/data_final_cascade_anomaly/nodes_data.csv",
        "edge": "/kaggle/input/datasets/kudo123a/dataset-v8/data_final_cascade_anomaly/edges_data.csv",
    },
]

NODE_FEAT_COLS = [
    "cpu_usage_millicores_norm",
    "memory_usage_bytes_norm",
    "pod_replicas_count_norm",
    "allocated_cpu_quota_millicores_norm",
    "error_rate_ratio_norm",
    "request_per_second_norm",
    "latency_p50_seconds_norm",
    "latency_p95_seconds_norm",
    "latency_p99_seconds_norm",
]

EDGE_FEAT_COLS = [
    "network_latency_seconds_norm",
    "payload_size_bytes_norm",
    "edge_request_rate_rps_norm",
    "edge_error_rate_ratio_norm",
]

TARGET_NAMES = [
    "cpu",
    "rps",
    "error_rate",
    "latency_p99",
]

TARGET_COLS_NORM = [
    "target_cpu_usage_millicores_norm",
    "target_request_per_second_norm",
    "target_error_rate_ratio_norm",
    "target_latency_p99_seconds_norm",
]

TARGET_COLS_REAL = [
    "target_cpu_usage_millicores",
    "target_request_per_second",
    "target_error_rate_ratio",
    "target_latency_p99_seconds",
]

NODE_FEAT_DIM = len(NODE_FEAT_COLS)
EDGE_FEAT_DIM = len(EDGE_FEAT_COLS)
NUM_TARGETS = len(TARGET_NAMES)

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

print(f"[*] Device: {device}")
if torch.cuda.is_available():
    print(f"[*] GPU: {torch.cuda.get_device_name(0)}")


# ============================================================
# 2. DATA LOADING
# ============================================================

def _check_columns(df, cols, file_name):
    missing = [c for c in cols if c not in df.columns]
    if missing:
        raise ValueError(f"{file_name} thiếu cột: {missing}")


def load_session(node_file: str, edge_file: str, t_window: int = T_WINDOW):
    nodes_df = pd.read_csv(node_file)
    edges_df = pd.read_csv(edge_file)

    required_node_cols = (
        ["timestamp", "service"]
        + NODE_FEAT_COLS
        + TARGET_COLS_NORM
        + TARGET_COLS_REAL
    )
    required_edge_cols = (
        ["timestamp", "src_service", "dst_service"]
        + EDGE_FEAT_COLS
    )

    _check_columns(nodes_df, required_node_cols, node_file)
    _check_columns(edges_df, required_edge_cols, edge_file)

    nodes_df = nodes_df[nodes_df["service"].isin(TARGET_SERVICES)].copy()
    edges_df = edges_df[
        edges_df["src_service"].isin(TARGET_SERVICES)
        & edges_df["dst_service"].isin(TARGET_SERVICES)
    ].copy()

    service2id = {s: i for i, s in enumerate(TARGET_SERVICES)}

    all_ts = sorted(nodes_df["timestamp"].unique())
    ts2idx = {ts: idx for idx, ts in enumerate(all_ts)}
    num_ts = len(all_ts)

    if num_ts <= t_window:
        return None

    A = torch.zeros(NUM_NODES, NUM_NODES, dtype=torch.float32)
    E_raw = torch.zeros(num_ts, NUM_NODES, NUM_NODES, EDGE_FEAT_DIM, dtype=torch.float32)
    X_raw = torch.zeros(num_ts, NUM_NODES, NODE_FEAT_DIM, dtype=torch.float32)
    Y_norm = torch.zeros(num_ts, NUM_NODES, NUM_TARGETS, dtype=torch.float32)
    Y_real = torch.zeros(num_ts, NUM_NODES, NUM_TARGETS, dtype=torch.float32)

    for _, row in edges_df.iterrows():
        ts = row["timestamp"]
        if ts not in ts2idx:
            continue

        src = row["src_service"]
        dst = row["dst_service"]

        if src not in service2id or dst not in service2id:
            continue

        t = ts2idx[ts]
        i = service2id[src]
        j = service2id[dst]

        A[i, j] = 1.0
        E_raw[t, i, j] = torch.tensor(
            [float(row[c]) for c in EDGE_FEAT_COLS],
            dtype=torch.float32,
        )

    for i in range(NUM_NODES):
        A[i, i] = 1.0

    for _, row in nodes_df.iterrows():
        ts = row["timestamp"]
        service = row["service"]

        if ts not in ts2idx or service not in service2id:
            continue

        t = ts2idx[ts]
        i = service2id[service]

        X_raw[t, i] = torch.tensor(
            [float(row[c]) for c in NODE_FEAT_COLS],
            dtype=torch.float32,
        )

        Y_norm[t, i] = torch.tensor(
            [float(row[c]) for c in TARGET_COLS_NORM],
            dtype=torch.float32,
        )

        Y_real[t, i] = torch.tensor(
            [float(row[c]) for c in TARGET_COLS_REAL],
            dtype=torch.float32,
        )

    X_seq, E_seq, Yn_seq, Yr_seq = [], [], [], []

    for t in range(num_ts - t_window):
        X_seq.append(X_raw[t:t + t_window])
        E_seq.append(E_raw[t:t + t_window])
        Yn_seq.append(Y_norm[t + t_window])
        Yr_seq.append(Y_real[t + t_window])

    if len(X_seq) == 0:
        return None

    return {
        "X": torch.stack(X_seq),
        "E": torch.stack(E_seq),
        "Yn": torch.stack(Yn_seq),
        "Yr": torch.stack(Yr_seq),
        "A": A,
    }


class GraphDataset(Dataset):
    def __init__(self, X, E, Y):
        self.X = X
        self.E = E
        self.Y = Y

    def __len__(self):
        return len(self.X)

    def __getitem__(self, idx):
        return self.X[idx], self.E[idx], self.Y[idx]


# ============================================================
# 3. SPLIT / SAMPLER
# ============================================================

def make_window_labels(Yn, err_thr=0.01, lat_thr=0.05):
    max_err = Yn[:, :, 2].max(dim=1).values
    max_lat = Yn[:, :, 3].max(dim=1).values

    labels = torch.zeros(len(Yn), dtype=torch.long)

    lat_mask = max_lat > lat_thr
    err_mask = max_err > err_thr

    labels[lat_mask] = 1
    labels[err_mask] = 2
    labels[lat_mask & err_mask] = 3

    return labels.numpy()


def safe_stratified_split(Yn, test_size=0.2, random_state=SEED):
    n_samples = len(Yn)
    idx = np.arange(n_samples)

    labels = make_window_labels(Yn)

    unique, counts = np.unique(labels, return_counts=True)
    min_count = counts.min()

    can_stratify = (
        len(unique) > 1
        and min_count >= 2
        and int(n_samples * test_size) >= len(unique)
    )

    if can_stratify:
        train_idx, val_idx = train_test_split(
            idx,
            test_size=test_size,
            random_state=random_state,
            stratify=labels,
            shuffle=True,
        )
        split_type = "stratified"
    else:
        train_idx, val_idx = train_test_split(
            idx,
            test_size=test_size,
            random_state=random_state,
            shuffle=True,
        )
        split_type = "random"

    return train_idx, val_idx, labels, split_type


def make_balanced_sampler(Y_train):
    """
    Bản giảm false positive error_rate:
    - Error spike chỉ oversample nhẹ.
    - Latency spike vẫn giữ ưu tiên vì latency_p99 là tín hiệu autoscaling chính.
    """
    max_err = Y_train[:, :, 2].max(dim=1).values
    max_lat = Y_train[:, :, 3].max(dim=1).values

    weights = torch.ones(len(Y_train), dtype=torch.float32)

    weights[max_err > 0.01] *= 1.25
    weights[max_lat > 0.15] *= 1.80
    weights[(max_err > 0.01) & (max_lat > 0.15)] *= 1.20

    return WeightedRandomSampler(
        weights=weights.tolist(),
        num_samples=len(weights),
        replacement=True,
    )


# ============================================================
# 4. MODEL
# ============================================================

class EdgeAwareGATLayer(nn.Module):
    def __init__(self, in_node, in_edge, out_dim, dropout=0.2):
        super().__init__()

        self.W_q = nn.Linear(in_node, out_dim, bias=False)
        self.W_k = nn.Linear(in_node, out_dim, bias=False)
        self.W_v = nn.Linear(in_node, out_dim, bias=False)
        self.W_e = nn.Linear(in_edge, out_dim, bias=False)

        self.a = nn.Linear(3 * out_dim, 1, bias=False)
        self.dropout = nn.Dropout(dropout)

    def forward(self, X, E, A):
        B, N, _ = X.shape

        hq = self.W_q(X).unsqueeze(2).expand(-1, -1, N, -1)
        hk = self.W_k(X).unsqueeze(1).expand(-1, N, -1, -1)
        he = self.W_e(E)

        attn_in = torch.cat([hq, hk, he], dim=-1)
        e = F.leaky_relu(self.a(attn_in)).squeeze(-1)

        mask = torch.where(
            A.unsqueeze(0) > 0,
            e,
            torch.full_like(e, -9e15),
        )

        alpha = F.softmax(mask, dim=-1)
        alpha = self.dropout(alpha)

        out = torch.matmul(alpha, self.W_v(X))
        return F.elu(out)


class GAT_GRU_Model(nn.Module):
    EMB_DIM = 12

    def __init__(self, hidden_dim=48, dropout=0.25):
        super().__init__()

        self.node_emb = nn.Embedding(NUM_NODES, self.EMB_DIM)

        self.gat1 = EdgeAwareGATLayer(
            NODE_FEAT_DIM + self.EMB_DIM,
            EDGE_FEAT_DIM,
            hidden_dim,
            dropout,
        )
        self.ln1 = nn.LayerNorm(hidden_dim)

        self.gat2 = EdgeAwareGATLayer(
            hidden_dim,
            EDGE_FEAT_DIM,
            hidden_dim,
            dropout,
        )
        self.ln2 = nn.LayerNorm(hidden_dim)

        self.gru = nn.GRU(
            input_size=hidden_dim,
            hidden_size=hidden_dim,
            batch_first=True,
        )
        self.ln3 = nn.LayerNorm(hidden_dim)

        fc_in = hidden_dim + NODE_FEAT_DIM + self.EMB_DIM

        self.fc = nn.Sequential(
            nn.Linear(fc_in, 64),
            nn.LayerNorm(64),
            nn.ReLU(),
            nn.Dropout(dropout),

            nn.Linear(64, 32),
            nn.LayerNorm(32),
            nn.ReLU(),

            nn.Linear(32, NUM_TARGETS),
        )

        self.err_cls_head = nn.Sequential(
            nn.Linear(fc_in, 32),
            nn.ReLU(),
            nn.Linear(32, 1),
        )

        self.scale = nn.Parameter(torch.ones(NUM_NODES, NUM_TARGETS))
        self.bias = nn.Parameter(torch.zeros(NUM_NODES, NUM_TARGETS))

    def forward(self, X_seq, E_seq, A):
        B, T, N, _ = X_seq.shape
        Fe = E_seq.shape[-1]

        node_ids = torch.arange(N, device=X_seq.device)
        emb = self.node_emb(node_ids)
        emb = emb.view(1, 1, N, -1).expand(B, T, N, -1)

        X_in = torch.cat([X_seq, emb], dim=-1)

        X_flat = X_in.reshape(B * T, N, -1)
        E_flat = E_seq.reshape(B * T, N, N, Fe)

        h1 = self.gat1(X_flat, E_flat, A)
        h1 = self.ln1(h1)

        A2 = torch.matmul(A, A).clamp(0.0, 1.0)

        h2 = self.gat2(h1, E_flat, A2)
        h2 = self.ln2(h2 + h1)

        h2 = h2.reshape(B, T, N, -1)
        h2 = h2.permute(0, 2, 1, 3).reshape(B * N, T, -1)

        gru_out, _ = self.gru(h2)
        final = self.ln3(gru_out[:, -1, :])

        skip = X_seq[:, -1, :, :].reshape(B * N, -1)
        emb_skip = emb[:, -1, :, :].reshape(B * N, -1)

        combined = torch.cat([final, skip, emb_skip], dim=-1)

        reg_raw = self.fc(combined).view(B, N, NUM_TARGETS)
        reg_raw = reg_raw * self.scale.unsqueeze(0) + self.bias.unsqueeze(0)
        reg_out = torch.sigmoid(reg_raw)

        err_logit = self.err_cls_head(combined).view(B, N, 1)
        err_prob = torch.sigmoid(err_logit)

        final_out = reg_out.clone()
        final_out[:, :, 2] = err_prob.squeeze(-1) * reg_out[:, :, 2]

        return final_out, err_logit


# ============================================================
# 5. LOSS / METRICS
# ============================================================

def multi_task_loss(pred_tuple, target):
    """
    Bản giảm false positive error_rate:
    - Giảm base weight của error_rate.
    - Giảm spike multiplier của error_rate.
    - pos_weight dùng sqrt(neg/pos), max 8 thay vì max 30.
    - BCE coefficient giảm từ 0.20 xuống 0.15.
    """
    pred, err_logit = pred_tuple

    base_w = torch.tensor(
        [1.0, 1.0, 1.5, 3.0],
        dtype=torch.float32,
        device=pred.device,
    )

    err_label = (target[:, :, 2] > 0.01).float()
    lat_label = (target[:, :, 3] > 0.15).float()

    se = (pred - target) ** 2

    se[:, :, 0] *= base_w[0]
    se[:, :, 1] *= base_w[1]
    se[:, :, 2] *= base_w[2] * (1.0 + 2.0 * err_label)
    se[:, :, 3] *= base_w[3] * (1.0 + 3.0 * lat_label)

    reg_loss = se.mean()

    err_label_bce = err_label.unsqueeze(-1)

    pos_count = err_label_bce.sum()
    neg_count = err_label_bce.numel() - pos_count

    raw_ratio = neg_count / (pos_count + 1e-6)

    pos_weight = torch.clamp(
        torch.sqrt(raw_ratio),
        min=1.0,
        max=8.0,
    )

    bce_loss = F.binary_cross_entropy_with_logits(
        err_logit,
        err_label_bce,
        pos_weight=pos_weight.detach(),
    )

    return reg_loss + 0.15 * bce_loss


def smape(pred, true, eps=1e-8):
    return (
        2.0
        * torch.abs(pred - true)
        / (torch.abs(pred) + torch.abs(true) + eps)
    ).mean() * 100.0


def inspect_target_distribution(Y, name="Train"):
    y = Y.reshape(-1, NUM_TARGETS).cpu().numpy()

    print(f"\n=== {name} target distribution ===")

    for i, target_name in enumerate(TARGET_NAMES):
        vals = y[:, i]
        print(
            f"{target_name:12s} "
            f"min={vals.min():.5f} "
            f"p50={np.percentile(vals, 50):.5f} "
            f"p90={np.percentile(vals, 90):.5f} "
            f"p95={np.percentile(vals, 95):.5f} "
            f"p99={np.percentile(vals, 99):.5f} "
            f"max={vals.max():.5f} "
            f"nonzero={(vals > 1e-6).mean() * 100:.2f}%"
        )


def predict_in_batches(model, X, E, A, batch_size=BATCH_SIZE):
    model.eval()

    preds = []
    logits = []

    with torch.no_grad():
        for start in range(0, len(X), batch_size):
            end = start + batch_size

            final_out, err_logit = model(
                X[start:end].to(device),
                E[start:end].to(device),
                A.to(device),
            )

            preds.append(final_out.detach().cpu())
            logits.append(err_logit.detach().cpu())

    return torch.cat(preds, dim=0), torch.cat(logits, dim=0)


def evaluate(model, X_test, E_test, Y_test, Yr_test, A):
    final_out, err_logit = predict_in_batches(
        model,
        X_test,
        E_test,
        A,
        batch_size=BATCH_SIZE,
    )

    pn = final_out.reshape(-1, NUM_TARGETS)
    tn = Y_test.cpu().reshape(-1, NUM_TARGETS)

    print(f"\n{'=' * 65}")
    print("KẾT QUẢ ĐÁNH GIÁ NORMALIZED")
    print(f"{'=' * 65}")
    print(f" {'Target':<14} {'MAE_norm':>10} {'RMSE_norm':>10} {'sMAPE%':>9} {'R²':>8}")
    print("-" * 65)

    rows = []

    for k, name in enumerate(TARGET_NAMES):
        p = pn[:, k]
        t = tn[:, k]

        mae = torch.abs(p - t).mean().item()
        rmse = torch.sqrt(((p - t) ** 2).mean()).item()
        sp = smape(p, t).item()

        ss_res = ((t - p) ** 2).sum()
        ss_tot = ((t - t.mean()) ** 2).sum()
        r2 = (1.0 - ss_res / (ss_tot + 1e-8)).item()

        print(f" {name:<14} {mae:>10.4f} {rmse:>10.4f} {sp:>9.2f} {r2:>8.4f}")

        rows.append({
            "target": name,
            "mae_norm": mae,
            "rmse_norm": rmse,
            "smape": sp,
            "r2": r2,
        })

    pd.DataFrame(rows).to_csv(
        OUTPUT_DIR / "dl_eval_metrics_norm.csv",
        index=False,
    )


def evaluate_spikes(model, X_test, E_test, Y_test, A, err_thr=0.01, lat_thr=0.15):
    final_out, err_logit = predict_in_batches(
        model,
        X_test,
        E_test,
        A,
        batch_size=BATCH_SIZE,
    )

    pred = final_out.cpu()
    true = Y_test.cpu()

    print(f"\n{'=' * 65}")
    print("ĐÁNH GIÁ SPIKE / SLA VIOLATION")
    print(f"{'=' * 65}")

    rows = []

    for idx, name, thr in [
        (2, "error_rate", err_thr),
        (3, "latency_p99", lat_thr),
    ]:
        p_bin = pred[:, :, idx] > thr
        t_bin = true[:, :, idx] > thr

        tp = (p_bin & t_bin).sum().item()
        fp = (p_bin & ~t_bin).sum().item()
        fn = (~p_bin & t_bin).sum().item()
        tn = (~p_bin & ~t_bin).sum().item()

        precision = tp / (tp + fp + 1e-8)
        recall = tp / (tp + fn + 1e-8)
        f1 = 2.0 * precision * recall / (precision + recall + 1e-8)

        mae_all = torch.abs(pred[:, :, idx] - true[:, :, idx]).mean().item()

        if t_bin.any():
            mae_spike = torch.abs(
                pred[:, :, idx][t_bin] - true[:, :, idx][t_bin]
            ).mean().item()
        else:
            mae_spike = float("nan")

        print(
            f"{name} spike | thr={thr} | "
            f"P={precision:.3f} R={recall:.3f} F1={f1:.3f} | "
            f"MAE_all={mae_all:.4f} MAE_spike={mae_spike:.4f} | "
            f"TP={tp} FP={fp} FN={fn} TN={tn} | "
            f"true_spikes={t_bin.sum().item()} pred_spikes={p_bin.sum().item()}"
        )

        rows.append({
            "metric": name,
            "threshold": thr,
            "precision": precision,
            "recall": recall,
            "f1": f1,
            "mae_all": mae_all,
            "mae_spike": mae_spike,
            "tp": tp,
            "fp": fp,
            "fn": fn,
            "tn": tn,
            "true_spikes": int(t_bin.sum().item()),
            "pred_spikes": int(p_bin.sum().item()),
        })

    out_name = f"dl_spike_eval_err{err_thr}_lat{lat_thr}.csv".replace(".", "p")
    pd.DataFrame(rows).to_csv(
        OUTPUT_DIR / out_name,
        index=False,
    )


# ============================================================
# 6. TRAIN / VALIDATION
# ============================================================

def train_one_epoch(model, loader, optimizer, A_global):
    model.train()

    total_loss = 0.0
    n_batches = 0

    for bX, bE, bY in loader:
        bX = bX.to(device)
        bE = bE.to(device)
        bY = bY.to(device)

        optimizer.zero_grad(set_to_none=True)

        pred = model(bX, bE, A_global)
        loss = multi_task_loss(pred, bY)

        loss.backward()
        torch.nn.utils.clip_grad_norm_(model.parameters(), 1.0)

        optimizer.step()

        total_loss += loss.item()
        n_batches += 1

    return total_loss / max(n_batches, 1)


def validate_one_epoch(model, loader, A_global):
    model.eval()

    total_loss = 0.0
    n_batches = 0

    with torch.no_grad():
        for bX, bE, bY in loader:
            bX = bX.to(device)
            bE = bE.to(device)
            bY = bY.to(device)

            pred = model(bX, bE, A_global)
            loss = multi_task_loss(pred, bY)

            total_loss += loss.item()
            n_batches += 1

    return total_loss / max(n_batches, 1)


# ============================================================
# 7. MAIN
# ============================================================

if __name__ == "__main__":
    print("=" * 70)
    print("TRAIN V4 FINAL FIXED - LESS ERROR FALSE POSITIVE")
    print("=" * 70)

    train_X, train_E, train_Yn, train_Yr = [], [], [], []
    val_X, val_E, val_Yn, val_Yr = [], [], [], []

    A_global = torch.zeros(NUM_NODES, NUM_NODES, dtype=torch.float32)

    session_summary_rows = []

    for cfg in DATASET_SESSIONS:
        try:
            sess = load_session(
                node_file=cfg["node"],
                edge_file=cfg["edge"],
                t_window=T_WINDOW,
            )

            if sess is None:
                print(f"⚠ Skip empty session: {cfg['name']}")
                continue

            A_global = (A_global + sess["A"]).clamp(0.0, 1.0)

            n_samples = len(sess["X"])

            train_idx, val_idx, labels, split_type = safe_stratified_split(
                sess["Yn"],
                test_size=0.2,
                random_state=SEED,
            )

            train_X.append(sess["X"][train_idx])
            train_E.append(sess["E"][train_idx])
            train_Yn.append(sess["Yn"][train_idx])
            train_Yr.append(sess["Yr"][train_idx])

            val_X.append(sess["X"][val_idx])
            val_E.append(sess["E"][val_idx])
            val_Yn.append(sess["Yn"][val_idx])
            val_Yr.append(sess["Yr"][val_idx])

            unique, counts = np.unique(labels, return_counts=True)
            label_stats = dict(zip(unique.tolist(), counts.tolist()))

            print(
                f"✓ Loaded {cfg['name']}: {n_samples} windows "
                f"(Train={len(train_idx)}, Val={len(val_idx)}, Split={split_type})"
            )
            print(f"  Label distribution: {label_stats}")

            session_summary_rows.append({
                "session": cfg["name"],
                "windows": n_samples,
                "train": len(train_idx),
                "val": len(val_idx),
                "split_type": split_type,
                "label_distribution": str(label_stats),
            })

        except Exception as e:
            print(f"⚠ Lỗi load {cfg['name']}: {e}")

    if len(train_X) == 0:
        raise RuntimeError("Không load được session nào. Kiểm tra lại path dataset.")

    X_tr = torch.cat(train_X, dim=0)
    E_tr = torch.cat(train_E, dim=0)
    Yn_tr = torch.cat(train_Yn, dim=0)

    X_va = torch.cat(val_X, dim=0)
    E_va = torch.cat(val_E, dim=0)
    Yn_va = torch.cat(val_Yn, dim=0)
    Yr_va = torch.cat(val_Yr, dim=0)

    A_global = A_global.to(device)

    pd.DataFrame(session_summary_rows).to_csv(
        OUTPUT_DIR / "dl_session_split_summary.csv",
        index=False,
    )

    print(f"\n[*] Tổng dữ liệu: Train={len(X_tr)} windows | Val={len(X_va)} windows")
    print(f"[*] X_tr shape: {tuple(X_tr.shape)}")
    print(f"[*] E_tr shape: {tuple(E_tr.shape)}")
    print(f"[*] Yn_tr shape: {tuple(Yn_tr.shape)}")
    print(f"[*] A_global shape: {tuple(A_global.shape)}")

    inspect_target_distribution(Yn_tr, "Train")
    inspect_target_distribution(Yn_va, "Val")

    sampler = make_balanced_sampler(Yn_tr)

    train_loader = DataLoader(
        GraphDataset(X_tr, E_tr, Yn_tr),
        batch_size=BATCH_SIZE,
        sampler=sampler,
        num_workers=0,
        pin_memory=torch.cuda.is_available(),
    )

    val_loader = DataLoader(
        GraphDataset(X_va, E_va, Yn_va),
        batch_size=BATCH_SIZE,
        shuffle=False,
        num_workers=0,
        pin_memory=torch.cuda.is_available(),
    )

    model = GAT_GRU_Model(
        hidden_dim=48,
        dropout=0.25,
    ).to(device)

    optimizer = torch.optim.AdamW(
        model.parameters(),
        lr=LR,
        weight_decay=WEIGHT_DECAY,
    )

    scheduler = torch.optim.lr_scheduler.ReduceLROnPlateau(
        optimizer,
        mode="min",
        factor=0.5,
        patience=8,
        min_lr=1e-5,
    )

    best_val_loss = float("inf")
    best_epoch = -1
    best_weights = copy.deepcopy(model.state_dict())

    patience_counter = 0

    tr_losses = []
    va_losses = []

    print("\n[*] Bắt đầu train from scratch...")

    for epoch in range(EPOCHS):
        avg_tr_loss = train_one_epoch(
            model=model,
            loader=train_loader,
            optimizer=optimizer,
            A_global=A_global,
        )

        avg_va_loss = validate_one_epoch(
            model=model,
            loader=val_loader,
            A_global=A_global,
        )

        scheduler.step(avg_va_loss)

        tr_losses.append(avg_tr_loss)
        va_losses.append(avg_va_loss)

        if avg_va_loss < best_val_loss:
            best_val_loss = avg_va_loss
            best_epoch = epoch
            best_weights = copy.deepcopy(model.state_dict())
            patience_counter = 0
        else:
            patience_counter += 1

        if epoch % 10 == 0 or epoch == EPOCHS - 1:
            print(
                f"Epoch {epoch:03d} | "
                f"Train Loss={avg_tr_loss:.6f} | "
                f"Val Loss={avg_va_loss:.6f} | "
                f"Best={best_val_loss:.6f} @ {best_epoch} | "
                f"LR={optimizer.param_groups[0]['lr']:.6f}"
            )

        if patience_counter >= EARLY_STOP_PATIENCE:
            print(f"[*] Early stopping tại epoch {epoch}")
            break

    model.load_state_dict(best_weights)

    print(f"\n[*] Best epoch: {best_epoch}")
    print(f"[*] Best val loss: {best_val_loss:.6f}")

    evaluate(
        model=model,
        X_test=X_va,
        E_test=E_va,
        Y_test=Yn_va,
        Yr_test=Yr_va,
        A=A_global,
    )

    evaluate_spikes(
        model=model,
        X_test=X_va,
        E_test=E_va,
        Y_test=Yn_va,
        A=A_global,
        err_thr=0.01,
        lat_thr=0.05,
    )

    evaluate_spikes(
        model=model,
        X_test=X_va,
        E_test=E_va,
        Y_test=Yn_va,
        A=A_global,
        err_thr=0.01,
        lat_thr=0.15,
    )

    save_path = OUTPUT_DIR / "gat_gru_final_v4.pt"

    torch.save(
        {
            "model_state": model.state_dict(),
            "num_nodes": NUM_NODES,
            "node_dim": NODE_FEAT_DIM,
            "edge_dim": EDGE_FEAT_DIM,
            "num_targets": NUM_TARGETS,
            "t_window": T_WINDOW,
            "target_services": TARGET_SERVICES,
            "node_feat_cols": NODE_FEAT_COLS,
            "edge_feat_cols": EDGE_FEAT_COLS,
            "target_cols_norm": TARGET_COLS_NORM,
            "target_cols_real": TARGET_COLS_REAL,
            "best_epoch": best_epoch,
            "best_val_loss": best_val_loss,
            "note": "less_error_false_positive_version",
        },
        save_path,
    )

    print(f"\n✅ Đã lưu model tại: {save_path}")

    history_df = pd.DataFrame({
        "epoch": np.arange(len(tr_losses)),
        "train_loss": tr_losses,
        "val_loss": va_losses,
    })

    history_path = OUTPUT_DIR / "dl_training_history.csv"
    history_df.to_csv(history_path, index=False)

    plt.figure(figsize=(10, 5))
    plt.plot(tr_losses, label="Train Loss")
    plt.plot(va_losses, label="Val Loss")

    if len(va_losses) > 0:
        plt.axvline(
            x=int(np.argmin(va_losses)),
            linestyle="--",
            label="Best Model",
        )

    plt.title("GAT-GRU Final Learning Curve")
    plt.xlabel("Epoch")
    plt.ylabel("Loss")
    plt.legend()
    plt.grid(True, alpha=0.3)
    plt.tight_layout()

    curve_path = OUTPUT_DIR / "dl_learning_curve.png"
    plt.savefig(curve_path, dpi=150)
    plt.show()

    print(f"✅ Đã lưu training history: {history_path}")
    print(f"✅ Đã lưu biểu đồ loss: {curve_path}")

In [ ]:
# ============================================================
# POST-TRAINING VISUALIZATION FOR DL MODEL - FIXED VERSION
# Compatible with new DL model: return (final_out, err_logit)
# ============================================================

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import torch

# ------------------------------------------------------------
# 1. Generate predictions on validation set by batch
# ------------------------------------------------------------

def predict_validation_in_batches(model, X, E, A, batch_size=64):
    model.eval()

    preds = []
    logits = []

    with torch.no_grad():
        for start in range(0, len(X), batch_size):
            end = start + batch_size

            final_out, err_logit = model(
                X[start:end].to(device),
                E[start:end].to(device),
                A.to(device),
            )

            preds.append(final_out.detach().cpu())
            logits.append(err_logit.detach().cpu())

    pred_va = torch.cat(preds, dim=0)
    err_logit_va = torch.cat(logits, dim=0)
    err_prob_va = torch.sigmoid(err_logit_va)

    return pred_va, err_logit_va, err_prob_va


pred_va, err_logit_va, err_prob_va = predict_validation_in_batches(
    model=model,
    X=X_va,
    E=E_va,
    A=A_global,
    batch_size=64,
)

true_va = Yn_va.detach().cpu()

pred_flat = pred_va.reshape(-1, NUM_TARGETS).numpy()
true_flat = true_va.reshape(-1, NUM_TARGETS).numpy()

# ------------------------------------------------------------
# 2. Save training history
# ------------------------------------------------------------

history_df = pd.DataFrame({
    "epoch": np.arange(len(tr_losses)),
    "train_loss": tr_losses,
    "validation_loss": va_losses,
})

history_path = OUTPUT_DIR / "dl_training_history.csv"
history_df.to_csv(history_path, index=False)

print(f"Saved training history: {history_path}")

# ------------------------------------------------------------
# 3. Learning curve
# ------------------------------------------------------------

plt.figure(figsize=(10, 5))
plt.plot(history_df["epoch"], history_df["train_loss"], label="Training Loss")
plt.plot(history_df["epoch"], history_df["validation_loss"], label="Validation Loss")

best_epoch = int(np.argmin(va_losses))

plt.axvline(
    x=best_epoch,
    linestyle="--",
    label=f"Best Epoch: {best_epoch}",
)

plt.title("GAT-GRU Learning Curve")
plt.xlabel("Epoch")
plt.ylabel("Loss")
plt.grid(True, alpha=0.3)
plt.legend()
plt.tight_layout()

save_path = OUTPUT_DIR / "dl_learning_curve_final.png"
plt.savefig(save_path, dpi=150)
plt.show()

print(f"Saved: {save_path}")

# ------------------------------------------------------------
# 4. Regression metrics by target
# ------------------------------------------------------------

metric_rows = []

for i, target_name in enumerate(TARGET_NAMES):
    pred_i = pred_flat[:, i]
    true_i = true_flat[:, i]

    mae = np.mean(np.abs(pred_i - true_i))
    rmse = np.sqrt(np.mean((pred_i - true_i) ** 2))

    ss_res = np.sum((true_i - pred_i) ** 2)
    ss_tot = np.sum((true_i - np.mean(true_i)) ** 2)
    r2 = 1.0 - ss_res / (ss_tot + 1e-8)

    smape_value = (
        2.0
        * np.mean(
            np.abs(pred_i - true_i)
            / (np.abs(pred_i) + np.abs(true_i) + 1e-8)
        )
        * 100.0
    )

    metric_rows.append({
        "target": target_name,
        "MAE": mae,
        "RMSE": rmse,
        "R2": r2,
        "sMAPE": smape_value,
    })

metrics_df = pd.DataFrame(metric_rows)

metrics_path = OUTPUT_DIR / "dl_regression_metrics.csv"
metrics_df.to_csv(metrics_path, index=False)

print("\n=== Regression Metrics ===")
print(metrics_df.to_string(index=False))
print(f"Saved: {metrics_path}")

# ------------------------------------------------------------
# 5. Metrics grouped in one figure
# ------------------------------------------------------------

metric_names = ["MAE", "RMSE", "R2", "sMAPE"]

fig, axes = plt.subplots(2, 2, figsize=(14, 9))
axes = axes.flatten()

for ax, metric_name in zip(axes, metric_names):
    ax.bar(metrics_df["target"], metrics_df[metric_name])
    ax.set_title(metric_name)
    ax.set_xlabel("Target")
    ax.set_ylabel(metric_name)
    ax.grid(axis="y", alpha=0.3)

    for tick in ax.get_xticklabels():
        tick.set_rotation(20)

fig.suptitle("Validation Metrics by Prediction Target", fontsize=14, fontweight="bold")
fig.tight_layout()

save_path = OUTPUT_DIR / "dl_metrics_by_target.png"
plt.savefig(save_path, dpi=150)
plt.show()

print(f"Saved: {save_path}")

# ------------------------------------------------------------
# 6. Prediction vs ground truth
# ------------------------------------------------------------

sample_limit = min(300, len(pred_va))

fig, axes = plt.subplots(2, 2, figsize=(16, 9))
axes = axes.flatten()

for i, target_name in enumerate(TARGET_NAMES):
    true_series = true_va[:sample_limit, :, i].mean(dim=1).numpy()
    pred_series = pred_va[:sample_limit, :, i].mean(dim=1).numpy()

    axes[i].plot(true_series, label="Ground Truth", linewidth=2)
    axes[i].plot(pred_series, label="Prediction", linewidth=2, alpha=0.8)
    axes[i].set_title(f"{target_name}: Prediction vs Ground Truth")
    axes[i].set_xlabel("Validation Window")
    axes[i].set_ylabel("Normalized Value")
    axes[i].grid(True, alpha=0.3)
    axes[i].legend()

fig.suptitle("Prediction vs Ground Truth Across Targets", fontsize=14, fontweight="bold")
fig.tight_layout()

save_path = OUTPUT_DIR / "dl_prediction_vs_ground_truth.png"
plt.savefig(save_path, dpi=150)
plt.show()

print(f"Saved: {save_path}")

# ------------------------------------------------------------
# 7. Absolute error distribution
# ------------------------------------------------------------

abs_error = np.abs(pred_flat - true_flat)

fig, axes = plt.subplots(2, 2, figsize=(14, 9))
axes = axes.flatten()

for i, target_name in enumerate(TARGET_NAMES):
    axes[i].hist(abs_error[:, i], bins=40)
    axes[i].set_title(f"{target_name}: Absolute Error Distribution")
    axes[i].set_xlabel("Absolute Error")
    axes[i].set_ylabel("Frequency")
    axes[i].grid(True, alpha=0.3)

fig.suptitle("Absolute Error Distribution by Target", fontsize=14, fontweight="bold")
fig.tight_layout()

save_path = OUTPUT_DIR / "dl_error_distribution.png"
plt.savefig(save_path, dpi=150)
plt.show()

print(f"Saved: {save_path}")

# ------------------------------------------------------------
# 8. Spike detection metrics
# ------------------------------------------------------------

def compute_spike_metrics(pred, true, target_idx, threshold):
    pred_bin = pred[:, :, target_idx] > threshold
    true_bin = true[:, :, target_idx] > threshold

    tp = (pred_bin & true_bin).sum().item()
    fp = (pred_bin & ~true_bin).sum().item()
    fn = (~pred_bin & true_bin).sum().item()
    tn = (~pred_bin & ~true_bin).sum().item()

    precision = tp / (tp + fp + 1e-8)
    recall = tp / (tp + fn + 1e-8)
    f1 = 2 * precision * recall / (precision + recall + 1e-8)

    mae_all = torch.abs(pred[:, :, target_idx] - true[:, :, target_idx]).mean().item()

    if true_bin.any():
        mae_spike = torch.abs(
            pred[:, :, target_idx][true_bin] - true[:, :, target_idx][true_bin]
        ).mean().item()
    else:
        mae_spike = float("nan")

    return {
        "Precision": precision,
        "Recall": recall,
        "F1": f1,
        "MAE_all": mae_all,
        "MAE_spike": mae_spike,
        "TP": tp,
        "FP": fp,
        "FN": fn,
        "TN": tn,
        "true_spikes": int(true_bin.sum().item()),
        "pred_spikes": int(pred_bin.sum().item()),
    }


spike_rows = []

for target_idx, target_name, threshold in [
    (2, "error_rate", 0.01),
    (3, "latency_p99_low", 0.05),
    (3, "latency_p99_high", 0.15),
]:
    result = compute_spike_metrics(
        pred=pred_va,
        true=true_va,
        target_idx=target_idx,
        threshold=threshold,
    )

    spike_rows.append({
        "target": target_name,
        "threshold": threshold,
        **result,
    })

spike_df = pd.DataFrame(spike_rows)

spike_path = OUTPUT_DIR / "dl_spike_metrics.csv"
spike_df.to_csv(spike_path, index=False)

print("\n=== Spike Detection Metrics ===")
print(spike_df.to_string(index=False))
print(f"Saved: {spike_path}")

# ------------------------------------------------------------
# 9. Spike detection chart
# ------------------------------------------------------------

x = np.arange(len(spike_df))
width = 0.25

fig, ax = plt.subplots(figsize=(12, 6))

ax.bar(x - width, spike_df["Precision"], width, label="Precision")
ax.bar(x, spike_df["Recall"], width, label="Recall")
ax.bar(x + width, spike_df["F1"], width, label="F1")

ax.set_title("Spike Detection Performance")
ax.set_xlabel("Spike Type")
ax.set_ylabel("Score")
ax.set_xticks(x)
ax.set_xticklabels(spike_df["target"], rotation=15)
ax.set_ylim(0.0, 1.05)
ax.grid(axis="y", alpha=0.3)
ax.legend()

fig.tight_layout()

save_path = OUTPUT_DIR / "dl_spike_detection_metrics.png"
plt.savefig(save_path, dpi=150)
plt.show()

print(f"Saved: {save_path}")

# ------------------------------------------------------------
# 10. Target distribution
# ------------------------------------------------------------

fig, axes = plt.subplots(2, 2, figsize=(14, 9))
axes = axes.flatten()

for i, target_name in enumerate(TARGET_NAMES):
    axes[i].hist(true_flat[:, i], bins=50)
    axes[i].set_title(f"{target_name}: Validation Target Distribution")
    axes[i].set_xlabel("Normalized Value")
    axes[i].set_ylabel("Frequency")
    axes[i].grid(True, alpha=0.3)

fig.suptitle("Validation Target Distribution by Target", fontsize=14, fontweight="bold")
fig.tight_layout()

save_path = OUTPUT_DIR / "dl_target_distribution.png"
plt.savefig(save_path, dpi=150)
plt.show()

print(f"Saved: {save_path}")

# ------------------------------------------------------------
# 11. Error classification head diagnostics
# ------------------------------------------------------------

err_true_bin = (true_va[:, :, 2] > 0.01)
err_cls_bin = (err_prob_va.squeeze(-1) > 0.5)

tp = (err_cls_bin & err_true_bin).sum().item()
fp = (err_cls_bin & ~err_true_bin).sum().item()
fn = (~err_cls_bin & err_true_bin).sum().item()
tn = (~err_cls_bin & ~err_true_bin).sum().item()

precision = tp / (tp + fp + 1e-8)
recall = tp / (tp + fn + 1e-8)
f1 = 2 * precision * recall / (precision + recall + 1e-8)

err_head_df = pd.DataFrame([{
    "threshold_prob": 0.5,
    "Precision": precision,
    "Recall": recall,
    "F1": f1,
    "TP": tp,
    "FP": fp,
    "FN": fn,
    "TN": tn,
    "true_error_spikes": int(err_true_bin.sum().item()),
    "pred_error_spikes": int(err_cls_bin.sum().item()),
}])

err_head_path = OUTPUT_DIR / "dl_error_head_metrics.csv"
err_head_df.to_csv(err_head_path, index=False)

print("\n=== Error Classification Head Metrics ===")
print(err_head_df.to_string(index=False))
print(f"Saved: {err_head_path}")

plt.figure(figsize=(8, 5))
plt.hist(err_prob_va.reshape(-1).numpy(), bins=50)
plt.title("Error Head Probability Distribution")
plt.xlabel("Predicted Error Probability")
plt.ylabel("Frequency")
plt.grid(True, alpha=0.3)
plt.tight_layout()

save_path = OUTPUT_DIR / "dl_error_head_probability_distribution.png"
plt.savefig(save_path, dpi=150)
plt.show()

print(f"Saved: {save_path}")

print("\nPost-training DL visualization completed.")

In [ ]:
# import re
# import copy
# import torch
# from pathlib import Path
# from torch.utils.data import DataLoader, WeightedRandomSampler

# # ============================================================
# # FINE-TUNE CONFIG
# # ============================================================

# MODEL_SEARCH_DIRS = [
#     Path("/kaggle/working"),
#     Path("/kaggle/input/base-model"),
# ]

# MODEL_PREFIX = "gat_gru_v"
# MODEL_SUFFIX = ".pt"

# FT_OUTPUT_DIR = Path("/kaggle/working")

# FT_T_WINDOW = 4
# FT_BATCH_SIZE = 32
# FT_WEIGHT_DECAY = 3e-3

# FT_STAGE1_EPOCHS = 10
# FT_STAGE1_LR = 2e-4

# FT_STAGE2_EPOCHS = 10
# FT_STAGE2_LR = 3e-5

# # cân bằng hơn, không để anomaly bị chìm
# NORMAL_KEEP_RATIO = 0.70
# ANOMALY_WEIGHT = 3.0
# LATENCY_ANOMALY_WEIGHT = 2.0
# ERROR_ANOMALY_WEIGHT = 4.0

# FT_NORMAL = {
#     "name": "normal",
#     "node": "/kaggle/input/datasets/kudo123a/dataset-v5/data_normal/nodes_data.csv",
#     "edge": "/kaggle/input/datasets/kudo123a/dataset-v5/data_normal/edges_data.csv",
# }

# FT_ANOMALIES = [
#     {
#         "name": "deep_anomaly",
#         "node": "/kaggle/input/datasets/kudo123a/dataset-v5/data_deep_anomaly/nodes_data.csv",
#         "edge": "/kaggle/input/datasets/kudo123a/dataset-v5/data_deep_anomaly/edges_data.csv",
#     },
#     {
#         "name": "edge_propagation",
#         "node": "/kaggle/input/datasets/kudo123a/dataset-v5/data_deep_anomaly-v2/nodes_data.csv",
#         "edge": "/kaggle/input/datasets/kudo123a/dataset-v5/data_deep_anomaly-v2/edges_data.csv",
#     },
#     {
#         "name": "final_cascade",
#         "node": "/kaggle/input/datasets/kudo123a/dataset-v5/data_final_cascade_anomaly/nodes_data.csv",
#         "edge": "/kaggle/input/datasets/kudo123a/dataset-v5/data_final_cascade_anomaly/edges_data.csv",
#     },
# ]

# device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
# print(f"Device: {device}")
# if torch.cuda.is_available():
#     print(f"GPU: {torch.cuda.get_device_name(0)}")


# # ============================================================
# # VERSION HELPERS
# # ============================================================

# def extract_model_version(path: Path, prefix: str = MODEL_PREFIX, suffix: str = MODEL_SUFFIX):
#     pattern = rf"^{re.escape(prefix)}(\d+){re.escape(suffix)}$"
#     m = re.match(pattern, path.name)
#     return int(m.group(1)) if m else None


# def find_all_versioned_models(search_dirs):
#     models = []

#     for d in search_dirs:
#         if not d.exists():
#             continue

#         for p in d.glob(f"{MODEL_PREFIX}*{MODEL_SUFFIX}"):
#             v = extract_model_version(p)
#             if v is not None:
#                 models.append((v, p))

#     return sorted(models, key=lambda x: x[0])


# def get_latest_compatible_model(search_dirs, expected_edge_dim=4):
#     models = find_all_versioned_models(search_dirs)

#     compatible = []

#     for v, p in models:
#         try:
#             ckpt = torch.load(p, map_location="cpu")
#             if ckpt.get("edge_dim") == expected_edge_dim:
#                 compatible.append((v, p))
#         except Exception:
#             pass

#     if not compatible:
#         raise FileNotFoundError("Không tìm thấy checkpoint compatible edge_dim=4.")

#     return compatible[-1]


# def get_next_model_version(search_dirs):
#     models = find_all_versioned_models(search_dirs)
#     if not models:
#         return 1
#     return max(v for v, _ in models) + 1


# # ============================================================
# # FREEZE HELPERS
# # ============================================================

# def freeze_gat_layers(model):
#     for p in model.gat1.parameters():
#         p.requires_grad = False
#     for p in model.gat2.parameters():
#         p.requires_grad = False
#     print("✓ Frozen GAT layers")


# def unfreeze_all_layers(model):
#     for p in model.parameters():
#         p.requires_grad = True
#     print("✓ Unfrozen all layers")


# def count_trainable_params(model):
#     return sum(p.numel() for p in model.parameters() if p.requires_grad)


# # ============================================================
# # DATA HELPERS
# # ============================================================

# def load_one_session(cfg):
#     print(f"\nLoading session: {cfg['name']}")

#     X, E, Yn, Yr, A = load_session(
#         cfg["node"],
#         cfg["edge"],
#         T_window=FT_T_WINDOW,
#     )

#     print(f"  windows: {len(X)}")

#     return {
#         "name": cfg["name"],
#         "X": X,
#         "E": E,
#         "Yn": Yn,
#         "Yr": Yr,
#         "A": A,
#     }


# def split_session(sess, val_ratio=0.25):
#     n = len(sess["X"])
#     cut = int(n * (1 - val_ratio))

#     return {
#         "train": {
#             "X": sess["X"][:cut],
#             "E": sess["E"][:cut],
#             "Yn": sess["Yn"][:cut],
#             "Yr": sess["Yr"][:cut],
#         },
#         "val": {
#             "X": sess["X"][cut:],
#             "E": sess["E"][cut:],
#             "Yn": sess["Yn"][cut:],
#             "Yr": sess["Yr"][cut:],
#         },
#     }


# def subsample_normal(train_normal, keep_ratio=0.70):
#     n = len(train_normal["X"])
#     keep_n = max(1, int(n * keep_ratio))

#     idx = torch.linspace(0, n - 1, keep_n).long()

#     return {
#         "X": train_normal["X"][idx],
#         "E": train_normal["E"][idx],
#         "Yn": train_normal["Yn"][idx],
#         "Yr": train_normal["Yr"][idx],
#     }


# def concat_parts(parts):
#     return {
#         "X": torch.cat([p["X"] for p in parts], dim=0),
#         "E": torch.cat([p["E"] for p in parts], dim=0),
#         "Yn": torch.cat([p["Yn"] for p in parts], dim=0),
#         "Yr": torch.cat([p["Yr"] for p in parts], dim=0),
#     }


# def build_balanced_finetune_splits(normal_sess, anomaly_sess_list):
#     normal_sp = split_session(normal_sess, val_ratio=0.20)

#     anomaly_splits = [
#         split_session(s, val_ratio=0.35)
#         for s in anomaly_sess_list
#     ]

#     normal_train_bal = subsample_normal(
#         normal_sp["train"],
#         keep_ratio=NORMAL_KEEP_RATIO,
#     )

#     train_parts = [normal_train_bal] + [sp["train"] for sp in anomaly_splits]
#     val_parts = [normal_sp["val"]] + [sp["val"] for sp in anomaly_splits]

#     train = concat_parts(train_parts)
#     val = concat_parts(val_parts)

#     # test lấy validation của anomaly cuối cùng để kiểm tra case mới nhất/cascade
#     test = anomaly_splits[-1]["val"]

#     print("\n" + "=" * 70)
#     print("BALANCED FINE-TUNE SPLIT")
#     print("=" * 70)
#     print(f"Normal train kept : {len(normal_train_bal['X'])}")
#     print(f"Train total       : {len(train['X'])}")
#     print(f"Val total         : {len(val['X'])}")
#     print(f"Test total        : {len(test['X'])}")
#     print("Anomaly sessions:")
#     for s, sp in zip(anomaly_sess_list, anomaly_splits):
#         print(f"  {s['name']:<20} train={len(sp['train']['X'])} val/test={len(sp['val']['X'])}")

#     return {
#         "X_train": train["X"],
#         "E_train": train["E"],
#         "Y_train": train["Yn"],
#         "Yr_train": train["Yr"],

#         "X_val": val["X"],
#         "E_val": val["E"],
#         "Y_val": val["Yn"],
#         "Yr_val": val["Yr"],

#         "X_test": test["X"],
#         "E_test": test["E"],
#         "Y_test": test["Yn"],
#         "Yr_test": test["Yr"],
#     }


# def make_balanced_weighted_sampler(Y_train):
#     """
#     Tăng trọng số cho:
#     - windows có error
#     - windows có latency cao
#     """

#     max_err = Y_train[:, :, 2].max(dim=1).values
#     max_lat = Y_train[:, :, 3].max(dim=1).values

#     weights = torch.ones(len(Y_train))

#     error_mask = max_err > 0.01
#     latency_mask = max_lat > 0.30

#     weights[latency_mask] *= LATENCY_ANOMALY_WEIGHT
#     weights[error_mask] *= ERROR_ANOMALY_WEIGHT

#     n_err = error_mask.sum().item()
#     n_lat = latency_mask.sum().item()

#     print("\n" + "=" * 70)
#     print("SAMPLER STATS")
#     print("=" * 70)
#     print(f"Error anomaly windows   : {n_err}/{len(Y_train)}")
#     print(f"Latency anomaly windows : {n_lat}/{len(Y_train)}")
#     print(f"Weight range            : {weights.min().item():.2f} → {weights.max().item():.2f}")

#     return WeightedRandomSampler(
#         weights=weights.tolist(),
#         num_samples=len(weights),
#         replacement=True,
#     )


# # ============================================================
# # LOAD LATEST COMPATIBLE CHECKPOINT
# # ============================================================

# latest_version, latest_model_path = get_latest_compatible_model(
#     MODEL_SEARCH_DIRS,
#     expected_edge_dim=4,
# )

# print("=" * 70)
# print("FINE-TUNE: LOAD LATEST COMPATIBLE MODEL")
# print("=" * 70)
# print(f"Latest compatible version : v{latest_version}")
# print(f"Latest compatible path    : {latest_model_path}")

# ckpt = torch.load(latest_model_path, map_location=device)

# model = GAT_GRU_Model_v3(
#     num_nodes=ckpt["num_nodes"],
#     node_dim=ckpt["node_dim"],
#     edge_dim=ckpt["edge_dim"],
#     num_targets=ckpt["num_targets"],
#     hidden_dim=48,
#     dropout=0.25,
# ).to(device)

# model.load_state_dict(ckpt["model_state"])
# print("✓ Model loaded")


# # ============================================================
# # LOAD DATA
# # ============================================================

# print("\n" + "=" * 70)
# print("LOAD FINE-TUNE DATASET")
# print("=" * 70)

# normal_sess = load_one_session(FT_NORMAL)

# anomaly_sess_list = []
# for cfg in FT_ANOMALIES:
#     try:
#         anomaly_sess_list.append(load_one_session(cfg))
#     except Exception as e:
#         print(f"Skip {cfg['name']} because: {e}")

# if len(anomaly_sess_list) == 0:
#     raise RuntimeError("Không có anomaly session nào load được.")

# splits = build_balanced_finetune_splits(
#     normal_sess,
#     anomaly_sess_list,
# )

# A_all = normal_sess["A"].clone()
# for s in anomaly_sess_list:
#     A_all = (A_all + s["A"]).clamp(0.0, 1.0)

# A_ft = A_all.to(device)


# # ============================================================
# # TRAIN LOADER
# # ============================================================

# sampler = make_balanced_weighted_sampler(splits["Y_train"])

# train_loader_ft = DataLoader(
#     GraphDataset(
#         splits["X_train"],
#         splits["E_train"],
#         splits["Y_train"],
#     ),
#     batch_size=FT_BATCH_SIZE,
#     sampler=sampler,
# )

# print(f"\nFine-tune train batches: {len(train_loader_ft)}")


# # ============================================================
# # STAGE 1 — FREEZE GAT, TRAIN GRU + FC
# # ============================================================

# print("\n" + "=" * 70)
# print("STAGE 1: FREEZE GAT, TRAIN GRU + FC")
# print("=" * 70)

# freeze_gat_layers(model)

# LR = FT_STAGE1_LR
# WEIGHT_DECAY = FT_WEIGHT_DECAY

# print(f"Stage 1 LR       : {LR}")
# print(f"Stage 1 epochs   : {FT_STAGE1_EPOCHS}")
# print(f"Trainable params : {count_trainable_params(model):,}")

# model, best_val_s1, tr_l_s1, va_l_s1 = train(
#     model=model,
#     train_loader=train_loader_ft,
#     X_val=splits["X_val"],
#     E_val=splits["E_val"],
#     Y_val=splits["Y_val"],
#     A=A_ft,
#     epochs=FT_STAGE1_EPOCHS,
# )

# stage1_state = copy.deepcopy(model.state_dict())


# # ============================================================
# # STAGE 2 — UNFREEZE ALL, LOW LR
# # ============================================================

# print("\n" + "=" * 70)
# print("STAGE 2: UNFREEZE ALL, LOW LR")
# print("=" * 70)

# unfreeze_all_layers(model)

# LR = FT_STAGE2_LR
# WEIGHT_DECAY = FT_WEIGHT_DECAY

# print(f"Stage 2 LR       : {LR}")
# print(f"Stage 2 epochs   : {FT_STAGE2_EPOCHS}")
# print(f"Trainable params : {count_trainable_params(model):,}")

# model, best_val_s2, tr_l_s2, va_l_s2 = train(
#     model=model,
#     train_loader=train_loader_ft,
#     X_val=splits["X_val"],
#     E_val=splits["E_val"],
#     Y_val=splits["Y_val"],
#     A=A_ft,
#     epochs=FT_STAGE2_EPOCHS,
# )

# stage2_state = copy.deepcopy(model.state_dict())


# # ============================================================
# # SELECT BEST STAGE
# # ============================================================

# if best_val_s2 <= best_val_s1:
#     selected_stage = "stage2"
#     selected_state = stage2_state
#     best_val = best_val_s2
#     print("\n✓ Stage 2 selected")
# else:
#     selected_stage = "stage1"
#     selected_state = stage1_state
#     best_val = best_val_s1
#     print("\n✓ Stage 1 selected")

# model.load_state_dict(selected_state)


# # ============================================================
# # EVALUATE
# # ============================================================

# print("\n" + "=" * 70)
# print("EVALUATE FINE-TUNED MODEL")
# print("=" * 70)

# id2service = ckpt.get(
#     "id2service",
#     {i: s for i, s in enumerate(TARGET_SERVICES)}
# )

# evaluate(
#     model,
#     splits["X_test"],
#     splits["E_test"],
#     splits["Y_test"],
#     splits["Yr_test"],
#     id2service,
#     A_ft,
# )


# # ============================================================
# # SAVE NEXT VERSION
# # ============================================================

# next_version = get_next_model_version(MODEL_SEARCH_DIRS)
# save_path = FT_OUTPUT_DIR / f"{MODEL_PREFIX}{next_version}{MODEL_SUFFIX}"

# ft_ckpt = {
#     "model_state": model.state_dict(),

#     "num_nodes": ckpt["num_nodes"],
#     "node_dim": ckpt["node_dim"],
#     "edge_dim": ckpt["edge_dim"],
#     "num_targets": ckpt["num_targets"],

#     "id2service": id2service,
#     "target_names": ckpt.get("target_names", TARGET_NAMES),

#     "best_val_loss": best_val,
#     "best_val_stage1": best_val_s1,
#     "best_val_stage2": best_val_s2,

#     "train_losses": tr_l_s1 + tr_l_s2,
#     "val_losses": va_l_s1 + va_l_s2,

#     "base_model_version": latest_version,
#     "base_model_path": str(latest_model_path),

#     "model_version": next_version,
#     "model_name": f"{MODEL_PREFIX}{next_version}",

#     "t_window": FT_T_WINDOW,
#     "selected_stage": selected_stage,

#     "fine_tune_strategy": "balanced_multi_anomaly_T4_freeze_then_low_lr",
#     "normal_keep_ratio": NORMAL_KEEP_RATIO,
#     "stage1_lr": FT_STAGE1_LR,
#     "stage1_epochs": FT_STAGE1_EPOCHS,
#     "stage2_lr": FT_STAGE2_LR,
#     "stage2_epochs": FT_STAGE2_EPOCHS,

#     "sampler": {
#         "latency_anomaly_weight": LATENCY_ANOMALY_WEIGHT,
#         "error_anomaly_weight": ERROR_ANOMALY_WEIGHT,
#     },

#     "fine_tune_dataset": {
#         "normal": FT_NORMAL,
#         "anomalies": FT_ANOMALIES,
#     },

#     "architecture": "GAT_GRU_v3_2hop_sigmoid_edge4_T4_balanced_multianomaly",
# }

# torch.save(ft_ckpt, save_path)

# print("\n" + "=" * 70)
# print("FINE-TUNE COMPLETE")
# print("=" * 70)
# print(f"Loaded model      : v{latest_version}")
# print(f"Saved model       : v{next_version}")
# print(f"Selected stage    : {selected_stage}")
# print(f"Save path         : {save_path}")
# print(f"Best val stage 1  : {best_val_s1:.5f}")
# print(f"Best val stage 2  : {best_val_s2:.5f}")
# print(f"Best val overall  : {best_val:.5f}")

In [ ]:
# ── Extract test tensors (Cập nhật cho V4) ────────────────────

# Trong bản V4, tập Test/Val chính là các biến có hậu tố _va
X_test = X_va.to(device)
E_test = E_va.to(device)
Y_test = Yn_va.to(device)          # Dữ liệu Normalize
Y_test_real = Yr_va.to(device)     # Dữ liệu gốc (Real)

A = A_global.to(device)

print("X_test:", X_test.shape)
print("E_test:", E_test.shape)
print("Y_test:", Y_test.shape)

# Nếu bạn muốn chạy lại hàm evaluate để in ra bảng kết quả chi tiết:
# evaluate(model, X_test, E_test, Y_test, Y_test_real, A)
id2service = {i: name for i, name in enumerate(TARGET_SERVICES)}

In [ ]:
# ════════════════════════════════════════════════════════════════
#  BLOCK 5: EVALUATION METRICS (FIXED FOR GPU + V2 MODEL)
# ════════════════════════════════════════════════════════════════

print("\n" + "="*60)
print("BLOCK 5: EVALUATION METRICS (TEST SET)")
print("="*60)

# ── Ensure tensors on GPU ──────────────────────────────────────
A = A.to(device)

X_test = X_test.to(device)
E_test = E_test.to(device)
Y_test = Y_test.to(device)
Y_test_real = Y_test_real.to(device)

# ── Forward pass ───────────────────────────────────────────────
model.eval()
with torch.no_grad():
    pred_tuple = model(X_test, E_test, A)
    pred_norm, err_prob = pred_tuple   # unpack tuple
    # pred_norm: [S_test, N, NUM_TARGETS]
    # err_prob : [S_test, N, 1]


# ───────────────────────────────────────────────────────────────
# Metrics on normalized space [0,1]
# ───────────────────────────────────────────────────────────────

def compute_metrics(P_norm, T_norm, target_name):
    """
    Compute metrics on normalized space.
    
    Args:
        P_norm : predicted normalized values
        T_norm : ground truth normalized values
    """

    mae = torch.mean(
        torch.abs(P_norm - T_norm)
    ).item()

    rmse = torch.sqrt(
        torch.mean((P_norm - T_norm) ** 2)
    ).item()

    # Avoid divide-by-zero in MAPE
    nz = T_norm > 1e-6

    if nz.sum() > 0:
        mape = (
            torch.mean(
                torch.abs(
                    (P_norm[nz] - T_norm[nz]) / T_norm[nz]
                )
            ) * 100
        ).item()
    else:
        mape = 0.0

    # R² score
    ss_res = torch.sum((T_norm - P_norm) ** 2)

    ss_tot = torch.sum(
        (T_norm - T_norm.mean()) ** 2
    )

    r2 = (
        1 - ss_res / (ss_tot + 1e-8)
    ).item()

    # Pearson correlation
    if len(P_norm) > 1:
        corr = torch.corrcoef(
            torch.stack([
                P_norm.flatten(),
                T_norm.flatten()
            ])
        )[0, 1].item()
    else:
        corr = 0.0

    return {
        "name": target_name,
        "MAE": mae,
        "RMSE": rmse,
        "MAPE": mape,
        "R2": r2,
        "CORR": corr,
    }

# ───────────────────────────────────────────────────────────────
# Flatten tensors
# ───────────────────────────────────────────────────────────────

all_metrics = []

pred_norm_flat = pred_norm.reshape(-1, NUM_TARGETS)
true_norm_flat = Y_test.reshape(-1, NUM_TARGETS)

# ───────────────────────────────────────────────────────────────
# Per-target metrics
# ───────────────────────────────────────────────────────────────

for k, name in enumerate(TARGET_NAMES):

    P_n = pred_norm_flat[:, k]
    T_n = true_norm_flat[:, k]

    metrics = compute_metrics(
        P_n,
        T_n,
        name
    )

    all_metrics.append(metrics)

# ───────────────────────────────────────────────────────────────
# Print summary
# ───────────────────────────────────────────────────────────────

print(
    f"\n"
    f"{'Target':<15}"
    f"{'MAE':>10}"
    f"{'RMSE':>10}"
    f"{'MAPE%':>10}"
    f"{'R²':>10}"
    f"{'Corr':>10}"
)

print("-" * 65)

for m in all_metrics:

    print(
        f"  {m['name']:<13}"
        f"{m['MAE']:>10.4f}"
        f"{m['RMSE']:>10.4f}"
        f"{m['MAPE']:>10.2f}"
        f"{m['R2']:>10.4f}"
        f"{m['CORR']:>10.4f}"
    )

# ───────────────────────────────────────────────────────────────
# Overall metrics
# ───────────────────────────────────────────────────────────────

overall_mae = torch.mean(
    torch.abs(pred_norm - Y_test)
).item()

overall_rmse = torch.sqrt(
    torch.mean((pred_norm - Y_test) ** 2)
).item()

print("\n" + "-" * 65)
print("OVERALL")
print("-" * 65)

print(f"  Overall MAE  : {overall_mae:.4f}")
print(f"  Overall RMSE : {overall_rmse:.4f}")

# ───────────────────────────────────────────────────────────────
# Per-service × per-target MAE
# ───────────────────────────────────────────────────────────────

print(f"\n── PER-SERVICE MAE (normalized) ──")

print(f"  {'Service':<28}", end="")

for name in TARGET_NAMES:
    print(f" {name:>12}", end="")

print()

print("  " + "-" * (28 + 13 * NUM_TARGETS))

mae_per_svc_tgt = torch.abs(
    pred_norm - Y_test
).mean(dim=0)   # (N, 4)

for i in range(len(id2service)):

    print(f"  {id2service[i]:<28}", end="")

    for k in range(NUM_TARGETS):

        print(
            f" {mae_per_svc_tgt[i, k].item():>12.4f}",
            end=""
        )

    print()

# ───────────────────────────────────────────────────────────────
# Worst services analysis
# ───────────────────────────────────────────────────────────────

print(f"\n── WORST SERVICES (average MAE across targets) ──")

svc_mae = mae_per_svc_tgt.mean(dim=1)

sorted_idx = torch.argsort(
    svc_mae,
    descending=True
)

for rank, idx in enumerate(sorted_idx[:5]):

    idx = idx.item()

    print(
        f"  {rank+1}. "
        f"{id2service[idx]:<28} "
        f"MAE={svc_mae[idx].item():.4f}"
    )

# ───────────────────────────────────────────────────────────────
# Save predictions for scenario tests
# ───────────────────────────────────────────────────────────────

pred_norm_test = pred_norm.detach()

print("\n✓ Evaluation complete")

In [ ]:
# ════════════════════════════════════════════════════════════════
#  BLOCK 6: SCENARIO TESTS + ROLLOUT VALIDATION (ĐÃ FIX CHO V4)
# ════════════════════════════════════════════════════════════════

print("\n" + "="*70)
print("SCENARIO TESTS + TEMPORAL VALIDATION (V4)")
print("="*70)

# ───────────────────────────────────────────────────────────────
# SYNC BIẾN VÀ THIẾT BỊ
# ───────────────────────────────────────────────────────────────
A = A_global.to(device)

X_val = X_va.to(device)
E_val = E_va.to(device)
Y_val = Yn_va.to(device)

# Vì V4 gộp chung test vào val, ta dùng luôn tập Val cho các Scenario Test
X_test = X_va.to(device)
E_test = E_va.to(device)
Y_test = Yn_va.to(device)

id2service = {i: s for i, s in enumerate(TARGET_SERVICES)}
num_nodes = len(id2service)

# ───────────────────────────────────────────────────────────────
# FEATURE INDEX
# ───────────────────────────────────────────────────────────────
FEAT_IDX = {
    'cpu_norm': 0,
    'mem_norm': 1,
    'replicas_norm': 2,
    'alloc_cpu_norm': 3,
    'error_norm': 4,
    'rps_norm': 5,
    'p50_norm': 6,
    'p95_norm': 7,
    'p99_norm': 8,
}

# ───────────────────────────────────────────────────────────────
# BASE SAMPLE (Lấy mẫu đầu tiên của tập Test để làm gốc)
# ───────────────────────────────────────────────────────────────
frontend_idx = list(id2service.values()).index('frontend')

test_base = X_test[0].clone()   # (T, N, 9)
edge_base = E_test[0].clone()   # (T, N, N, 4)

# BASE PREDICTION
model.eval()
with torch.no_grad():
    pred_tuple = model(
        test_base.unsqueeze(0),
        edge_base.unsqueeze(0),
        A
    )
    final_out, err_prob = pred_tuple
    pred_base = final_out[0]   # (N, 4)

    # Cần định nghĩa pred_norm_test cho phần Overfit check
    pred_tuple_test = model(X_test, E_test, A)
    pred_norm_test, _ = pred_tuple_test


# ════════════════════════════════════════════════════════════════
#  SCENARIO 1 — CPU SPIKE TẠI FRONTEND
# ════════════════════════════════════════════════════════════════

print("\n" + "="*70)
print("SCENARIO TEST 1: CPU SPIKE @ FRONTEND")
print("="*70)

test_cpu = test_base.clone()

# Bơm thêm CPU vào frontend
test_cpu[:, frontend_idx, FEAT_IDX['cpu_norm']] += 1.5
test_cpu = torch.clamp(test_cpu, 0.0, 1.0) # Clamp tránh vượt range

with torch.no_grad():
    pred_tuple = model(test_cpu.unsqueeze(0), edge_base.unsqueeze(0), A)
    pred_cpu, _ = pred_tuple
    pred_cpu = pred_cpu[0]   # (N, 4)

print(f"\nSpike: frontend cpu_norm +1.5")
print(f"\n{'Service':<28}", end="")
for name in TARGET_NAMES:
    print(f"{name:>18}", end="")
print("\n" + "-" * (28 + 18 * NUM_TARGETS))

for i in range(num_nodes):
    svc = id2service[i]
    print(f"{svc:<28}", end="")
    for k in range(NUM_TARGETS):
        b = pred_base[i, k].item()
        s = pred_cpu[i, k].item()
        d = s - b
        mark = "↑" if d > 0.01 else ("↓" if d < -0.01 else "=")
        print(f"{b:.3f}→{s:.3f}({d:+.3f}){mark:>2}", end="   ")
    print()

print("\nExpected: frontend CPU spike → predicted cpu/rps should increase")

# ════════════════════════════════════════════════════════════════
#  SCENARIO 2 — ERROR SPIKE TOÀN CỤM
# ════════════════════════════════════════════════════════════════

print("\n" + "="*70)
print("SCENARIO TEST 2: ERROR RATE SPIKE")
print("="*70)

test_err = test_base.clone()

# Tăng error rate trên toàn bộ các node
test_err[:, :, FEAT_IDX['error_norm']] += 2.0
test_err = torch.clamp(test_err, 0.0, 1.0)

with torch.no_grad():
    pred_tuple = model(test_err.unsqueeze(0), edge_base.unsqueeze(0), A)
    pred_err, _ = pred_tuple
    pred_err = pred_err[0]   # (N, 4)

print("\nGlobal error_norm +2.0")
print(f"\n{'Service':<28} {'base_err':>12} {'pred_err':>12} {'Δerror':>12} {'Δlatency':>12}")
print("-" * 80)

for i in range(num_nodes):
    b_err, s_err = pred_base[i, 2].item(), pred_err[i, 2].item()
    b_lat, s_lat = pred_base[i, 3].item(), pred_err[i, 3].item()

    print(
        f"{id2service[i]:<28} "
        f"{b_err:>12.4f} {s_err:>12.4f} "
        f"{(s_err-b_err):>+12.4f} {(s_lat-b_lat):>+12.4f}"
    )

print("\nExpected: error spike → higher predicted error_rate & latency")

# ════════════════════════════════════════════════════════════════
#  SCENARIO 3 — ZERO TRAFFIC
# ════════════════════════════════════════════════════════════════

print("\n" + "="*70)
print("SCENARIO TEST 3: ZERO TRAFFIC")
print("="*70)

test_zero = test_base.clone()
test_zero[:, :, FEAT_IDX['cpu_norm']] = 0.0
test_zero[:, :, FEAT_IDX['rps_norm']] = 0.0
test_zero[:, :, FEAT_IDX['error_norm']] = 0.0
edge_zero = torch.zeros_like(edge_base) # Xóa traffic ở edge

with torch.no_grad():
    pred_tuple = model(test_zero.unsqueeze(0), edge_zero.unsqueeze(0), A)
    pred_zero, _ = pred_tuple
    pred_zero = pred_zero[0]   # (N, 4)

print(f"\n{'Service':<28} {'cpu_base':>10} {'cpu_zero':>10} {'rps_base':>10} {'rps_zero':>10}")
print("-" * 74)

for i in range(num_nodes):
    b_cpu, z_cpu = pred_base[i, 0].item(), pred_zero[i, 0].item()
    b_rps, z_rps = pred_base[i, 1].item(), pred_zero[i, 1].item()

    flag = "✓ lower" if z_cpu < b_cpu else "⚠ same/higher"
    print(f"{id2service[i]:<28} {b_cpu:>10.4f} {z_cpu:>10.4f} {b_rps:>10.4f} {z_rps:>10.4f} {flag}")

print("\nExpected: zero traffic → lower predicted cpu & rps")

# ════════════════════════════════════════════════════════════════
#  TEMPORAL VALIDATION / OVERFIT CHECK
# ════════════════════════════════════════════════════════════════

print("\n" + "="*70)
print("TEMPORAL ROLLOUT VALIDATION")
print("="*70)

with torch.no_grad():
    val_pred_tuple = model(X_val, E_val, A)
    val_pred_norm, _ = val_pred_tuple   # lấy regression output

val_mae_per_target = torch.abs(val_pred_norm - Y_val).mean(dim=(0, 1))

test_pred_tuple = model(X_test, E_test, A)
pred_norm_test, _ = test_pred_tuple

test_mae_per_target = torch.abs(pred_norm_test - Y_test).mean(dim=(0, 1))


print(f"\nVal samples : {len(X_val)}")
print(f"Test samples: {len(X_test)}")

print(f"\n{'Target':<18} {'Val MAE':>12} {'Test MAE':>12} {'Gap %':>12} {'Status':>12}")
print("-" * 72)

overfit = False

for k, name in enumerate(TARGET_NAMES):
    v = val_mae_per_target[k].item()
    t = test_mae_per_target[k].item()
    gap = abs(v - t) / (t + 1e-8) * 100
    status = "⚠ OVERFIT" if gap > 30 else "✓ OK"
    if gap > 30: overfit = True
    print(f"{name:<18} {v:>12.4f} {t:>12.4f} {gap:>11.1f}% {status:>12}")

val_overall = val_mae_per_target.mean().item()
test_overall = test_mae_per_target.mean().item()
overall_gap = abs(val_overall - test_overall) / (test_overall + 1e-8) * 100

print("-" * 72)
print(f"{'OVERALL':<18} {val_overall:>12.4f} {test_overall:>12.4f} {overall_gap:>11.1f}%")

if overfit:
    print("\n⚠ Overfitting detected. Consider: increase dropout, weight decay.")
else:
    print("\n✓ Model generalizes well")

print("\nTraining Summary")
print(f"  Train loss: {tr_losses[0]:.5f} → {tr_losses[-1]:.5f}")
print(f"  Val loss  : {va_losses[0]:.5f} → {va_losses[-1]:.5f}")
print(f"  Best val loss: {best_val_loss:.5f}")

**!!!! PHA RL !!!********

In [ ]:
# ============================================================
# FINAL RL AUTOSCALING WITH FROZEN GAT-GRU PREDICTOR
# Option C: DL = workload/demand forecast, RL = capacity planning
#
# Final changes:
# - Không sửa state dimension: 10 features/service
# - GAT-GRU output được xem là demand/pressure forecast, không phải post-action state
# - Action space mở rộng: {-2, -1, 0, +1, +2} để scale có tác động rõ hơn khi spike/cascade
# - Hybrid transition:
#       CPU = queueing/physics backbone + learned CPU correction nhỏ từ capacity model
#       LAT = queueing/physics fallback vì learned LAT chưa thắng persistence
#       RPS = persistence vì phụ thuộc external workload
#       ERR = overload-risk rule để scale up/down có tác động hai chiều
# - Capacity model chỉ là correction/diagnostic, KHÔNG là digital twin chính
# - Reward ưu tiên SLA hơn cost để tránh PPO học chính sách under-provisioning
# - Reset replicas theo trace replicas, không ép mọi episode bắt đầu từ 1 pod
# - Curriculum gồm 4 stage theo 4 kịch bản, có rehearsal trace cũ để tránh quên
# - PPO dùng 4 env song song, target_kl, eval nhiều episode hơn
# - Đánh giá nhiều checkpoint và chọn best tổng hợp cuối cùng
# ============================================================

import sys
import subprocess
import importlib.util

def install_if_missing(package_name, pip_name=None):
    if importlib.util.find_spec(package_name) is None:
        subprocess.check_call([
            sys.executable, "-m", "pip", "install", "-q", pip_name or package_name
        ])

install_if_missing("gymnasium")
install_if_missing("stable_baselines3", "stable-baselines3")

import os
import random
from pathlib import Path

import numpy as np
import pandas as pd
import torch
import gymnasium as gym
import matplotlib.pyplot as plt

from gymnasium import spaces
from stable_baselines3 import PPO as SB3PPO
from stable_baselines3.common.monitor import Monitor
from stable_baselines3.common.vec_env import DummyVecEnv
from stable_baselines3.common.callbacks import (
    BaseCallback,
    EvalCallback,
    StopTrainingOnNoModelImprovement,
    CallbackList,
)

class PPO(SB3PPO):
    """
    PPO wrapper để lưu lại train metrics sau mỗi lần update.
    Lý do: callback của SB3 chạy trong lúc collect_rollout, còn train/value_loss
    chỉ được ghi sau rollout. Nếu đọc trực tiếp logger.name_to_value trong callback
    thì Stage 1 thường bị toàn NaN hoặc thiếu value_loss.
    """
    def train(self) -> None:
        super().train()
        self.last_train_metrics = dict(getattr(self.logger, "name_to_value", {}))


# ============================================================
# SEED CONFIG
# ============================================================

SEED = 42

random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)

if torch.cuda.is_available():
    torch.cuda.manual_seed_all(SEED)
    torch.backends.cudnn.benchmark = True

# ============================================================
# DEVICE CONFIG
# ============================================================

N_GPU = torch.cuda.device_count()

if N_GPU >= 2:
    PPO_DEVICE = torch.device("cuda:0")
    PREDICTOR_DEVICE = torch.device("cuda:1")
elif N_GPU == 1:
    PPO_DEVICE = torch.device("cuda:0")
    PREDICTOR_DEVICE = torch.device("cuda:0")
else:
    PPO_DEVICE = torch.device("cpu")
    PREDICTOR_DEVICE = torch.device("cpu")

print("GPU count:", N_GPU)
print("PPO device:", PPO_DEVICE)
print("Predictor device:", PREDICTOR_DEVICE)

# ============================================================
# DL-COMPATIBLE CONFIG
# ============================================================

CHECKPOINT_PATH = "/kaggle/working/gat_gru_final_v4.pt"

OUTPUT_DIR = Path("/kaggle/working")
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

T_WINDOW = 4

TARGET_SERVICES = sorted([
    "adservice",
    "cartservice",
    "checkoutservice",
    "currencyservice",
    "emailservice",
    "frontend",
    "paymentservice",
    "productcatalogservice",
    "recommendationservice",
    "shippingservice",
])

NUM_NODES = len(TARGET_SERVICES)
N_SERVICES = NUM_NODES

NODE_FEAT_COLS = [
    "cpu_usage_millicores_norm",
    "memory_usage_bytes_norm",
    "pod_replicas_count_norm",
    "allocated_cpu_quota_millicores_norm",
    "error_rate_ratio_norm",
    "request_per_second_norm",
    "latency_p50_seconds_norm",
    "latency_p95_seconds_norm",
    "latency_p99_seconds_norm",
]

EDGE_FEAT_COLS = [
    "network_latency_seconds_norm",
    "payload_size_bytes_norm",
    "edge_request_rate_rps_norm",
    "edge_error_rate_ratio_norm",
]

TARGET_NAMES = [
    "cpu",
    "rps",
    "error_rate",
    "latency_p99",
]

TARGET_COLS_NORM = [
    "target_cpu_usage_millicores_norm",
    "target_request_per_second_norm",
    "target_error_rate_ratio_norm",
    "target_latency_p99_seconds_norm",
]

TARGET_COLS_REAL = [
    "target_cpu_usage_millicores",
    "target_request_per_second",
    "target_error_rate_ratio",
    "target_latency_p99_seconds",
]

NODE_FEAT_DIM = len(NODE_FEAT_COLS)
EDGE_FEAT_DIM = len(EDGE_FEAT_COLS)
NUM_TARGETS = len(TARGET_NAMES)

OBS_FEATURES_PER_SERVICE = 10

DATASET_SESSIONS = [
    {
        "name": "normal",
        "node": "/kaggle/input/datasets/kudo123a/dataset-v8/data_normal/nodes_data.csv",
        "edge": "/kaggle/input/datasets/kudo123a/dataset-v8/data_normal/edges_data.csv",
    },
    {
        "name": "anomaly_main",
        "node": "/kaggle/input/datasets/kudo123a/dataset-v8/data_anomaly/nodes_data.csv",
        "edge": "/kaggle/input/datasets/kudo123a/dataset-v8/data_anomaly/edges_data.csv",
    },
    {
        "name": "anomaly_edge",
        "node": "/kaggle/input/datasets/kudo123a/dataset-v8/data_deep_anomaly/nodes_data.csv",
        "edge": "/kaggle/input/datasets/kudo123a/dataset-v8/data_deep_anomaly/edges_data.csv",
    },
    {
        "name": "anomaly_cascade",
        "node": "/kaggle/input/datasets/kudo123a/dataset-v8/data_final_cascade_anomaly/nodes_data.csv",
        "edge": "/kaggle/input/datasets/kudo123a/dataset-v8/data_final_cascade_anomaly/edges_data.csv",
    },
]

# ============================================================
# RL CONFIG
# ============================================================

R_MIN = 1
R_MAX = 10

# Action space mới: action id 0..4 tương ứng delta {-2,-1,0,+1,+2}.
# Lý do: với SCALE_DELAY_STEPS=2, chỉ +/-1 pod thường phản ứng chậm trong spike/cascade;
# +/-2 giúp PPO tạo khác biệt rõ hơn trong digital-twin env.
MAX_DELTA = 2
ACTION_DIM = 2 * MAX_DELTA + 1
HOLD_ACTION = MAX_DELTA

# Capacity model được train chủ yếu trên action/effective_delta thuộc {-1,0,+1}.
# Vì vậy khi PPO dùng +/-2, ta vẫn chỉ đưa delta đã clip về [-1,+1] vào learned correction
# để tránh out-of-distribution; queueing backbone mới là phần xử lý +/-2.
CAPACITY_MODEL_MAX_DELTA = 1.0

EPISODE_LEN = 200
SCALE_DELAY_STEPS = 2
COOLDOWN_STEPS = 2

CPU_SOFT_THR_NORM = 0.45
CPU_HARD_THR_NORM = 0.80
CPU_THR_NORM = CPU_HARD_THR_NORM

ERR_THR_NORM = 0.01
LAT_THR_NORM = 0.10
LAT_CRITICAL_THR_NORM = 0.20

# Reward ưu tiên SLA hơn cost. Lần train trước PPO có xu hướng tiết kiệm tài nguyên/churn
# nên SLA gần baseline; giảm cost penalty để PPO dám scale trong anomaly/cascade.
W_SLA   = 0.82
W_COST  = 0.12
W_CHURN = 0.06

W_LAT = 0.34
W_LAT_CRITICAL = 0.26
W_ERR = 0.16
W_CPU = 0.24

REWARD_SCALE = 1.0  # unused; reward dùng raw_reward trực tiếp

# Capacity model params từ phân tích pattern:
# *_from_data = 0 nên KHÔNG dùng; dùng fallback queueing-inspired.
CPU_ALPHA = 0.85
LAT_BETA = 0.60
ERR_GAMMA = 0.50

# Phần có thể cải thiện bằng scale.
# Phân tích edge/network cho thấy latency/error có phần dependency/business không scale được.
LAT_SCALABLE_FRAC = 0.45
ERR_SCALABLE_FRAC = 0.30  
LEARNED_CPU_BLEND = 0.25
ERR_OVERLOAD_SCALE = 0.05

MIN_CAPACITY_GAIN = 1.0
MAX_CAPACITY_GAIN = 4.0

SERVICE_WEIGHTS = {
    "frontend": 1.5,
    "checkoutservice": 1.5,
    "cartservice": 1.3,
    "paymentservice": 1.3,
    "productcatalogservice": 1.2,
    "recommendationservice": 1.1,
    "adservice": 1.0,
    "currencyservice": 1.0,
    "emailservice": 1.0,
    "shippingservice": 1.0,
}

SERVICE_WEIGHT_VEC = np.array(
    [SERVICE_WEIGHTS[s] for s in TARGET_SERVICES],
    dtype=np.float32,
)


# ============================================================
# LEARNED CAPACITY MODEL CONFIG
# ============================================================
# Kaggle input cố định của bạn.
# Lưu ý: file này PHẢI là checkpoint learned-delta được train từ
# train_capacity_model_final_delta.py, có transition_mode chứa "delta"
# và in_dim = 19 = 9 base features + 10 service one-hot.

CAPACITY_MODEL_PATH = "/kaggle/input/datasets/kudo123a/dataset-v8/capacity_model_v3.pt"
REQUIRE_LEARNED_CAPACITY = True

print("\n========== CAPACITY MODEL PATH ==========")
print("Using:", CAPACITY_MODEL_PATH)
print("=========================================\n")

# Hybrid cuối cùng sau phân tích các lần train:
#   - Queueing/physics rule là transition chính để action scale có tác động nhân quả rõ.
#   - CPU learned-delta chỉ dùng làm correction nhỏ vì model action-effect chưa thắng persistence ổn định.
#   - LAT learned-delta KHÔNG dùng trực tiếp vì validation LAT thua persistence.
#   - RPS là external workload nên persistence.
#   - ERR dùng overload-risk rule để scale up có thể giảm risk và scale down có thể tăng risk.
#   - Capacity model vẫn load out_dim=2 để log/diagnose learned delta.
USE_LEARNED_CPU = True
USE_LEARNED_LAT = False
USE_LEARNED_RPS = False
USE_LEARNED_ERR = False

CAPACITY_TARGET_NAMES = ["cpu", "lat"]


# ============================================================
# SMOOTH CURRICULUM CONFIG
# 4 kịch bản gốc nhưng chuyển tiếp mượt bằng nhiều mini-stage.
# Ý tưởng:
# - Không nhảy đột ngột Normal -> Anomaly Main -> Edge -> Cascade
# - Stage sau vẫn giữ rehearsal từ stage trước
# - Weight dịch dần theo mixing ratio
# - LR/Entropy giảm dần theo độ khó
# ============================================================

# ============================================================
# 4-STAGE CURRICULUM + SHORT SMOOTH BLEND
# ============================================================

BLEND_WARMUP_EVALS = 10   
BLEND_HOLD_EVALS = 2

CURRICULUM = [
    {
        "name": "S1 - Normal",
        "trace_filter": ["normal"],
        "weights": [1.0],
        "timesteps": 100_000,
        "ent_coef": 0.04,
        "lr": 3e-4,
        "eval_freq": 4096,
        "n_eval_episodes": 10,
        "early_stop_min_evals": 10,
        "early_stop_patience": 12,
    },
    {
        "name": "S2 - Main Anomaly",
        "trace_filter": ["normal", "anomaly_main"],
        "weights": [0.35, 0.65],      # không đổi
        "timesteps": 130_000,
        "ent_coef": 0.04,
        "lr": 2.5e-4,
        "eval_freq": 4096,
        "n_eval_episodes": 25,
        "early_stop_min_evals": 18,
        "early_stop_patience": 20,
    },
    {
        "name": "S3 - Edge Anomaly",
        "trace_filter": ["normal", "anomaly_main", "anomaly_edge"],
        "weights": [0.15, 0.25, 0.60],  # ← tăng normal 10→15
        "timesteps": 140_000,
        "ent_coef": 0.025,
        "lr": 1.5e-4,
        "eval_freq": 4096,
        "n_eval_episodes": 25,
        "early_stop_min_evals": 20,
        "early_stop_patience": 22,
    },
    {
        "name": "S4 - Final Cascade Mix",
        "trace_filter": ["normal", "anomaly_main", "anomaly_edge", "anomaly_cascade"],
        "weights": [0.12, 0.15, 0.28, 0.45],  # ← tăng normal 10→12
        "timesteps": 250_000,                   # ← tăng 200K→250K
        "ent_coef": 0.015,
        "lr": 1.2e-4,
        "eval_freq": 4096,
        "n_eval_episodes": 30,
        "early_stop_min_evals": 35,
        "early_stop_patience": 32,
    },
]

TOTAL_CURRICULUM_STEPS = sum(s["timesteps"] for s in CURRICULUM)
TOTAL_BLEND_STEPS = (len(CURRICULUM) - 1) * BLEND_WARMUP_EVALS * 4096

print(f"Total curriculum max steps: {TOTAL_CURRICULUM_STEPS:,}")
print(f"Total blend overhead steps : {TOTAL_BLEND_STEPS:,}")

# ============================================================
# LOAD FROZEN GAT-GRU PREDICTOR
# ============================================================

if "GAT_GRU_Model" not in globals():
    raise RuntimeError("Hãy chạy code DL trước để có class GAT_GRU_Model.")

if "load_session" not in globals():
    raise RuntimeError("Hãy chạy code DL trước để có hàm load_session.")

if not os.path.exists(CHECKPOINT_PATH):
    raise FileNotFoundError(f"Không tìm thấy checkpoint: {CHECKPOINT_PATH}")

ckpt = torch.load(CHECKPOINT_PATH, map_location=PREDICTOR_DEVICE)

expected_meta = {
    "num_nodes": NUM_NODES,
    "node_dim": NODE_FEAT_DIM,
    "edge_dim": EDGE_FEAT_DIM,
    "num_targets": NUM_TARGETS,
    "t_window": T_WINDOW,
}

for key, expected_value in expected_meta.items():
    if key in ckpt and ckpt[key] != expected_value:
        raise ValueError(
            f"Checkpoint không khớp {key}: ckpt={ckpt[key]}, expected={expected_value}"
        )

predictor = GAT_GRU_Model(hidden_dim=48, dropout=0.25).to(PREDICTOR_DEVICE)
predictor.load_state_dict(ckpt["model_state"], strict=True)
predictor.eval()

for p in predictor.parameters():
    p.requires_grad = False

print("✓ Frozen GAT-GRU predictor loaded")
print("  node_dim:", ckpt.get("node_dim"))
print("  edge_dim:", ckpt.get("edge_dim"))
print("  t_window:", ckpt.get("t_window", T_WINDOW))


# ============================================================
# LOAD LEARNED CAPACITY MODEL
# ============================================================

class LearnedCapacityModel(torch.nn.Module):
    """
    Class phải khớp với train_capacity_model_cpu_lat.py:
      Linear -> ReLU -> Dropout -> Linear -> ReLU -> Dropout -> Linear
    Output là delta normalized [cpu,lat], KHÔNG sigmoid.
    """
    def __init__(self, in_dim: int, hidden: int = 64, out_dim: int = 2, dropout: float = 0.10):
        super().__init__()
        self.net = torch.nn.Sequential(
            torch.nn.Linear(in_dim, hidden),
            torch.nn.ReLU(),
            torch.nn.Dropout(dropout),
            torch.nn.Linear(hidden, hidden),
            torch.nn.ReLU(),
            torch.nn.Dropout(dropout),
            torch.nn.Linear(hidden, out_dim),
        )

    def forward(self, x: torch.Tensor) -> torch.Tensor:
        return self.net(x)


def find_capacity_model_path():
    """
    Dùng đúng path Kaggle input đã cấu hình, không auto-search.

    Lý do không auto-search:
    - Tránh load nhầm checkpoint cũ trong /kaggle/working hoặc dataset khác.
    - Nếu path sai, fail sớm để không train PPO với transition model không đúng.
    """
    path = Path(CAPACITY_MODEL_PATH)

    if not path.exists():
        raise FileNotFoundError(
            "Không tìm thấy capacity model tại path cố định:\n"
            f"  {path}\n"
            "Hãy kiểm tra Kaggle Dataset đã được add vào notebook chưa, "
            "và file capacity_model.pt có nằm đúng trong dataset-v8 không."
        )

    return path


def load_learned_capacity_model():
    path = find_capacity_model_path()

    if path is None:
        # Nhánh này gần như không xảy ra vì find_capacity_model_path() đã raise nếu thiếu file.
        msg = f"Không tìm thấy capacity_model.pt tại {CAPACITY_MODEL_PATH}"
        if REQUIRE_LEARNED_CAPACITY:
            raise FileNotFoundError(msg)
        print("⚠", msg)
        return None, None, None

    cap_ckpt = torch.load(path, map_location=PREDICTOR_DEVICE)

    required_keys = [
        "model_state", "in_dim", "hidden", "out_dim",
        "feature_names", "services", "norm_ranges", "transition_mode",
    ]
    missing = [k for k in required_keys if k not in cap_ckpt]
    if missing:
        raise ValueError(f"Capacity checkpoint thiếu keys: {missing}")

    transition_mode = cap_ckpt.get("transition_mode", "")
    if transition_mode != "learned_delta_cpu_lat_only":
        raise ValueError(
            "Capacity model hiện tại không phải bản CPU/LAT-only. "
            f"transition_mode={transition_mode}. Hãy dùng checkpoint train từ train_capacity_model_cpu_lat.py."
        )

    cap_services = list(cap_ckpt["services"])
    if cap_services != TARGET_SERVICES:
        raise ValueError(
            "Thứ tự services trong capacity model không khớp TARGET_SERVICES.\n"
            f"capacity services={cap_services}\n"
            f"target services  ={TARGET_SERVICES}"
        )

    expected_in_dim = 9 + len(TARGET_SERVICES)
    if int(cap_ckpt["in_dim"]) != expected_in_dim:
        raise ValueError(
            f"Capacity in_dim không khớp: ckpt={cap_ckpt['in_dim']}, expected={expected_in_dim}"
        )

    if int(cap_ckpt["out_dim"]) != 2:
        raise ValueError(
            f"Capacity out_dim không khớp: ckpt={cap_ckpt['out_dim']}, expected=2 cho [delta_cpu, delta_lat]"
        )

    cap_model = LearnedCapacityModel(
        in_dim=int(cap_ckpt["in_dim"]),
        hidden=int(cap_ckpt["hidden"]),
        out_dim=int(cap_ckpt["out_dim"]),
        dropout=0.10,
    ).to(PREDICTOR_DEVICE)

    cap_model.load_state_dict(cap_ckpt["model_state"], strict=True)
    cap_model.eval()

    for p in cap_model.parameters():
        p.requires_grad = False

    print("✓ Learned Capacity Model loaded")
    print("  path           :", path)
    print("  transition_mode:", transition_mode)
    print("  in_dim/hidden/out_dim:", cap_ckpt["in_dim"], cap_ckpt["hidden"], cap_ckpt["out_dim"])
    print("  hybrid usage   : CPU learned =", USE_LEARNED_CPU,
          "| LAT learned =", USE_LEARNED_LAT,
          "| RPS learned =", USE_LEARNED_RPS,
          "| ERR learned =", USE_LEARNED_ERR)

    return cap_model, cap_ckpt, path


capacity_model, capacity_ckpt, capacity_model_path = load_learned_capacity_model()

CAPACITY_NORM = capacity_ckpt["norm_ranges"] if capacity_ckpt is not None else None
CAPACITY_FEATURE_NAMES = capacity_ckpt["feature_names"] if capacity_ckpt is not None else None


def build_capacity_features_for_env(r_old, r_new, before_metrics):
    """
    Build input đúng thứ tự feature_names của capacity model.

    r_old, r_new: shape (N,), replica count trước/sau action có hiệu lực.
    before_metrics: shape (N,4), normalized [cpu,rps,err,lat] từ GAT-GRU demand forecast.

    Return: np.ndarray shape (N, 9 + N_SERVICES)
    """
    r_old = np.clip(np.asarray(r_old, dtype=np.float32), R_MIN, R_MAX)
    r_new = np.clip(np.asarray(r_new, dtype=np.float32), R_MIN, R_MAX)
    before_metrics = np.clip(np.asarray(before_metrics, dtype=np.float32), 0.0, 1.0)

    effective_delta = r_new - r_old

    # Learned capacity model đã được thu chủ yếu với delta {-1,0,+1}.
    # PPO hiện có thể chọn +/-2, nhưng phần learned correction không nên ngoại suy quá xa.
    # Queueing backbone xử lý biên độ +/-2; capacity model chỉ thấy delta đã clip về miền train.
    cap_effective_delta = np.clip(
        effective_delta,
        -CAPACITY_MODEL_MAX_DELTA,
        CAPACITY_MODEL_MAX_DELTA,
    )
    action = cap_effective_delta.copy()

    # load_norm không có trong GAT-GRU output trực tiếp.
    # Dùng mean RPS demand làm proxy workload level normalized.
    # Lý do: capacity model được train với load_level; trong Kaggle RL trace không có Locust users thật.
    # Proxy này giúp model phân biệt low/high pressure mà không đổi state dimension.
    # Train capacity model dùng load_level/300, còn RL trace không có Locust users thật.
    # mean RPS normalized thường thấp hơn miền train, nên nhân 3.0 để đưa proxy gần miền train hơn.
    load_proxy_norm = np.clip(np.mean(before_metrics[:, 1]) * 3.0, 0.0, 1.0)
    load_col = np.full((N_SERVICES,), load_proxy_norm, dtype=np.float32)

    base = np.stack([
        r_old / float(R_MAX),
        r_new / float(R_MAX),
        action.astype(np.float32),
        cap_effective_delta.astype(np.float32),
        before_metrics[:, 0],
        before_metrics[:, 1],
        before_metrics[:, 2],
        before_metrics[:, 3],
        load_col,
    ], axis=1).astype(np.float32)

    service_onehot = np.eye(N_SERVICES, dtype=np.float32)
    features = np.concatenate([base, service_onehot], axis=1).astype(np.float32)

    if CAPACITY_FEATURE_NAMES is not None and features.shape[1] != len(CAPACITY_FEATURE_NAMES):
        raise ValueError(
            f"Capacity feature dim mismatch: features={features.shape[1]}, "
            f"ckpt_feature_names={len(CAPACITY_FEATURE_NAMES)}"
        )

    return features

# ============================================================
# LOAD RL TRACES
# ============================================================


def blend_stage_weights(prev_stage, cur_stage, alpha):
    """
    Nội suy tuyến tính distribution giữa stage trước và stage hiện tại.

    alpha = 0.0: dùng 100% distribution của prev_stage
    alpha = 1.0: dùng 100% distribution của cur_stage
    """
    alpha = float(np.clip(alpha, 0.0, 1.0))

    prev_w = dict(zip(prev_stage["trace_filter"], prev_stage["weights"]))
    cur_w = dict(zip(cur_stage["trace_filter"], cur_stage["weights"]))

    all_names = sorted(set(prev_w.keys()) | set(cur_w.keys()))

    blended = {}
    for name in all_names:
        blended[name] = (
            (1.0 - alpha) * float(prev_w.get(name, 0.0))
            + alpha * float(cur_w.get(name, 0.0))
        )

    total = float(sum(blended.values()))
    if total <= 0:
        raise ValueError("Blend weights sum must be > 0.")

    return {
        k: v / total
        for k, v in blended.items()
        if v > 1e-8
    }


class SmoothBlendCallback(BaseCallback):
    """
    Smooth blend ở đầu mỗi stage, không rebuild env và không set_env liên tục.

    Cơ chế:
    - Khi training stage mới bắt đầu, train env được ép về distribution của stage trước.
    - Sau mỗi eval_freq timesteps, alpha tăng dần tới 1.0.
    - Khi alpha=1.0, train env dùng distribution của stage hiện tại.
    """
    def __init__(
        self,
        prev_stage,
        cur_stage,
        blend_warmup_evals=BLEND_WARMUP_EVALS,
        blend_hold_evals=BLEND_HOLD_EVALS,
        eval_freq=4096,
        verbose=1,
    ):
        super().__init__(verbose)
        self.prev_stage = prev_stage
        self.cur_stage = cur_stage
        self.blend_warmup_evals = int(blend_warmup_evals)
        self.blend_hold_evals = int(blend_hold_evals)
        self.eval_freq = int(eval_freq)

        self.last_step = 0
        self.update_idx = 0

    def _on_training_start(self) -> None:
        weights = blend_stage_weights(
            self.prev_stage,
            self.cur_stage,
            alpha=0.0,
        )
        self.training_env.env_method("set_trace_weights", weights)

        if self.verbose:
            w_str = " | ".join([f"{k}={v:.2f}" for k, v in weights.items()])
            print(f"[SmoothBlend] start alpha=0.00 | {w_str}")

    def _on_step(self) -> bool:
        if self.num_timesteps - self.last_step < self.eval_freq:
            return True

        self.last_step = self.num_timesteps
        self.update_idx += 1

        # Mỗi alpha giữ blend_hold_evals lần eval interval
        alpha_idx = int(np.ceil(self.update_idx / self.blend_hold_evals))
        alpha = min(1.0, alpha_idx / float(self.blend_warmup_evals))

        weights = blend_stage_weights(
            self.prev_stage,
            self.cur_stage,
            alpha=alpha,
        )

        self.training_env.env_method("set_trace_weights", weights)

        if self.verbose:
            w_str = " | ".join([f"{k}={v:.2f}" for k, v in weights.items()])
            print(
                f"[SmoothBlend] update={self.update_idx} "
                f"| alpha={alpha:.2f} | {w_str}"
            )

        return True


def inspect_trace_distribution(Y, name):
    y = Y.reshape(-1, NUM_TARGETS).detach().cpu().numpy()

    print(f"\n=== Trace distribution: {name} ===")

    for i, tname in enumerate(TARGET_NAMES):
        vals = y[:, i]
        print(
            f"  {tname:12s} "
            f"p50={np.percentile(vals, 50):.5f} "
            f"p90={np.percentile(vals, 90):.5f} "
            f"p95={np.percentile(vals, 95):.5f} "
            f"p99={np.percentile(vals, 99):.5f} "
            f"max={vals.max():.5f} "
            f"nonzero={(vals > 1e-6).mean() * 100:.1f}%"
        )


def load_rl_trace(session_cfg):
    node_path = session_cfg["node"]
    edge_path = session_cfg["edge"]

    if not os.path.exists(node_path) or not os.path.exists(edge_path):
        print(f"Skip missing session: {session_cfg['name']}")
        return None

    sess = load_session(
        node_file=node_path,
        edge_file=edge_path,
        t_window=T_WINDOW,
    )

    if sess is None:
        print(f"Skip empty session: {session_cfg['name']}")
        return None

    return {
        "name": session_cfg["name"],
        "X": sess["X"].float().to(PREDICTOR_DEVICE),
        "E": sess["E"].float().to(PREDICTOR_DEVICE),
        "Y_norm": sess["Yn"].float().to(PREDICTOR_DEVICE),
        "Y_real": sess["Yr"].float().to(PREDICTOR_DEVICE),
        "A": sess["A"].float().to(PREDICTOR_DEVICE),
    }


traces = []

for cfg in DATASET_SESSIONS:
    tr = load_rl_trace(cfg)

    if tr is not None and len(tr["X"]) > EPISODE_LEN + 2:
        traces.append(tr)
        print(f"✓ Loaded {tr['name']}: {len(tr['X'])} windows")
        inspect_trace_distribution(tr["Y_norm"], tr["name"])
    else:
        print(f"Skip invalid/short session: {cfg['name']}")

if len(traces) == 0:
    raise RuntimeError("No valid RL traces loaded.")

A_global = traces[0]["A"].clone()

for tr in traces[1:]:
    A_global = (A_global + tr["A"]).clamp(0.0, 1.0)

A_global = A_global.to(PREDICTOR_DEVICE)

print("\n✓ Global adjacency merged")
print("A_global shape:", tuple(A_global.shape))

# ============================================================
# RL ENVIRONMENT
# ============================================================

class OnlineBoutiqueScalingEnv(gym.Env):
    metadata = {"render_modes": []}

    def __init__(
        self,
        traces,
        predictor,
        A,
        episode_len=EPISODE_LEN,
        force_trace_idx=None,
    ):
        super().__init__()

        self.traces = traces
        self.predictor = predictor
        self.A = A
        self.episode_len = episode_len
        self.num_services = N_SERVICES
        self.force_trace_idx = force_trace_idx

        self.action_space = spaces.MultiDiscrete([ACTION_DIM] * self.num_services)

        self.obs_features_per_service = OBS_FEATURES_PER_SERVICE

        self.observation_space = spaces.Box(
            low=0.0,
            high=1.0,
            shape=(self.num_services * self.obs_features_per_service,),
            dtype=np.float32,
        )

        self.trace = None
        self.trace_weight_override = None
        self.ptr = 0
        self.t0 = 0
        self.steps = 0

        self.effective_replicas = None
        self.desired_replicas = None
        self.prev_effective_replicas = None
        self.pending_actions = None
        self.cooldown = None

    def _sample_trace(self):
        if self.force_trace_idx is not None:
            return self.traces[self.force_trace_idx]

        if self.trace_weight_override is not None:
            weights = np.array(
                [self.trace_weight_override.get(tr["name"], 0.0) for tr in self.traces],
                dtype=np.float32,
            )
        else:
            weights = np.array(
                [tr.get("weight", 1.0) for tr in self.traces],
                dtype=np.float32,
            )

        if weights.sum() <= 0:
            weights = np.ones(len(self.traces), dtype=np.float32)

        weights = weights / weights.sum()
        idx = np.random.choice(len(self.traces), p=weights)

        return self.traces[idx]

    def _trace_replicas_at(self, ptr=None):
        """
        Lấy replicas quan sát trong trace.
        Dùng để khởi tạo episode cho realistic, không ép mọi episode bắt đầu 1 pod.
        """
        if ptr is None:
            ptr = self.ptr

        x_now = self.trace["X"][ptr, -1].detach().cpu().numpy()
        r_trace = np.rint(x_now[:, 2] * R_MAX).astype(np.int32)
        r_trace = np.clip(r_trace, R_MIN, R_MAX)

        return r_trace

    def reset(self, seed=None, options=None):
        super().reset(seed=seed)

        self.trace = self._sample_trace()

        max_start = len(self.trace["X"]) - self.episode_len - 2
        # Dùng RNG nội bộ của Gymnasium để reset(seed=...) thật sự deterministic.
        # Không dùng random.randint() vì nó tiêu thụ global RNG và có thể bị callback/eval làm lệch trajectory train.
        self.t0 = int(self.np_random.integers(0, max(1, max_start + 1)))

        self.ptr = self.t0
        self.steps = 0

        replica_from_trace = self._trace_replicas_at(self.ptr)

        self.effective_replicas = replica_from_trace.copy()
        self.desired_replicas = replica_from_trace.copy()
        self.prev_effective_replicas = replica_from_trace.copy()

        self.pending_actions = []
        self.cooldown = np.zeros(
            self.num_services,
            dtype=np.int32,
        )

        return self._build_obs(), {"trace_name": self.trace["name"]}

    def _predict_next(self):
        X_seq = self.trace["X"][self.ptr].unsqueeze(0)  # (1, T, N, F)
        E_seq = self.trace["E"][self.ptr].unsqueeze(0)  # (1, T, N, N, Fe)
    
        with torch.no_grad():
            final_out, _ = self.predictor(X_seq, E_seq, self.A)  # final_out: (1, N, 4)
        return final_out[0].detach().cpu().numpy()  # (N, 4)


    def _apply_capacity_model(self, demand_pred, replicas):
        before = np.clip(demand_pred.astype(np.float32), 0.0, 1.0)
    
        r_old = np.clip(self.prev_effective_replicas.astype(np.float32), R_MIN, R_MAX)
        r_new = np.clip(replicas.astype(np.float32), R_MIN, R_MAX)
    
        cpu_demand = before[:, 0]
        rps_demand = before[:, 1]
        err_demand = before[:, 2]
        lat_demand = before[:, 3]
    
        effective_delta = r_new - r_old
        ratio = np.clip(r_new / np.maximum(r_old, 1.0), 0.1, 10.0)
    
        cpu_gain = np.power(ratio, CPU_ALPHA)
        lat_gain = np.power(ratio, LAT_BETA)
    
        cpu_rule = cpu_demand / np.maximum(cpu_gain, 1e-6)
        lat_rule = lat_demand * (
            (LAT_SCALABLE_FRAC / np.maximum(lat_gain, 1e-6))
            + (1.0 - LAT_SCALABLE_FRAC)
        )
    
        cpu_rule = np.clip(cpu_rule, 0.0, 1.0)
        lat_rule = np.clip(lat_rule, 0.0, 1.0)
    
        learned_delta_cpu_lat = np.zeros((N_SERVICES, 2), dtype=np.float32)
        learned_cpu_after = cpu_rule.copy()
    
        if capacity_model is not None and USE_LEARNED_CPU:
            cap_features = build_capacity_features_for_env(
                r_old=r_old,
                r_new=r_new,
                before_metrics=before,
            )
    
            with torch.no_grad():
                cap_x = torch.tensor(cap_features, dtype=torch.float32, device=PREDICTOR_DEVICE)
                learned_delta_cpu_lat = (
                    capacity_model(cap_x)
                    .detach()
                    .cpu()
                    .numpy()
                    .astype(np.float32)
                )
    
            learned_cpu_after = np.clip(
                cpu_demand + learned_delta_cpu_lat[:, 0],
                0.0,
                1.0,
            )
    
        if USE_LEARNED_CPU:
            cpu_pressure = (
                (1.0 - LEARNED_CPU_BLEND) * cpu_rule
                + LEARNED_CPU_BLEND * learned_cpu_after
            )
        else:
            cpu_pressure = cpu_rule
    
        scale_up_mask = effective_delta > 0
        scale_down_mask = effective_delta < 0
        hold_mask = effective_delta == 0
    
        cpu_pressure[scale_up_mask] = np.minimum(cpu_pressure[scale_up_mask], cpu_demand[scale_up_mask])
        cpu_pressure[scale_down_mask] = np.maximum(cpu_pressure[scale_down_mask], cpu_demand[scale_down_mask])
        cpu_pressure[hold_mask] = cpu_demand[hold_mask]
    
        lat_pressure = lat_rule.copy()
        lat_pressure[scale_up_mask] = np.minimum(lat_pressure[scale_up_mask], lat_demand[scale_up_mask])
        lat_pressure[scale_down_mask] = np.maximum(lat_pressure[scale_down_mask], lat_demand[scale_down_mask])
        lat_pressure[hold_mask] = lat_demand[hold_mask]
    
        rps_pressure = rps_demand
    
        cpu_risk_before = np.maximum(
            0.0,
            (cpu_demand - CPU_HARD_THR_NORM) / (1.0 - CPU_HARD_THR_NORM + 1e-8),
        )
        cpu_risk_after = np.maximum(
            0.0,
            (cpu_pressure - CPU_HARD_THR_NORM) / (1.0 - CPU_HARD_THR_NORM + 1e-8),
        )
    
        lat_risk_before = np.maximum(
            0.0,
            (lat_demand - LAT_THR_NORM) / (1.0 - LAT_THR_NORM + 1e-8),
        )
        lat_risk_after = np.maximum(
            0.0,
            (lat_pressure - LAT_THR_NORM) / (1.0 - LAT_THR_NORM + 1e-8),
        )
    
        overload_risk_before = 0.5 * cpu_risk_before + 0.5 * lat_risk_before
        overload_risk_after = 0.5 * cpu_risk_after + 0.5 * lat_risk_after
    
        err_delta = (
            ERR_SCALABLE_FRAC
            * ERR_OVERLOAD_SCALE
            * (overload_risk_after - overload_risk_before)
        )
    
        err_pressure = np.clip(err_demand + err_delta, 0.0, 1.0)
    
        cpu_pressure = np.clip(cpu_pressure, 0.0, 1.0)
        rps_pressure = np.clip(rps_pressure, 0.0, 1.0)
        lat_pressure = np.clip(lat_pressure, 0.0, 1.0)
    
        eps = 1e-6
        cpu_gain_eff = np.clip(cpu_demand / (cpu_pressure + eps), 0.0, MAX_CAPACITY_GAIN)
        lat_gain_eff = np.clip(lat_demand / (lat_pressure + eps), 0.0, MAX_CAPACITY_GAIN)
        err_gain_eff = np.clip(err_demand / (err_pressure + eps), 0.0, MAX_CAPACITY_GAIN)
    
        capacity_info = {
            "avg_demand_cpu": float(np.mean(cpu_demand)),
            "avg_demand_rps": float(np.mean(rps_demand)),
            "avg_demand_err": float(np.mean(err_demand)),
            "avg_demand_lat": float(np.mean(lat_demand)),
            "avg_cpu_gain": float(np.mean(cpu_gain_eff)),
            "avg_lat_gain": float(np.mean(lat_gain_eff)),
            "avg_err_gain": float(np.mean(err_gain_eff)),
            "avg_learned_delta_cpu": float(np.mean(learned_delta_cpu_lat[:, 0])),
            "avg_learned_delta_rps": 0.0,
            "avg_learned_delta_err": 0.0,
            "avg_learned_delta_lat": float(np.mean(learned_delta_cpu_lat[:, 1])),
            "avg_capacity_r_old": float(np.mean(r_old)),
            "avg_capacity_r_new": float(np.mean(r_new)),
            "capacity_source_id": 2.0,
        }
    
        return cpu_pressure, rps_pressure, err_pressure, lat_pressure, capacity_info

    def set_trace_weights(self, weight_dict):
        """
        Cập nhật weight sampling của trace cho riêng env hiện tại.
        Không mutate trực tiếp self.traces để tránh nhiều env dùng chung dict bị ảnh hưởng lẫn nhau.
        """
        weights = {}

        for tr in self.traces:
            weights[tr["name"]] = float(weight_dict.get(tr["name"], 0.0))

        total = float(sum(weights.values()))
        if total <= 0:
            raise ValueError("Tổng trace weights phải > 0.")

        self.trace_weight_override = {
            name: value / total
            for name, value in weights.items()
        }

    def _build_obs(self):
        x_now = self.trace["X"][self.ptr, -1].detach().cpu().numpy()  # (N, 9)
    
        cpu_now = x_now[:, 0].reshape(-1)
        err_now = x_now[:, 4].reshape(-1)
        rps_now = x_now[:, 5].reshape(-1)
        lat_now = x_now[:, 8].reshape(-1)
    
        with torch.no_grad():
            demand_pred, _ = self.predictor(
                self.trace["X"][self.ptr].unsqueeze(0),
                self.trace["E"][self.ptr].unsqueeze(0),
                self.A
            )
        demand_pred = demand_pred[0].detach().cpu().numpy()  # (N, 4)
    
        cpu_demand = demand_pred[:, 0].reshape(-1)
        rps_demand = demand_pred[:, 1].reshape(-1)
        err_demand = demand_pred[:, 2].reshape(-1)
        lat_demand = demand_pred[:, 3].reshape(-1)
    
        replica_norm = (self.effective_replicas.astype(np.float32) / R_MAX).reshape(-1)
        cooldown_norm = (self.cooldown.astype(np.float32) / float(max(COOLDOWN_STEPS, 1))).reshape(-1)
    
        state = np.stack(
            [
                replica_norm,
                cpu_now,
                rps_now,
                err_now,
                lat_now,
                cpu_demand,
                rps_demand,
                err_demand,
                lat_demand,
                cooldown_norm,
            ],
            axis=1,
        )  # (N, 10)
    
        return np.clip(state, 0.0, 1.0).astype(np.float32).reshape(-1)  # flatten thành (N*10,)



    def _compute_reward(self, actual_delta, demand_pred, replicas):
        cpu_hat, rps_hat, err_hat, lat_hat, cap_info = self._apply_capacity_model(
            demand_pred,
            replicas,
        )
        
        # cpu_hat đã là post-action pressure từ hybrid physics-informed transition.
        # Không trộn lại raw demand ở đây để reward phản ánh trực tiếp tác động action.
        cpu_pressure = cpu_hat

        r_eff = replicas.astype(np.float32)

        lat_violation = np.clip(
            np.maximum(
                0.0,
                (lat_hat - LAT_THR_NORM) / (LAT_THR_NORM + 1e-8),
            ),
            0.0,
            2.0,
        )

        lat_critical_violation = np.clip(
            np.maximum(
                0.0,
                (lat_hat - LAT_CRITICAL_THR_NORM)
                / (LAT_CRITICAL_THR_NORM + 1e-8),
            ),
            0.0,
            2.0,
        )

        err_violation = np.clip(
            np.maximum(
                0.0,
                (err_hat - ERR_THR_NORM) / (ERR_THR_NORM + 1e-8),
            ),
            0.0,
            2.0,
        )

        cpu_soft_violation = np.clip(
            np.maximum(
                0.0,
                (cpu_pressure - CPU_SOFT_THR_NORM) / (CPU_HARD_THR_NORM - CPU_SOFT_THR_NORM + 1e-8),
            ),
            0.0,
            1.0,
        )
        
        cpu_hard_violation = np.clip(
            np.maximum(
                0.0,
                (cpu_pressure - CPU_HARD_THR_NORM) / (CPU_HARD_THR_NORM + 1e-8),
            ),
            0.0,
            2.0,
        )
        
        cpu_violation = 0.7 * cpu_soft_violation + 0.3 * cpu_hard_violation

        sla_i = (
            W_LAT * lat_violation
            + W_LAT_CRITICAL * lat_critical_violation
            + W_ERR * err_violation
            + W_CPU * cpu_violation
        )

        sla_t = float(
            np.sum(SERVICE_WEIGHT_VEC * sla_i)
            / np.sum(SERVICE_WEIGHT_VEC)
        )

        resource_cost_quad = float(
            np.sum((r_eff / R_MAX) ** 2) / self.num_services
        )

        resource_cost_linear = float(
            np.sum(r_eff) / (self.num_services * R_MAX)
        )

        churn_cost = float(
            np.sum(np.abs(actual_delta)) / (self.num_services * MAX_DELTA)
        )

        resource_cost = (
            0.4 * resource_cost_linear
            + 0.6 * resource_cost_quad
        )
        
        raw_reward = -(
            W_SLA * sla_t
            + W_COST * resource_cost
            + W_CHURN * churn_cost
        )

        reward = raw_reward

        info = {
            "sla": sla_t,
            "resource_cost": resource_cost,
            "resource_cost_linear": resource_cost_linear,
            "resource_cost_quad": resource_cost_quad,
            "churn_cost": churn_cost,
            "avg_effective_replicas": float(np.mean(self.effective_replicas)),
            "avg_desired_replicas": float(np.mean(self.desired_replicas)),
            "lat_violation": float(np.mean(lat_violation)),
            "lat_critical_violation": float(np.mean(lat_critical_violation)),
            "err_violation": float(np.mean(err_violation)),
            "cpu_violation": float(np.mean(cpu_violation)),
            "cpu_soft_violation": float(np.mean(cpu_soft_violation)),
            "cpu_hard_violation": float(np.mean(cpu_hard_violation)),
            "avg_pred_cpu": float(np.mean(cpu_hat)),
            "avg_pred_rps": float(np.mean(rps_hat)),
            "avg_pred_err": float(np.mean(err_hat)),
            "avg_pred_lat": float(np.mean(lat_hat)),
            "avg_cpu_pressure": float(np.mean(cpu_pressure)),
            **cap_info,
        }

        return reward, info

    def step(self, action):
        action = np.asarray(action, dtype=np.int64)

        # Demand forecast tại state hiện tại, lấy đúng một lần cho reward step này.
        demand_pred_t = self._predict_next()

        raw_delta = action - HOLD_ACTION
        delta = raw_delta.copy()

        # Service còn cooldown thì ép no-op.
        delta[self.cooldown > 0] = 0

        self.prev_effective_replicas = self.effective_replicas.copy()

        new_desired = np.clip(
            self.desired_replicas + delta,
            R_MIN,
            R_MAX,
        ).astype(np.int32)

        actual_delta = new_desired - self.desired_replicas
        self.desired_replicas = new_desired

        if np.any(actual_delta != 0):
            self.pending_actions.append(
                {
                    "remaining": SCALE_DELAY_STEPS,
                    "target": self.desired_replicas.copy(),
                }
            )

            self.cooldown[actual_delta != 0] = COOLDOWN_STEPS

        self.cooldown = np.maximum(self.cooldown - 1, 0)

        for item in self.pending_actions:
            item["remaining"] -= 1

        ready_items = [
            item for item in self.pending_actions
            if item["remaining"] <= 0
        ]

        if ready_items:
            self.effective_replicas = ready_items[-1]["target"].copy()

        self.pending_actions = [
            item for item in self.pending_actions
            if item["remaining"] > 0
        ]

        reward, info = self._compute_reward(
            actual_delta=actual_delta,
            demand_pred=demand_pred_t,
            replicas=self.effective_replicas,
        )

        self.ptr += 1
        self.steps += 1

        terminated = False
        truncated = self.steps >= self.episode_len

        info["pending_actions"] = len(self.pending_actions)
        info["cooldown_services"] = int((self.cooldown > 0).sum())
        info["raw_action_delta_mean"] = float(np.mean(raw_delta))
        info["actual_action_delta_mean"] = float(np.mean(actual_delta))
        info["trace_name"] = self.trace["name"]

        return self._build_obs(), reward, terminated, truncated, info


# ============================================================
# PURE PPO BASELINE ENVIRONMENT (NO GNN FORECAST IN OBS/ACTION)
# ============================================================

PURE_OBS_FEATURES_PER_SERVICE = 6

PURE_OBS_COLS_FROM_MAIN = [0, 1, 2, 3, 4, 9]
# main obs: [replica, cpu_now, rps_now, err_now, lat_now,
#            cpu_gnn, rps_gnn, err_gnn, lat_gnn, cooldown]
# pure obs: [replica, cpu_now, rps_now, err_now, lat_now, cooldown]


def main_obs_to_pure_obs(obs):
    """
    Convert observation của env chính (N x 10) sang observation PPO thuần (N x 6).
    Hàm này dùng khi evaluate Pure PPO trong cùng simulator với PPO chính.
    """
    obs = np.asarray(obs, dtype=np.float32)
    state = obs.reshape(N_SERVICES, OBS_FEATURES_PER_SERVICE)
    pure_state = state[:, PURE_OBS_COLS_FROM_MAIN]
    return np.clip(pure_state, 0.0, 1.0).astype(np.float32).reshape(-1)


class PurePPOPolicyAdapter:
    """
    Adapter để Pure PPO có thể chạy trong evaluation env chính.

    - Env chính trả obs N*10 vì các rule baseline và PPO chính cần GNN forecast.
    - Pure PPO được train với obs N*6, nên adapter này drop 4 cột forecast GNN
      trước khi gọi pure_ppo_model.predict().
    - Nhờ vậy tất cả policy vẫn được evaluate trong CÙNG simulator, nhưng Pure PPO
      không nhìn thấy GNN forecast khi ra action.
    """
    def __init__(self, model):
        self.model = model

    def predict(self, obs, deterministic=True):
        pure_obs = main_obs_to_pure_obs(obs)
        return self.model.predict(pure_obs, deterministic=deterministic)


class PurePPOScalingEnv(OnlineBoutiqueScalingEnv):
    """
    PPO thuần: policy không thấy GAT-GRU forecast trong observation.

    Khác với PPO + GNN:
      - observation chỉ có 6 feature/service:
        [replica, cpu_now, rps_now, err_now, lat_now, cooldown]

    Nhưng environment transition/reward vẫn giống env chính:
      - _predict_next() dùng lại của OnlineBoutiqueScalingEnv
      - tức là vẫn dùng cùng GAT-GRU/world model để tạo demand cho reward
      - như vậy so sánh công bằng: chỉ khác policy input, không khác môi trường
    """
    def __init__(
        self,
        traces,
        predictor,
        A,
        episode_len=EPISODE_LEN,
        force_trace_idx=None,
    ):
        super().__init__(
            traces=traces,
            predictor=predictor,
            A=A,
            episode_len=episode_len,
            force_trace_idx=force_trace_idx,
        )

        self.obs_features_per_service = PURE_OBS_FEATURES_PER_SERVICE
        self.observation_space = spaces.Box(
            low=0.0,
            high=1.0,
            shape=(self.num_services * self.obs_features_per_service,),
            dtype=np.float32,
        )

    def _build_obs(self):
        main_obs = OnlineBoutiqueScalingEnv._build_obs(self)
        return main_obs_to_pure_obs(main_obs)

# ============================================================
# CALLBACK LOGGING
# ============================================================

class CurriculumTrainingLogCallback(BaseCallback):
    def __init__(
        self,
        stage_name,
        stage_offset=0,
        model_start_timesteps=0,
        csv_path="/kaggle/working/curriculum_training_log.csv",
        log_freq=4096,
        verbose=1,
    ):
        super().__init__(verbose)

        self.stage_name = stage_name
        self.csv_path = csv_path
        self.log_freq = log_freq
        self.last_log_step = 0
        self.stage_offset = stage_offset
        self.model_start_timesteps = model_start_timesteps

        self._init_buffers()

    def _init_buffers(self):
        self.ep_rewards = []
        self.ep_lengths = []

        self.sla_buf = []
        self.cost_buf = []
        self.cost_quad_buf = []
        self.churn_buf = []
        self.replicas_buf = []
        self.desired_replicas_buf = []
        
        self.cpu_soft_buf = []
        self.cpu_hard_buf = []

        self.lat_buf = []
        self.lat_critical_buf = []
        self.err_buf = []
        self.cpu_buf = []
        self.cpu_pressure_buf = []

        self.pred_cpu_buf = []
        self.pred_rps_buf = []
        self.pred_err_buf = []
        self.pred_lat_buf = []

        self.demand_cpu_buf = []
        self.demand_rps_buf = []
        self.demand_err_buf = []
        self.demand_lat_buf = []
        self.cpu_gain_buf = []
        self.lat_gain_buf = []
        self.err_gain_buf = []

        self.learned_delta_cpu_buf = []
        self.learned_delta_rps_buf = []
        self.learned_delta_err_buf = []
        self.learned_delta_lat_buf = []

    def _on_step(self) -> bool:
        for info in self.locals.get("infos", []):
            if "episode" in info:
                self.ep_rewards.append(info["episode"]["r"])
                self.ep_lengths.append(info["episode"]["l"])

            key_buf_pairs = [
                ("sla", self.sla_buf),
                ("resource_cost", self.cost_buf),
                ("resource_cost_quad", self.cost_quad_buf),
                ("churn_cost", self.churn_buf),
                ("avg_effective_replicas", self.replicas_buf),
                ("avg_desired_replicas", self.desired_replicas_buf),
                ("lat_violation", self.lat_buf),
                ("lat_critical_violation", self.lat_critical_buf),
                ("err_violation", self.err_buf),
                ("cpu_violation", self.cpu_buf),
                ("avg_cpu_pressure", self.cpu_pressure_buf),
                ("avg_pred_cpu", self.pred_cpu_buf),
                ("avg_pred_rps", self.pred_rps_buf),
                ("avg_pred_err", self.pred_err_buf),
                ("avg_pred_lat", self.pred_lat_buf),
                ("avg_demand_cpu", self.demand_cpu_buf),
                ("avg_demand_rps", self.demand_rps_buf),
                ("avg_demand_err", self.demand_err_buf),
                ("avg_demand_lat", self.demand_lat_buf),
                ("avg_cpu_gain", self.cpu_gain_buf),
                ("avg_lat_gain", self.lat_gain_buf),
                ("avg_err_gain", self.err_gain_buf),
                ("avg_learned_delta_cpu", self.learned_delta_cpu_buf),
                ("avg_learned_delta_rps", self.learned_delta_rps_buf),
                ("avg_learned_delta_err", self.learned_delta_err_buf),
                ("avg_learned_delta_lat", self.learned_delta_lat_buf),
                ("cpu_soft_violation", self.cpu_soft_buf),
                ("cpu_hard_violation", self.cpu_hard_buf),
            ]

            for key, buf in key_buf_pairs:
                if key in info:
                    buf.append(info[key])

        if self.num_timesteps - self.last_log_step >= self.log_freq:
            # Tránh ghi dòng rác đầu mỗi stage khi chưa có episode hoàn chỉnh
            # và SB3 chưa có train/value_loss, train/approx_kl.
            if len(self.ep_rewards) == 0:
                return True

            self.last_log_step = self.num_timesteps

            # Ưu tiên dùng train metrics được lưu lại bởi LoggedPPO.train().
            # Đọc trực tiếp logger.name_to_value ở callback thường bị NaN,
            # đặc biệt ở Stage 1, vì callback chạy trước khi SB3 ghi train/value_loss.
            lv = getattr(self.model, "last_train_metrics", None)
            if not lv:
                lv = getattr(self.model.logger, "name_to_value", {})

            def mean(buf):
                return float(np.mean(buf)) if buf else float("nan")

            local_timesteps = self.num_timesteps - self.model_start_timesteps
            global_timesteps = self.stage_offset + local_timesteps

            row = {
                "stage": self.stage_name,
                "timesteps": global_timesteps,
                "episodes": len(self.ep_rewards),
                "mean_episode_reward": mean(self.ep_rewards),
                "mean_episode_length": mean(self.ep_lengths),
                "mean_sla": mean(self.sla_buf),
                "mean_resource_cost": mean(self.cost_buf),
                "mean_resource_cost_quad": mean(self.cost_quad_buf),
                "mean_churn": mean(self.churn_buf),
                "mean_effective_replicas": mean(self.replicas_buf),
                "mean_desired_replicas": mean(self.desired_replicas_buf),
                "mean_lat_violation": mean(self.lat_buf),
                "mean_lat_critical_violation": mean(self.lat_critical_buf),
                "mean_err_violation": mean(self.err_buf),
                "mean_cpu_violation": mean(self.cpu_buf),
                "mean_cpu_pressure": mean(self.cpu_pressure_buf),
                "mean_cpu_soft_violation": mean(self.cpu_soft_buf),
                "mean_cpu_hard_violation": mean(self.cpu_hard_buf),
                "mean_pred_cpu": mean(self.pred_cpu_buf),
                "mean_pred_rps": mean(self.pred_rps_buf),
                "mean_pred_err": mean(self.pred_err_buf),
                "mean_pred_lat": mean(self.pred_lat_buf),
                "mean_demand_cpu": mean(self.demand_cpu_buf),
                "mean_demand_rps": mean(self.demand_rps_buf),
                "mean_demand_err": mean(self.demand_err_buf),
                "mean_demand_lat": mean(self.demand_lat_buf),
                "mean_cpu_gain": mean(self.cpu_gain_buf),
                "mean_lat_gain": mean(self.lat_gain_buf),
                "mean_err_gain": mean(self.err_gain_buf),
                "mean_learned_delta_cpu": mean(self.learned_delta_cpu_buf),
                "mean_learned_delta_rps": mean(self.learned_delta_rps_buf),
                "mean_learned_delta_err": mean(self.learned_delta_err_buf),
                "mean_learned_delta_lat": mean(self.learned_delta_lat_buf),
                "approx_kl": lv.get("train/approx_kl", float("nan")),
                "clip_fraction": lv.get("train/clip_fraction", float("nan")),
                "entropy_loss": lv.get("train/entropy_loss", float("nan")),
                "explained_variance": lv.get("train/explained_variance", float("nan")),
                "learning_rate": lv.get("train/learning_rate", float("nan")),
                "loss": lv.get("train/loss", float("nan")),
                "policy_gradient_loss": lv.get("train/policy_gradient_loss", float("nan")),
                "value_loss": lv.get("train/value_loss", float("nan")),
            }

            df_row = pd.DataFrame([row])

            if os.path.exists(self.csv_path):
                df_row.to_csv(
                    self.csv_path,
                    mode="a",
                    header=False,
                    index=False,
                )
            else:
                df_row.to_csv(self.csv_path, index=False)

            print(
                f"[{self.stage_name}] "
                f"steps={row['timesteps']:,} | "
                f"R={row['mean_episode_reward']:+.3f} | "
                f"SLA={row['mean_sla']:.4f} | "
                f"Lat={row['mean_lat_violation']:.4f} | "
                f"Err={row['mean_err_violation']:.4f} | "
                f"CPU={row['mean_cpu_violation']:.4f} "
                f"(soft={row['mean_cpu_soft_violation']:.4f}, hard={row['mean_cpu_hard_violation']:.4f}) | "
                f"Cost={row['mean_resource_cost']:.4f} | "
                f"Reps={row['mean_effective_replicas']:.2f} | "
                f"Churn={row['mean_churn']:.4f} | "
                f"VLoss={row['value_loss']:.4f} | "
                f"KL={row['approx_kl']:.5f}"
            )

            self._init_buffers()

        return True



class PerTraceEvalCallback(BaseCallback):
    """
    Đánh giá cố định từng trace trong lúc train.

    Lý do cần callback này:
    - Train env ở stage sau sample random nhiều trace theo weight curriculum.
    - Vì mỗi trace có reward scale khác nhau, mean_episode_reward tổng sẽ rung mạnh.
    - Per-trace eval giúp vẽ đường learning curve đúng hơn cho curriculum:
        normal không bị quên,
        anomaly_main cải thiện sau S2,
        anomaly_edge cải thiện sau S3,
        cascade cải thiện sau S4.
    """
    def __init__(
        self,
        all_traces,
        stage_name,
        stage_offset=0,
        model_start_timesteps=0,
        eval_freq=8192,
        n_eval_episodes=5,
        csv_path="/kaggle/working/per_trace_eval_log.csv",
        verbose=1,
    ):
        super().__init__(verbose)
        self.all_traces = all_traces
        self.stage_name = stage_name
        self.stage_offset = stage_offset
        self.model_start_timesteps = model_start_timesteps
        self.eval_freq = int(eval_freq)
        self.n_eval_episodes = int(n_eval_episodes)
        self.csv_path = csv_path
        self.last_eval_step = 0

    def _on_step(self) -> bool:
        if self.num_timesteps - self.last_eval_step < self.eval_freq:
            return True

        self.last_eval_step = self.num_timesteps

        local_timesteps = self.num_timesteps - self.model_start_timesteps
        global_timesteps = self.stage_offset + local_timesteps

        rows = []

        for trace_idx, tr in enumerate(self.all_traces):
            env = OnlineBoutiqueScalingEnv(
                traces=self.all_traces,
                predictor=predictor,
                A=A_global,
                episode_len=min(EPISODE_LEN, len(tr["X"]) - 5),
                force_trace_idx=trace_idx,
            )

            returns = []
            mean_rewards = []
            slas = []
            costs = []
            cost_quads = []
            churns = []
            reps = []
            lat_vios = []
            err_vios = []
            cpu_vios = []

            for ep in range(self.n_eval_episodes):
                obs, _ = env.reset(seed=SEED + ep)
                done = False

                ep_rewards = []
                ep_sla = []
                ep_cost = []
                ep_cost_quad = []
                ep_churn = []
                ep_reps = []
                ep_lat = []
                ep_err = []
                ep_cpu = []

                while not done:
                    action, _ = self.model.predict(obs, deterministic=True)
                    obs, reward, terminated, truncated, info = env.step(action)
                    done = terminated or truncated

                    ep_rewards.append(reward)
                    ep_sla.append(info.get("sla", np.nan))
                    ep_cost.append(info.get("resource_cost", np.nan))
                    ep_cost_quad.append(info.get("resource_cost_quad", np.nan))
                    ep_churn.append(info.get("churn_cost", np.nan))
                    ep_reps.append(info.get("avg_effective_replicas", np.nan))
                    ep_lat.append(info.get("lat_violation", np.nan))
                    ep_err.append(info.get("err_violation", np.nan))
                    ep_cpu.append(info.get("cpu_violation", np.nan))

                returns.append(float(np.sum(ep_rewards)))
                mean_rewards.append(float(np.mean(ep_rewards)))
                slas.append(float(np.nanmean(ep_sla)))
                costs.append(float(np.nanmean(ep_cost)))
                cost_quads.append(float(np.nanmean(ep_cost_quad)))
                churns.append(float(np.nanmean(ep_churn)))
                reps.append(float(np.nanmean(ep_reps)))
                lat_vios.append(float(np.nanmean(ep_lat)))
                err_vios.append(float(np.nanmean(ep_err)))
                cpu_vios.append(float(np.nanmean(ep_cpu)))

            rows.append({
                "stage": self.stage_name,
                "timesteps": int(global_timesteps),
                "trace": tr["name"],
                "mean_return": float(np.mean(returns)),
                "std_return": float(np.std(returns)),
                "mean_reward": float(np.mean(mean_rewards)),
                "mean_sla": float(np.mean(slas)),
                "mean_resource_cost": float(np.mean(costs)),
                "mean_resource_cost_quad": float(np.mean(cost_quads)),
                "mean_churn": float(np.mean(churns)),
                "mean_replicas": float(np.mean(reps)),
                "mean_lat_violation": float(np.mean(lat_vios)),
                "mean_err_violation": float(np.mean(err_vios)),
                "mean_cpu_violation": float(np.mean(cpu_vios)),
            })

        df_row = pd.DataFrame(rows)

        if os.path.exists(self.csv_path):
            df_row.to_csv(self.csv_path, mode="a", header=False, index=False)
        else:
            df_row.to_csv(self.csv_path, index=False)

        if self.verbose:
            print("\n=== Per-trace fixed evaluation ===")
            print(
                df_row[
                    [
                        "stage",
                        "timesteps",
                        "trace",
                        "mean_return",
                        "mean_sla",
                        "mean_resource_cost",
                        "mean_replicas",
                    ]
                ].to_string(index=False)
            )

        return True

# ============================================================
# HELPERS
# ============================================================

def filter_traces(traces, names, weights):
    if len(names) != len(weights):
        raise ValueError("trace_filter và weights phải cùng độ dài.")

    name_to_weight = dict(zip(names, weights))
    result = []

    for tr in traces:
        if tr["name"] in name_to_weight:
            tr_copy = dict(tr)
            tr_copy["weight"] = float(name_to_weight[tr["name"]])
            result.append(tr_copy)

    found_names = [r["name"] for r in result]
    missing = [n for n in names if n not in found_names]

    if missing:
        raise ValueError(f"Không tìm thấy traces: {missing}")

    weight_sum = sum(r["weight"] for r in result)

    if weight_sum <= 0:
        raise ValueError("Tổng weights phải > 0.")

    for r in result:
        r["weight"] /= weight_sum

    return result


def make_stage_env(active_traces, episode_len=EPISODE_LEN, n_envs=4, env_cls=OnlineBoutiqueScalingEnv):
    def _make():
        # Copy dict trace metadata cho từng env để callback blend không làm các env khác
        # hoặc eval env bị mutate ngoài ý muốn. Tensor X/E/Y vẫn chỉ là reference, không copy dữ liệu lớn.
        env_traces = [dict(tr) for tr in active_traces]

        env = env_cls(
            traces=env_traces,
            predictor=predictor,
            A=A_global,
            episode_len=episode_len,
        )
        return Monitor(env)

    return DummyVecEnv([_make for _ in range(n_envs)])

def update_ppo_hyperparams(model, lr, ent_coef):
    model.learning_rate = lr
    model.ent_coef = ent_coef

    if hasattr(model, "_setup_lr_schedule"):
        model._setup_lr_schedule()

    for param_group in model.policy.optimizer.param_groups:
        param_group["lr"] = lr

# ============================================================
# EVALUATION HELPERS
# ============================================================

def evaluate_policy_by_trace(
    model,
    traces,
    label="PPO",
    n_episodes_per_trace=10,
    episode_len=EPISODE_LEN,
):
    rows = []

    for trace_idx, tr in enumerate(traces):
        env = OnlineBoutiqueScalingEnv(
            traces=traces,
            predictor=predictor,
            A=A_global,
            episode_len=episode_len,
            force_trace_idx=trace_idx,
        )

        for ep in range(n_episodes_per_trace):
            obs, _ = env.reset(seed=SEED + ep)

            done = False

            rewards = []
            slas = []
            costs = []
            cost_quads = []
            churns = []
            reps = []
            lats = []
            lat_crits = []
            errs = []
            cpus = []
            cpu_pressures = []

            while not done:
                action, _ = model.predict(obs, deterministic=True)

                obs, reward, terminated, truncated, info = env.step(action)

                done = terminated or truncated

                rewards.append(reward)
                slas.append(info.get("sla", float("nan")))
                costs.append(info.get("resource_cost", float("nan")))
                cost_quads.append(info.get("resource_cost_quad", float("nan")))
                churns.append(info.get("churn_cost", float("nan")))
                reps.append(info.get("avg_effective_replicas", float("nan")))
                lats.append(info.get("lat_violation", float("nan")))
                lat_crits.append(info.get("lat_critical_violation", float("nan")))
                errs.append(info.get("err_violation", float("nan")))
                cpus.append(info.get("cpu_violation", float("nan")))
                cpu_pressures.append(info.get("avg_cpu_pressure", float("nan")))

            rows.append(
                {
                    "policy": label,
                    "trace": tr["name"],
                    "episode": ep,
                    "return": float(np.sum(rewards)),
                    "mean_reward": float(np.mean(rewards)),
                    "mean_sla": float(np.nanmean(slas)),
                    "mean_resource_cost": float(np.nanmean(costs)),
                    "mean_resource_cost_quad": float(np.nanmean(cost_quads)),
                    "mean_churn": float(np.nanmean(churns)),
                    "mean_replicas": float(np.nanmean(reps)),
                    "mean_lat_violation": float(np.nanmean(lats)),
                    "mean_lat_critical_violation": float(np.nanmean(lat_crits)),
                    "mean_err_violation": float(np.nanmean(errs)),
                    "mean_cpu_violation": float(np.nanmean(cpus)),
                    "mean_cpu_pressure": float(np.nanmean(cpu_pressures)),
                }
            )

    return pd.DataFrame(rows)


def score_eval_dataframe(df, policy_name):
    sub = df[df["policy"] == policy_name].copy()
    if len(sub) == 0:
        return -np.inf

    mean_reward = sub["mean_reward"].mean()

    normal = sub[sub["trace"] == "normal"]
    anomaly = sub[sub["trace"].isin(["anomaly_main", "anomaly_edge", "anomaly_cascade"])]

    normal_cost = normal["mean_resource_cost"].mean()
    anomaly_sla = anomaly["mean_sla"].mean()
    cascade_sla = sub[sub["trace"] == "anomaly_cascade"]["mean_sla"].mean()

    penalty = 0.0
    penalty += max(0.0, normal_cost - 0.08) * 0.20
    penalty += max(0.0, anomaly_sla - 0.13) * 0.30
    penalty += max(0.0, cascade_sla - 0.19) * 0.35

    return float(mean_reward - penalty)

def evaluate_and_select_best_checkpoint(traces, save_dir="/kaggle/working"):
    """
    Chỉ chọn model trong stage cuối.
    Không xét stage 1/2/3 để tránh chọn model chưa học đủ curriculum.
    """
    last_stage_no = len(CURRICULUM)

    candidate_paths = []

    stage_final = Path(save_dir) / f"curriculum_stage_{last_stage_no}_final.zip"
    if stage_final.exists():
        candidate_paths.append((f"stage_{last_stage_no}_final", str(stage_final)))

    stage_best = Path(save_dir) / f"curriculum_stage_{last_stage_no}_best" / "best_model.zip"
    if stage_best.exists():
        candidate_paths.append((f"stage_{last_stage_no}_best", str(stage_best)))

    final_path = Path(save_dir) / "ppo_curriculum_final.zip"
    if final_path.exists():
        candidate_paths.append(("ppo_curriculum_final", str(final_path)))

    if len(candidate_paths) == 0:
        print("⚠ Không tìm thấy checkpoint stage cuối để select.")
        return None, None

    rows = []
    best_score = -np.inf
    best_name = None
    best_model = None

    eval_traces = filter_traces(
        traces,
        ["normal", "anomaly_main", "anomaly_edge", "anomaly_cascade"],
        [0.25, 0.25, 0.25, 0.25],
    )
    dummy_env = make_stage_env(eval_traces, n_envs=1)

    for name, path in candidate_paths:
        print(f"\n[*] Evaluating final-stage checkpoint: {name} -> {path}")
        model = PPO.load(path, env=dummy_env, device=PPO_DEVICE)

        df = evaluate_policy_by_trace(
            model,
            traces,
            label=name,
            n_episodes_per_trace=10,
        )

        score = score_eval_dataframe(df, name)

        rows.append({
            "checkpoint": name,
            "path": path,
            "score": score,
            "mean_reward": df[df["policy"] == name]["mean_reward"].mean(),
            "mean_sla": df[df["policy"] == name]["mean_sla"].mean(),
            "mean_cost": df[df["policy"] == name]["mean_resource_cost"].mean(),
            "normal_cost": df[
                (df["policy"] == name) & (df["trace"] == "normal")
            ]["mean_resource_cost"].mean(),
            "cascade_sla": df[
                (df["policy"] == name) & (df["trace"] == "anomaly_cascade")
            ]["mean_sla"].mean(),
        })

        if score > best_score:
            best_score = score
            best_name = name
            best_model = model

    result_df = pd.DataFrame(rows).sort_values("score", ascending=False)
    result_path = Path(save_dir) / "checkpoint_selection_summary.csv"
    result_df.to_csv(result_path, index=False)

    print("\n=== FINAL-STAGE CHECKPOINT SELECTION SUMMARY ===")
    print(result_df.to_string(index=False))
    print(f"\n✓ Saved: {result_path}")

    if best_model is not None:
        selected_path = Path(save_dir) / "ppo_selected_best"
        best_model.save(str(selected_path))
        print(f"\n✓ Selected best final-stage checkpoint: {best_name}")
        print(f"✓ Saved selected model: {selected_path}.zip")

    return best_model, result_df

# ============================================================
# PLOT TRAINING CURVES
# ============================================================

def plot_curriculum_training_curves(
    log_csv="/kaggle/working/curriculum_training_log.csv",
    smooth_window=5,
):
    if not os.path.exists(log_csv):
        print(f"Không tìm thấy: {log_csv}")
        return

    df = pd.read_csv(log_csv)

    print("=== TRAINING LOG HEAD ===")
    print(df.head())

    print("\n=== TRAINING LOG TAIL ===")
    print(df.tail())

    stage_labels = []

    for stage_name in df["stage"].unique():
        sub = df[df["stage"] == stage_name]
        stage_labels.append(
            (
                sub["timesteps"].min(),
                sub["timesteps"].max(),
                stage_name,
            )
        )

    plot_cols = [
        "mean_episode_reward",
        "mean_sla",
        "mean_lat_violation",
        "mean_lat_critical_violation",
        "mean_err_violation",
        "mean_cpu_violation",
        "mean_cpu_pressure",
        "mean_demand_cpu",
        "mean_demand_rps",
        "mean_demand_err",
        "mean_demand_lat",
        "mean_cpu_gain",
        "mean_lat_gain",
        "mean_err_gain",
        "mean_learned_delta_cpu",
        "mean_learned_delta_rps",
        "mean_learned_delta_err",
        "mean_learned_delta_lat",
        "mean_resource_cost",
        "mean_resource_cost_quad",
        "mean_churn",
        "mean_effective_replicas",
        "value_loss",
        "policy_gradient_loss",
        "entropy_loss",
        "approx_kl",
        "explained_variance",
        "mean_cpu_soft_violation",
        "mean_cpu_hard_violation",
    ]

    for col in plot_cols:
        if col not in df.columns:
            continue

        valid = df[["timesteps", "stage", col]].copy()

        # Các metric PPO train/loss thường bị NaN ở đầu stage,
        # đặc biệt S1 vì SB3 chưa kịp ghi train/value_loss, approx_kl...
        loss_like_cols = [
            "value_loss",
            "loss",
            "policy_gradient_loss",
            "entropy_loss",
            "approx_kl",
            "clip_fraction",
            "explained_variance",
        ]

        if col in loss_like_cols:
            # Forward-fill trong từng stage để không làm mất cả stage khi có NaN rải rác.
            valid[col] = valid.groupby("stage")[col].ffill()

            # Sau ffill, nếu stage vẫn toàn NaN thì bỏ stage đó,
            # vì không có dữ liệu thật để vẽ.
            valid = valid.dropna(subset=[col])
        else:
            valid = valid.dropna(subset=[col])

        if len(valid) == 0:
            print(f"Skip {col}: no valid data.")
            continue

        plt.figure(figsize=(13, 4))

        plotted_stages = set()

        for stage_name in df["stage"].unique():
            sub = valid[valid["stage"] == stage_name].copy()

            if len(sub) == 0:
                print(f"  Note: {col} has no valid data for {stage_name}.")
                continue

            plotted_stages.add(stage_name)

            smooth = (
                sub[col]
                .rolling(window=smooth_window, min_periods=1)
                .mean()
            )

            plt.plot(
                sub["timesteps"],
                sub[col],
                alpha=0.25,
                linewidth=1,
            )

            plt.plot(
                sub["timesteps"],
                smooth,
                linewidth=2,
                label=stage_name,
            )

        for start, _, _ in stage_labels[1:]:
            plt.axvline(
                x=start,
                linestyle="--",
                alpha=0.6,
                linewidth=1,
            )

        y_min, y_max = plt.ylim()

        for start, end, sname in stage_labels:
            plt.text(
                (start + end) / 2,
                y_max,
                sname.split(" - ")[0],
                ha="center",
                va="bottom",
                fontsize=8,
            )

        missing_stages = [
            sname for _, _, sname in stage_labels
            if sname not in plotted_stages
        ]

        if missing_stages:
            plt.figtext(
                0.01,
                0.01,
                "No valid data for: " + ", ".join(
                    [s.split(" - ")[0] for s in missing_stages]
                ),
                fontsize=8,
                ha="left",
            )

        plt.xlabel("Global timesteps")
        plt.ylabel(col)
        plt.title(f"Curriculum: {col}")
        plt.grid(True, alpha=0.3)
        plt.legend(fontsize=8, ncol=2)
        plt.tight_layout()

        save_path = OUTPUT_DIR / f"ppo_curve_{col}.png"
        plt.savefig(save_path, dpi=150)
        plt.show()

        print(f"  Saved: {save_path}")

def plot_per_trace_eval(
    csv_path="/kaggle/working/per_trace_eval_log.csv",
    smooth_window=3,
):
    """
    Vẽ learning curve đánh giá cố định từng trace.
    Đây là biểu đồ nên dùng chính trong báo cáo curriculum learning.
    """
    if not os.path.exists(csv_path):
        print(f"Không tìm thấy: {csv_path}")
        return

    df = pd.read_csv(csv_path)

    if len(df) == 0:
        print(f"File rỗng: {csv_path}")
        return

    print("=== PER-TRACE EVAL HEAD ===")
    print(df.head())

    print("\n=== PER-TRACE EVAL TAIL ===")
    print(df.tail())

    stage_labels = []
    for stage_name in df["stage"].unique():
        sub = df[df["stage"] == stage_name]
        stage_labels.append((sub["timesteps"].min(), sub["timesteps"].max(), stage_name))

    plot_cols = [
        "mean_return",
        "mean_reward",
        "mean_sla",
        "mean_resource_cost",
        "mean_resource_cost_quad",
        "mean_churn",
        "mean_replicas",
        "mean_lat_violation",
        "mean_err_violation",
        "mean_cpu_violation",
    ]

    for col in plot_cols:
        if col not in df.columns:
            continue

        valid = df[["timesteps", "stage", "trace", col]].dropna()
        if len(valid) == 0:
            continue

        plt.figure(figsize=(13, 4))

        for trace_name in valid["trace"].unique():
            sub = valid[valid["trace"] == trace_name].copy()
            sub = sub.sort_values("timesteps")

            smooth = sub[col].rolling(window=smooth_window, min_periods=1).mean()
            plt.plot(sub["timesteps"], smooth, linewidth=2, label=trace_name)

        for start, _, _ in stage_labels[1:]:
            plt.axvline(x=start, linestyle="--", alpha=0.6, linewidth=1)

        y_max = plt.ylim()[1]
        for start, end, sname in stage_labels:
            plt.text(
                (start + end) / 2,
                y_max,
                sname.split(" - ")[0],
                ha="center",
                va="bottom",
                fontsize=8,
            )

        plt.xlabel("Global timesteps")
        plt.ylabel(col)
        plt.title(f"Per-trace curriculum evaluation: {col}")
        plt.grid(True, alpha=0.3)
        plt.legend(fontsize=8, ncol=2)
        plt.tight_layout()

        save_path = OUTPUT_DIR / f"per_trace_curve_{col}.png"
        plt.savefig(save_path, dpi=150)
        plt.show()

        print(f"  Saved: {save_path}")



def generate_per_trace_eval_log_after_training(
    model,
    traces,
    csv_path="/kaggle/working/per_trace_eval_log_after_train.csv",
    n_episodes_per_trace=10,
):
    """
    Optional: chạy SAU khi train xong để có bảng/biểu đồ per-trace.
    Không dùng hàm này trong callback training vì sẽ làm chậm và gây nhiễu trajectory.
    """
    rows = []

    for trace_idx, tr in enumerate(traces):
        env = OnlineBoutiqueScalingEnv(
            traces=traces,
            predictor=predictor,
            A=A_global,
            episode_len=min(EPISODE_LEN, len(tr["X"]) - 5),
            force_trace_idx=trace_idx,
        )

        for ep in range(n_episodes_per_trace):
            obs, _ = env.reset(seed=SEED + ep)
            done = False

            rewards = []
            slas = []
            costs = []
            cost_quads = []
            churns = []
            reps = []
            lat_vios = []
            err_vios = []
            cpu_vios = []

            while not done:
                action, _ = model.predict(obs, deterministic=True)
                obs, reward, terminated, truncated, info = env.step(action)
                done = terminated or truncated

                rewards.append(reward)
                slas.append(info.get("sla", np.nan))
                costs.append(info.get("resource_cost", np.nan))
                cost_quads.append(info.get("resource_cost_quad", np.nan))
                churns.append(info.get("churn_cost", np.nan))
                reps.append(info.get("avg_effective_replicas", np.nan))
                lat_vios.append(info.get("lat_violation", np.nan))
                err_vios.append(info.get("err_violation", np.nan))
                cpu_vios.append(info.get("cpu_violation", np.nan))

            rows.append({
                "trace": tr["name"],
                "episode": ep,
                "mean_return": float(np.sum(rewards)),
                "mean_reward": float(np.mean(rewards)),
                "mean_sla": float(np.nanmean(slas)),
                "mean_resource_cost": float(np.nanmean(costs)),
                "mean_resource_cost_quad": float(np.nanmean(cost_quads)),
                "mean_churn": float(np.nanmean(churns)),
                "mean_replicas": float(np.nanmean(reps)),
                "mean_lat_violation": float(np.nanmean(lat_vios)),
                "mean_err_violation": float(np.nanmean(err_vios)),
                "mean_cpu_violation": float(np.nanmean(cpu_vios)),
            })

    df = pd.DataFrame(rows)
    df.to_csv(csv_path, index=False)
    print(f"✓ Saved optional per-trace eval: {csv_path}")
    return df

# ============================================================
# CURRICULUM TRAINING LOOP
# ============================================================

def run_curriculum(
    traces,
    predictor,
    A_global,
    curriculum,
    save_dir="/kaggle/working",
):
    print("=" * 70)
    print("CURRICULUM LEARNING — PPO AUTOSCALING + SHORT SMOOTH BLEND")
    print(f"Stages: {len(curriculum)}")
    print(f"Max train steps : {sum(s['timesteps'] for s in curriculum):,}")
    print(f"Blend warmup    : {BLEND_WARMUP_EVALS} eval intervals at each transition")
    print("=" * 70)

    os.makedirs(save_dir, exist_ok=True)

    log_path = os.path.join(save_dir, "curriculum_training_log.csv")
    per_trace_log_path = os.path.join(save_dir, "per_trace_eval_log.csv")

    # Xóa log cũ để tránh nối nhầm kết quả nhiều lần chạy.
    if os.path.exists(log_path):
        os.remove(log_path)

    if os.path.exists(per_trace_log_path):
        os.remove(per_trace_log_path)

    ppo_model = None
    global_offset = 0

    for stage_idx, stage in enumerate(curriculum):
        stage_no = stage_idx + 1

        print("\n" + "-" * 70)
        print(f"Stage {stage_no}/{len(curriculum)}: {stage.get('name', f'Stage {stage_no}')}")
        print(f"  {stage.get('desc', '')}")
        print(f"  Traces : {list(zip(stage['trace_filter'], stage['weights']))}")
        print(
            f"  Max steps: {stage['timesteps']:,} | "
            f"LR: {stage['lr']} | "
            f"Ent: {stage['ent_coef']}"
        )
        print(
            f"  Early stop: min_evals={stage.get('early_stop_min_evals', 10)}, "
            f"patience={stage.get('early_stop_patience', 12)}, "
            f"eval_freq={stage.get('eval_freq', 4096)}"
        )
        if stage_idx > 0:
            print(
                f"  Smooth blend: {curriculum[stage_idx - 1]['name']} -> {stage['name']} "
                f"trong {BLEND_WARMUP_EVALS} lần cập nhật"
            )
        print("-" * 70)

        active_traces = filter_traces(
            traces,
            stage["trace_filter"],
            stage["weights"],
        )

        for tr in active_traces:
            print(
                f"  ✓ {tr['name']}: "
                f"weight={tr['weight']:.3f}, "
                f"windows={len(tr['X'])}"
            )

        train_env = make_stage_env(active_traces, n_envs=4)
        eval_env = make_stage_env(active_traces, n_envs=1)

        if ppo_model is None:
            ppo_model = PPO(
                policy="MlpPolicy",
                env=train_env,
                learning_rate=stage["lr"],

                # Với n_envs=4:
                # rollout thật = n_steps * n_envs = 1024 * 4 = 4096
                n_steps=1024,
                batch_size=512,
                n_epochs=6,

                gamma=0.99,
                gae_lambda=0.95,
                clip_range=0.2,
                ent_coef=stage["ent_coef"],
                vf_coef=0.60,
                max_grad_norm=0.3,

                # Tránh PPO update quá mạnh khi reward dao động.
                target_kl=0.015,

                verbose=0,
                device=PPO_DEVICE,
                seed=SEED,
                policy_kwargs=dict(
                    net_arch=dict(
                        pi=[256, 256],
                        vf=[256, 256],
                    )
                ),
                tensorboard_log=os.path.join(save_dir, "tensorboard_logs"),
            )
        else:
            # Giữ timeline liên tục sau khi load best_model của stage trước.
            ppo_model.num_timesteps = int(global_offset)
            ppo_model.set_env(train_env)
            update_ppo_hyperparams(
                ppo_model,
                stage["lr"],
                stage["ent_coef"],
            )

        stage_save_path = os.path.join(
            save_dir,
            f"curriculum_stage_{stage_no}_best",
        )
        os.makedirs(stage_save_path, exist_ok=True)

        model_start_timesteps = int(ppo_model.num_timesteps)

        callbacks = []

        # Blend ngắn ở đầu mỗi stage, trừ stage 1.
        # Không thêm steps riêng; blend diễn ra trong chính stage training.
        if stage_idx > 0:
            callbacks.append(
                SmoothBlendCallback(
                    prev_stage=curriculum[stage_idx - 1],
                    cur_stage=stage,
                    blend_warmup_evals=BLEND_WARMUP_EVALS,
                    eval_freq=stage.get("eval_freq", 4096),
                    verbose=1,
                )
            )

        stop_cb = StopTrainingOnNoModelImprovement(
            max_no_improvement_evals=stage.get("early_stop_patience", 12),
            min_evals=stage.get("early_stop_min_evals", 10),
            verbose=1,
        )

        eval_cb = EvalCallback(
            eval_env,
            eval_freq=max(stage.get("eval_freq", 4096) // 4, 1024),
            n_eval_episodes=stage.get("n_eval_episodes", 10),
            best_model_save_path=stage_save_path,
            log_path=stage_save_path,
            callback_after_eval=stop_cb,
            verbose=1,
        )

        log_cb = CurriculumTrainingLogCallback(
            stage_name=stage["name"],
            stage_offset=global_offset,
            model_start_timesteps=model_start_timesteps,
            csv_path=log_path,
            log_freq=4096,
            verbose=1,
        )

        callbacks.extend([eval_cb, log_cb])

        ppo_model.learn(
            total_timesteps=stage["timesteps"],
            callback=CallbackList(callbacks),
            reset_num_timesteps=False,
            progress_bar=False,
        )

        actual_stage_steps = int(ppo_model.num_timesteps) - model_start_timesteps

        # Nếu SB3 bị dừng sớm thì actual_stage_steps vẫn phản ánh số bước thật.
        # Nếu bị lỗi timeline do load checkpoint, không để global_offset bị âm hoặc quá nhỏ.
        actual_stage_steps = max(0, actual_stage_steps)
        global_offset += actual_stage_steps

        print(
            f"✓ Stage {stage_no} actual trained steps: "
            f"{actual_stage_steps:,}"
        )
        print(f"✓ Global offset now: {global_offset:,}")

        best_path = os.path.join(stage_save_path, "best_model.zip")

        if os.path.exists(best_path):
            ppo_model = PPO.load(
                best_path,
                env=train_env,
                device=PPO_DEVICE,
            )

            update_ppo_hyperparams(
                ppo_model,
                stage["lr"],
                stage["ent_coef"],
            )

            # Quan trọng: best_model được lưu tại một checkpoint giữa stage,
            # nên num_timesteps trong file có thể thấp hơn global_offset.
            # Gán lại để stage sau log/tính offset không bị lệch.
            ppo_model.num_timesteps = int(global_offset)

            print(f"✓ Loaded best model stage {stage_no}")
        else:
            print(
                f"⚠ best_model.zip không tìm thấy stage {stage_no}, "
                "dùng model hiện tại."
            )
            ppo_model.num_timesteps = int(global_offset)

        stage_final = os.path.join(
            save_dir,
            f"curriculum_stage_{stage_no}_final",
        )

        ppo_model.save(stage_final)
        print(f"✓ Stage {stage_no} final saved: {stage_final}")

    final_path = os.path.join(save_dir, "ppo_curriculum_final")
    ppo_model.save(final_path)

    print("\n" + "=" * 70)
    print("✓ Curriculum training complete")
    print(f"✓ Final model: {final_path}")
    print(f"✓ Training log: {log_path}")
    print("=" * 70)

    # 1) Biểu đồ training tổng: dùng để xem PPO update/loss có ổn không.
    plot_curriculum_training_curves(log_path)

    return ppo_model


# ============================================================
# TRAIN PURE PPO BASELINE
# ============================================================

TRAIN_PURE_PPO_BASELINE = True
PURE_PPO_TOTAL_TIMESTEP_RATIO = 1.0


def run_pure_ppo_baseline(
    traces,
    predictor,
    A_global,
    curriculum,
    save_dir="/kaggle/working",
):
    """
    Train baseline PPO thuần để so với PPO chính.

    Pure PPO khác PPO chính ở 2 điểm quan trọng:
      1) Observation không có GNN forecast: chỉ dùng current metrics + replicas + cooldown.
      2) Env không gọi GAT-GRU predictor để sinh demand; dùng Y_norm thực tế từ trace.

    Giữ cùng action space, reward, cooldown, scale delay, curriculum để so sánh công bằng.
    """
    print("=" * 70)
    print("PURE PPO BASELINE — NO GNN FORECAST")
    print(f"Stages: {len(curriculum)}")
    print(f"Max train steps : {sum(s['timesteps'] for s in curriculum):,}")
    print("=" * 70)

    os.makedirs(save_dir, exist_ok=True)

    log_path = os.path.join(save_dir, "pure_ppo_training_log.csv")
    if os.path.exists(log_path):
        os.remove(log_path)

    pure_ppo_model = None
    global_offset = 0

    for stage_idx, stage in enumerate(curriculum):
        stage_no = stage_idx + 1
        stage_steps = int(stage["timesteps"] * PURE_PPO_TOTAL_TIMESTEP_RATIO)

        print("\n" + "-" * 70)
        print(f"Pure PPO Stage {stage_no}/{len(curriculum)}: {stage.get('name', f'Stage {stage_no}')}")
        print(f"  Traces : {list(zip(stage['trace_filter'], stage['weights']))}")
        print(f"  Max steps: {stage_steps:,} | LR: {stage['lr']} | Ent: {stage['ent_coef']}")
        print("-" * 70)

        active_traces = filter_traces(
            traces,
            stage["trace_filter"],
            stage["weights"],
        )

        train_env = make_stage_env(
            active_traces,
            n_envs=4,
            env_cls=PurePPOScalingEnv,
        )
        eval_env = make_stage_env(
            active_traces,
            n_envs=1,
            env_cls=PurePPOScalingEnv,
        )

        if pure_ppo_model is None:
            pure_ppo_model = PPO(
                policy="MlpPolicy",
                env=train_env,
                learning_rate=stage["lr"],
                n_steps=1024,
                batch_size=512,
                n_epochs=6,
                gamma=0.99,
                gae_lambda=0.95,
                clip_range=0.2,
                ent_coef=stage["ent_coef"],
                vf_coef=0.60,
                max_grad_norm=0.3,
                target_kl=0.015,
                verbose=0,
                device=PPO_DEVICE,
                seed=SEED + 777,
                policy_kwargs=dict(
                    net_arch=dict(
                        pi=[256, 256],
                        vf=[256, 256],
                    )
                ),
                tensorboard_log=os.path.join(save_dir, "tensorboard_logs_pure_ppo"),
            )
        else:
            pure_ppo_model.num_timesteps = int(global_offset)
            pure_ppo_model.set_env(train_env)
            update_ppo_hyperparams(
                pure_ppo_model,
                stage["lr"],
                stage["ent_coef"],
            )

        stage_save_path = os.path.join(
            save_dir,
            f"pure_ppo_stage_{stage_no}_best",
        )
        os.makedirs(stage_save_path, exist_ok=True)

        model_start_timesteps = int(pure_ppo_model.num_timesteps)

        stop_cb = StopTrainingOnNoModelImprovement(
            max_no_improvement_evals=stage.get("early_stop_patience", 12),
            min_evals=stage.get("early_stop_min_evals", 10),
            verbose=1,
        )

        eval_cb = EvalCallback(
            eval_env,
            eval_freq=max(stage.get("eval_freq", 4096) // 4, 1024),
            n_eval_episodes=stage.get("n_eval_episodes", 10),
            best_model_save_path=stage_save_path,
            log_path=stage_save_path,
            callback_after_eval=stop_cb,
            verbose=1,
        )

        log_cb = CurriculumTrainingLogCallback(
            stage_name="Pure " + stage["name"],
            stage_offset=global_offset,
            model_start_timesteps=model_start_timesteps,
            csv_path=log_path,
            log_freq=4096,
            verbose=1,
        )

        pure_ppo_model.learn(
            total_timesteps=stage_steps,
            callback=CallbackList([eval_cb, log_cb]),
            reset_num_timesteps=False,
            progress_bar=False,
        )

        actual_stage_steps = int(pure_ppo_model.num_timesteps) - model_start_timesteps
        actual_stage_steps = max(0, actual_stage_steps)
        global_offset += actual_stage_steps

        best_path = os.path.join(stage_save_path, "best_model.zip")
        if os.path.exists(best_path):
            pure_ppo_model = PPO.load(
                best_path,
                env=train_env,
                device=PPO_DEVICE,
            )
            update_ppo_hyperparams(
                pure_ppo_model,
                stage["lr"],
                stage["ent_coef"],
            )
            pure_ppo_model.num_timesteps = int(global_offset)
            print(f"✓ Loaded Pure PPO best model stage {stage_no}")
        else:
            pure_ppo_model.num_timesteps = int(global_offset)
            print(f"⚠ Pure PPO best_model.zip không tìm thấy stage {stage_no}, dùng model hiện tại.")

        stage_final = os.path.join(save_dir, f"pure_ppo_stage_{stage_no}_final")
        pure_ppo_model.save(stage_final)
        print(f"✓ Pure PPO stage {stage_no} final saved: {stage_final}")

    final_path = os.path.join(save_dir, "pure_ppo_curriculum_final")
    pure_ppo_model.save(final_path)

    print("\n" + "=" * 70)
    print("✓ Pure PPO baseline training complete")
    print(f"✓ Final model: {final_path}")
    print(f"✓ Training log: {log_path}")
    print("=" * 70)

    plot_curriculum_training_curves(log_path)

    return pure_ppo_model

# ============================================================
# RUN
# ============================================================

ppo_model = run_curriculum(
    traces=traces,
    predictor=predictor,
    A_global=A_global,
    curriculum=CURRICULUM,
    save_dir="/kaggle/working",
)

# Chọn checkpoint tốt nhất giữa các checkpoint stage cuối.
# Phần này KHÔNG so sánh với baseline; baseline evaluation đã tách sang notebook/đoạn code riêng.
selected_model, checkpoint_summary = evaluate_and_select_best_checkpoint(
    traces=traces,
    save_dir="/kaggle/working",
)

# Train PPO thuần làm baseline: không dùng GNN forecast trong observation/action.
# Nếu muốn chạy nhanh để debug, có thể đặt TRAIN_PURE_PPO_BASELINE = False
# và load lại model đã train từ /kaggle/working/pure_ppo_curriculum_final.zip trong file evaluate.
if TRAIN_PURE_PPO_BASELINE:
    pure_ppo_model = run_pure_ppo_baseline(
        traces=traces,
        predictor=predictor,
        A_global=A_global,
        curriculum=CURRICULUM,
        save_dir="/kaggle/working",
    )
else:
    pure_ppo_model = None

print("\n✅ Hoàn tất training RL.")
print("  Final model    : /kaggle/working/ppo_curriculum_final.zip")
print("  Selected model : /kaggle/working/ppo_selected_best.zip")
print("  Log            : /kaggle/working/curriculum_training_log.csv")
print("  Per-trace eval : /kaggle/working/per_trace_eval_log.csv")
print("  Checkpoints    : /kaggle/working/checkpoint_selection_summary.csv")
print("\nBaseline comparison đã được tách riêng. Chạy đoạn hard evaluation riêng sau khi training xong.")


In [ ]:
import os
import matplotlib.pyplot as plt
import pandas as pd
import numpy as np
import random


# Action mapping follows the training env:
# action id 0..4 -> delta {-2,-1,0,+1,+2}
# If these globals already exist from the train notebook, the following assignments are no-ops in meaning.
ACTION_DIM = globals().get("ACTION_DIM", 2 * MAX_DELTA + 1)
HOLD_ACTION = globals().get("HOLD_ACTION", MAX_DELTA)


# ============================================================
# PURE PPO BASELINE ADAPTER
# ============================================================

PURE_OBS_FEATURES_PER_SERVICE = 6
PURE_OBS_COLS_FROM_MAIN = [0, 1, 2, 3, 4, 9]


def main_obs_to_pure_obs(obs):
    """
    Main env obs: [replica, cpu_now, rps_now, err_now, lat_now,
                   cpu_gnn, rps_gnn, err_gnn, lat_gnn, cooldown]
    Pure PPO obs: [replica, cpu_now, rps_now, err_now, lat_now, cooldown]
    """
    obs = np.asarray(obs, dtype=np.float32)
    state = obs.reshape(N_SERVICES, OBS_FEATURES_PER_SERVICE)
    pure_state = state[:, PURE_OBS_COLS_FROM_MAIN]
    return np.clip(pure_state, 0.0, 1.0).astype(np.float32).reshape(-1)


class PurePPOPolicyAdapter:
    """
    Cho phép Pure PPO được evaluate trong cùng OnlineBoutiqueScalingEnv với PPO chính,
    nhưng policy chỉ nhìn thấy 6 feature hiện tại và không nhìn thấy forecast GNN.
    """
    def __init__(self, model):
        self.model = model

    def predict(self, obs, deterministic=True):
        pure_obs = main_obs_to_pure_obs(obs)
        return self.model.predict(pure_obs, deterministic=deterministic)


def resolve_pure_ppo_model():
    """
    Ưu tiên dùng biến pure_ppo_model nếu vừa train trong notebook.
    Nếu không có, tự load checkpoint đã lưu từ train file.
    """
    if "pure_ppo_model" in globals() and pure_ppo_model is not None:
        return pure_ppo_model

    pure_path = "/kaggle/working/pure_ppo_curriculum_final.zip"
    if os.path.exists(pure_path):
        return PPO.load(pure_path, device=PPO_DEVICE)

    raise FileNotFoundError(
        "Không tìm thấy Pure PPO baseline. Hãy chạy phần train Pure PPO trong file train, "
        "hoặc đảm bảo tồn tại /kaggle/working/pure_ppo_curriculum_final.zip"
    )

# ============================================================
# REFERENCE SCALING POLICIES
# ============================================================

class HPAPolicy:
    """
    HPA-like baseline.
    Uses only current CPU metric.

    Action mapping:
      0=-2, 1=-1, 2=hold, 3=+1, 4=+2
    HPA is intentionally allowed to use +/-2 only for very strong signals,
    so the baseline remains simple but not artificially handicapped.
    """
    def __init__(self, target_cpu=0.70, down_threshold=0.35):
        self.target_cpu = target_cpu
        self.down_threshold = down_threshold

    def predict(self, obs, deterministic=True):
        state = obs.reshape(N_SERVICES, OBS_FEATURES_PER_SERVICE)

        cpu_now = state[:, 1]
        cooldown = state[:, 9]

        action = np.full(N_SERVICES, HOLD_ACTION, dtype=np.int64)

        for i in range(N_SERVICES):
            if cooldown[i] > 0:
                action[i] = HOLD_ACTION
            elif cpu_now[i] > 0.90:
                action[i] = min(HOLD_ACTION + 2, ACTION_DIM - 1)
            elif cpu_now[i] > self.target_cpu:
                action[i] = min(HOLD_ACTION + 1, ACTION_DIM - 1)
            elif cpu_now[i] < 0.18:
                action[i] = max(HOLD_ACTION - 2, 0)
            elif cpu_now[i] < self.down_threshold:
                action[i] = max(HOLD_ACTION - 1, 0)

        return action, None


class ReactiveRuleBasedPolicy:
    """
    Uses only current metrics. Allows aggressive +/-2 under critical pressure.
    """
    def predict(self, obs, deterministic=True):
        state = obs.reshape(N_SERVICES, OBS_FEATURES_PER_SERVICE)

        cpu_now = state[:, 1]
        rps_now = state[:, 2]
        err_now = state[:, 3]
        lat_now = state[:, 4]
        cooldown = state[:, 9]

        action = np.full(N_SERVICES, HOLD_ACTION, dtype=np.int64)

        for i in range(N_SERVICES):
            if cooldown[i] > 0:
                action[i] = HOLD_ACTION
            elif (
                lat_now[i] > LAT_CRITICAL_THR_NORM
                or err_now[i] > 4 * ERR_THR_NORM
                or cpu_now[i] > 0.90
            ):
                action[i] = min(HOLD_ACTION + 2, ACTION_DIM - 1)
            elif (
                lat_now[i] > LAT_THR_NORM
                or err_now[i] > 2 * ERR_THR_NORM
                or cpu_now[i] > CPU_THR_NORM
            ):
                action[i] = min(HOLD_ACTION + 1, ACTION_DIM - 1)
            elif (
                cpu_now[i] < 0.15
                and rps_now[i] < 0.20
                and lat_now[i] < LAT_THR_NORM * 0.35
                and err_now[i] < ERR_THR_NORM
            ):
                action[i] = max(HOLD_ACTION - 2, 0)
            elif (
                cpu_now[i] < 0.25
                and rps_now[i] < 0.30
                and lat_now[i] < LAT_THR_NORM * 0.5
                and err_now[i] < ERR_THR_NORM
            ):
                action[i] = max(HOLD_ACTION - 1, 0)

        return action, None


class DLAssistedRuleBasedPolicy:
    """
    Uses GAT-GRU forecast metrics from observation. Allows aggressive +/-2 under forecasted critical pressure.
    """
    def predict(self, obs, deterministic=True):
        state = obs.reshape(N_SERVICES, OBS_FEATURES_PER_SERVICE)

        cpu_pred = state[:, 5]
        rps_pred = state[:, 6]
        err_pred = state[:, 7]
        lat_pred = state[:, 8]
        cooldown = state[:, 9]

        action = np.full(N_SERVICES, HOLD_ACTION, dtype=np.int64)

        for i in range(N_SERVICES):
            if cooldown[i] > 0:
                action[i] = HOLD_ACTION
            elif (
                lat_pred[i] > LAT_CRITICAL_THR_NORM
                or err_pred[i] > 4 * ERR_THR_NORM
                or cpu_pred[i] > 0.90
            ):
                action[i] = min(HOLD_ACTION + 2, ACTION_DIM - 1)
            elif (
                lat_pred[i] > LAT_THR_NORM
                or err_pred[i] > 2 * ERR_THR_NORM
                or cpu_pred[i] > CPU_THR_NORM
            ):
                action[i] = min(HOLD_ACTION + 1, ACTION_DIM - 1)
            elif (
                cpu_pred[i] < 0.15
                and rps_pred[i] < 0.20
                and lat_pred[i] < LAT_THR_NORM * 0.35
                and err_pred[i] < ERR_THR_NORM
            ):
                action[i] = max(HOLD_ACTION - 2, 0)
            elif (
                cpu_pred[i] < 0.25
                and rps_pred[i] < 0.30
                and lat_pred[i] < LAT_THR_NORM * 0.5
                and err_pred[i] < ERR_THR_NORM
            ):
                action[i] = max(HOLD_ACTION - 1, 0)

        return action, None


class NoScalingPolicy:
    """
    Always hold.
    """
    def predict(self, obs, deterministic=True):
        return np.full(N_SERVICES, HOLD_ACTION, dtype=np.int64), None


# ============================================================
# REFERENCE ENV WITHOUT PPO CAPACITY MODEL
# ============================================================

class HandcraftedCapacityEvalEnv(OnlineBoutiqueScalingEnv):
    """
    Evaluation-only env for capacity-model ablation.

    It does NOT use the learned capacity model. It keeps reset/step/delay/cooldown/reward
    identical to OnlineBoutiqueScalingEnv, but the transition is pure physics/queueing:
      CPU/LAT = queueing-inspired capacity rule
      RPS     = persistence
      ERR     = same overload-risk rule as train env

    This class is NOT used for the main PPO-vs-reference-policy comparison.
    """
    def _apply_capacity_model(self, demand_pred, replicas):
        before = np.clip(demand_pred.astype(np.float32), 0.0, 1.0)

        r_old = np.clip(self.prev_effective_replicas.astype(np.float32), R_MIN, R_MAX)
        r_new = np.clip(replicas.astype(np.float32), R_MIN, R_MAX)

        cpu_demand = before[:, 0]
        rps_demand = before[:, 1]
        err_demand = before[:, 2]
        lat_demand = before[:, 3]

        effective_delta = r_new - r_old
        ratio = np.clip(r_new / np.maximum(r_old, 1.0), 0.1, 10.0)

        cpu_gain = np.power(ratio, CPU_ALPHA)
        lat_gain = np.power(ratio, LAT_BETA)

        cpu_pressure = cpu_demand / np.maximum(cpu_gain, 1e-6)
        lat_pressure = lat_demand * (
            (LAT_SCALABLE_FRAC / np.maximum(lat_gain, 1e-6))
            + (1.0 - LAT_SCALABLE_FRAC)
        )

        cpu_pressure = np.clip(cpu_pressure, 0.0, 1.0)
        lat_pressure = np.clip(lat_pressure, 0.0, 1.0)

        scale_up_mask = effective_delta > 0
        scale_down_mask = effective_delta < 0
        hold_mask = effective_delta == 0

        cpu_pressure[scale_up_mask] = np.minimum(cpu_pressure[scale_up_mask], cpu_demand[scale_up_mask])
        cpu_pressure[scale_down_mask] = np.maximum(cpu_pressure[scale_down_mask], cpu_demand[scale_down_mask])
        cpu_pressure[hold_mask] = cpu_demand[hold_mask]

        lat_pressure[scale_up_mask] = np.minimum(lat_pressure[scale_up_mask], lat_demand[scale_up_mask])
        lat_pressure[scale_down_mask] = np.maximum(lat_pressure[scale_down_mask], lat_demand[scale_down_mask])
        lat_pressure[hold_mask] = lat_demand[hold_mask]

        rps_pressure = np.clip(rps_demand, 0.0, 1.0)

        cpu_risk_before = np.maximum(
            0.0,
            (cpu_demand - CPU_HARD_THR_NORM) / (1.0 - CPU_HARD_THR_NORM + 1e-8),
        )
        cpu_risk_after = np.maximum(
            0.0,
            (cpu_pressure - CPU_HARD_THR_NORM) / (1.0 - CPU_HARD_THR_NORM + 1e-8),
        )

        lat_risk_before = np.maximum(
            0.0,
            (lat_demand - LAT_THR_NORM) / (1.0 - LAT_THR_NORM + 1e-8),
        )
        lat_risk_after = np.maximum(
            0.0,
            (lat_pressure - LAT_THR_NORM) / (1.0 - LAT_THR_NORM + 1e-8),
        )

        overload_risk_before = 0.5 * cpu_risk_before + 0.5 * lat_risk_before
        overload_risk_after = 0.5 * cpu_risk_after + 0.5 * lat_risk_after

        err_delta = (
            ERR_SCALABLE_FRAC
            * ERR_OVERLOAD_SCALE
            * (overload_risk_after - overload_risk_before)
        )
        err_pressure = np.clip(err_demand + err_delta, 0.0, 1.0)

        eps = 1e-6
        cpu_gain_eff = np.clip(cpu_demand / (cpu_pressure + eps), 0.0, MAX_CAPACITY_GAIN)
        lat_gain_eff = np.clip(lat_demand / (lat_pressure + eps), 0.0, MAX_CAPACITY_GAIN)
        err_gain_eff = np.clip(err_demand / (err_pressure + eps), 0.0, MAX_CAPACITY_GAIN)

        capacity_info = {
            "avg_demand_cpu": float(np.mean(cpu_demand)),
            "avg_demand_rps": float(np.mean(rps_demand)),
            "avg_demand_err": float(np.mean(err_demand)),
            "avg_demand_lat": float(np.mean(lat_demand)),
            "avg_cpu_gain": float(np.mean(cpu_gain_eff)),
            "avg_lat_gain": float(np.mean(lat_gain_eff)),
            "avg_err_gain": float(np.mean(err_gain_eff)),
            "avg_learned_delta_cpu": 0.0,
            "avg_learned_delta_rps": 0.0,
            "avg_learned_delta_err": 0.0,
            "avg_learned_delta_lat": 0.0,
            "avg_capacity_r_old": float(np.mean(r_old)),
            "avg_capacity_r_new": float(np.mean(r_new)),
            "capacity_source_id": 0.0,
        }

        return cpu_pressure, rps_pressure, err_pressure, lat_pressure, capacity_info

# ============================================================
# EVALUATION HELPERS
# ============================================================

def make_weighted_eval_env(trace_weights, episode_len=None, use_learned_capacity=True):
    """
    Main evaluation uses the same OnlineBoutiqueScalingEnv for all policies.
    Reference policies do not access the capacity model; it is only the simulator transition.
    """
    active_traces = filter_traces(
        traces,
        list(trace_weights.keys()),
        list(trace_weights.values()),
    )

    env_cls = OnlineBoutiqueScalingEnv if use_learned_capacity else HandcraftedCapacityEvalEnv

    env = env_cls(
        traces=active_traces,
        predictor=predictor,
        A=A_global,
        episode_len=episode_len or EPISODE_LEN,
        force_trace_idx=None,
    )

    env.set_trace_weights(trace_weights)

    return env


def evaluate_policy(model, env_raw, n_episodes=30, base_seed=42):
    rows = []

    for ep in range(n_episodes):
        random.seed(base_seed + ep)
        np.random.seed(base_seed + ep)

        obs, info = env_raw.reset(seed=base_seed + ep)

        done = False
        ep_reward = 0.0

        sla_list = []
        cost_list = []
        cost_quad_list = []
        churn_list = []
        replicas_list = []
        lat_list = []
        lat_critical_list = []
        err_list = []
        cpu_list = []
        cpu_pressure_list = []
        cpu_gain_list = []
        lat_gain_list = []
        err_gain_list = []
        cap_source_list = []

        trace_name = info.get("trace_name", "unknown")

        while not done:
            action, _ = model.predict(obs, deterministic=True)
            obs, reward, terminated, truncated, info = env_raw.step(action)

            done = terminated or truncated
            ep_reward += reward

            sla_list.append(info.get("sla", np.nan))
            cost_list.append(info.get("resource_cost", np.nan))
            cost_quad_list.append(info.get("resource_cost_quad", np.nan))
            churn_list.append(info.get("churn_cost", np.nan))
            replicas_list.append(info.get("avg_effective_replicas", np.nan))
            lat_list.append(info.get("lat_violation", np.nan))
            lat_critical_list.append(info.get("lat_critical_violation", np.nan))
            err_list.append(info.get("err_violation", np.nan))
            cpu_list.append(info.get("cpu_violation", np.nan))
            cpu_pressure_list.append(info.get("avg_cpu_pressure", np.nan))
            cpu_gain_list.append(info.get("avg_cpu_gain", np.nan))
            lat_gain_list.append(info.get("avg_lat_gain", np.nan))
            err_gain_list.append(info.get("avg_err_gain", np.nan))
            cap_source_list.append(info.get("capacity_source_id", np.nan))

        rows.append({
            "trace_sampled": trace_name,
            "return": float(ep_reward),
            "mean_reward": float(ep_reward / max(1, len(sla_list))),
            "mean_sla": float(np.nanmean(sla_list)),
            "p95_sla": float(np.nanpercentile(sla_list, 95)),
            "mean_cost": float(np.nanmean(cost_list)),
            "mean_cost_quad": float(np.nanmean(cost_quad_list)),
            "mean_churn": float(np.nanmean(churn_list)),
            "mean_replicas": float(np.nanmean(replicas_list)),
            "mean_lat_violation": float(np.nanmean(lat_list)),
            "p95_lat_violation": float(np.nanpercentile(lat_list, 95)),
            "mean_lat_critical_violation": float(np.nanmean(lat_critical_list)),
            "mean_err_violation": float(np.nanmean(err_list)),
            "mean_cpu_violation": float(np.nanmean(cpu_list)),
            "mean_cpu_pressure": float(np.nanmean(cpu_pressure_list)),
            "mean_cpu_gain": float(np.nanmean(cpu_gain_list)),
            "mean_lat_gain": float(np.nanmean(lat_gain_list)),
            "mean_err_gain": float(np.nanmean(err_gain_list)),
            "capacity_source_id": float(np.nanmean(cap_source_list)),
        })

    return pd.DataFrame(rows)


def evaluation_score(metrics):
    """
    Higher is better.
    Không dùng mean_reward để tránh double count vì reward đã chứa SLA/Cost/Churn.
    """
    return (
        -1.50 * metrics["mean_sla"]
        -0.80 * metrics["p95_sla"]
        -0.25 * metrics["mean_cost"]
        -0.10 * metrics["mean_churn"]
    )


# ============================================================
# EVALUATION SCENARIOS
# ============================================================

EVALUATION_SCENARIOS = [
    {
        "name": "NORMAL_COST_STRESS",
        "weights": {
            "normal": 0.80,
            "anomaly_main": 0.10,
            "anomaly_edge": 0.10,
        },
        "episodes": 40,
        "episode_len": EPISODE_LEN,
        "desc": "Predominantly normal traffic with limited anomaly exposure; evaluates resource efficiency while maintaining service quality.",
    },
    {
        "name": "ANOMALY_HEAVY",
        "weights": {
            "normal": 0.10,
            "anomaly_main": 0.45,
            "anomaly_edge": 0.35,
            "anomaly_cascade": 0.10,
        },
        "episodes": 40,
        "episode_len": EPISODE_LEN,
        "desc": "Workload emphasizing main and edge anomaly conditions.",
    },
    {
        "name": "CASCADE_HEAVY",
        "weights": {
            "normal": 0.05,
            "anomaly_main": 0.05,
            "anomaly_edge": 0.15,
            "anomaly_cascade": 0.75,
        },
        "episodes": 50,
        "episode_len": EPISODE_LEN,
        "desc": "Scenario emphasizing cascading failure patterns.",
    },
    {
        "name": "MIXED_STRESS",
        "weights": {
            "normal": 0.10,
            "anomaly_main": 0.25,
            "anomaly_edge": 0.30,
            "anomaly_cascade": 0.35,
        },
        "episodes": 50,
        "episode_len": EPISODE_LEN,
        "desc": "Mixed workload scenario for assessing generalization capability.",
    },
]


# ============================================================
# SELECT PPO MODEL FOR EVALUATION
# ============================================================

# Nếu bạn đã có biến selected/best model thì ưu tiên dùng nó.
# Nếu chưa, fallback sang ppo_model.
if "selected_model" in globals() and selected_model is not None:
    eval_ppo_model = selected_model
elif "selected_ppo_model" in globals() and selected_ppo_model is not None:
    eval_ppo_model = selected_ppo_model
elif "best_ppo_model" in globals() and best_ppo_model is not None:
    eval_ppo_model = best_ppo_model
else:
    eval_ppo_model = ppo_model

# Pure PPO baseline được train riêng với obs N*6, không có GNN forecast.
eval_pure_ppo_model = resolve_pure_ppo_model()
eval_pure_ppo_policy = PurePPOPolicyAdapter(eval_pure_ppo_model)


# ============================================================
# RUN EVALUATION
# ============================================================

print("\n" + "=" * 100)
print(" EVALUATION: PPO vs REFERENCE POLICIES ")
print("=" * 100)
print("Important setting:")
print("  Main comparison: PPO and all reference policies use the SAME simulator environment.")
print("  Simulator uses physics-informed hybrid transition: queueing CPU/LAT + learned CPU correction + overload-risk ERR.")
print("  Reference policies do NOT access the capacity model when choosing actions.")
print("  Pure PPO is trained separately and sees only current metrics: no GNN forecast columns.")
print("  HandcraftedCapacityEvalEnv is used only in the capacity-model ablation section.")
print("=" * 100)
print("Scenarios:")
for sc in EVALUATION_SCENARIOS:
    print(f"  - {sc['name']}: {sc['desc']}")
print("=" * 100)

policies = [
    ("PPO + GNN", eval_ppo_model),
    ("Pure PPO", eval_pure_ppo_policy),
    ("DL-Assisted Rule-Based", DLAssistedRuleBasedPolicy()),
    ("Reactive Rule-Based", ReactiveRuleBasedPolicy()),
    ("HPA-like", HPAPolicy(target_cpu=0.70, down_threshold=0.35)),
]

all_episode_rows = []
summary_rows = []

for scenario in EVALUATION_SCENARIOS:
    print("\n" + "-" * 100)
    print(f"Scenario: {scenario['name']}")
    print(f"Desc    : {scenario['desc']}")
    print(f"Weights : {scenario['weights']}")
    print(f"Episodes: {scenario['episodes']}")
    print("-" * 100)

    for policy_name, policy in policies:
        # Main evaluation phải dùng CÙNG một environment cho mọi policy.
        # Capacity model là simulator/world model, không phải thông tin policy được phép đọc.
        # HPA/Rule/NoScaling vẫn chỉ ra quyết định từ obs; chúng không gọi capacity_model.predict().
        use_learned_capacity = True

        env_eval = make_weighted_eval_env(
            trace_weights=scenario["weights"],
            episode_len=scenario["episode_len"],
            use_learned_capacity=use_learned_capacity,
        )

        df_ep = evaluate_policy(
            policy,
            env_eval,
            n_episodes=scenario["episodes"],
            base_seed=SEED,
        )

        df_ep["Scenario"] = scenario["name"]
        df_ep["Policy"] = policy_name
        df_ep["Uses_Learned_Capacity"] = use_learned_capacity

        all_episode_rows.append(df_ep)

        metrics = df_ep[
            [
                "return",
                "mean_reward",
                "mean_sla",
                "p95_sla",
                "mean_cost",
                "mean_cost_quad",
                "mean_churn",
                "mean_replicas",
                "mean_lat_violation",
                "p95_lat_violation",
                "mean_lat_critical_violation",
                "mean_err_violation",
                "mean_cpu_violation",
                "mean_cpu_pressure",
                "mean_cpu_gain",
                "mean_lat_gain",
                "mean_err_gain",
                "capacity_source_id",
            ]
        ].mean()

        score = evaluation_score(metrics)

        summary_rows.append({
            "Scenario": scenario["name"],
            "Policy": policy_name,
            "Uses_Learned_Capacity": bool(use_learned_capacity),
            "Evaluation_Score": float(score),
            "Return": float(metrics["return"]),
            "Mean_Reward": float(metrics["mean_reward"]),
            "SLA_Violation": float(metrics["mean_sla"]),
            "P95_SLA": float(metrics["p95_sla"]),
            "Cost": float(metrics["mean_cost"]),
            "Cost_Quad": float(metrics["mean_cost_quad"]),
            "Churn": float(metrics["mean_churn"]),
            "Replicas": float(metrics["mean_replicas"]),
            "Latency_Violation": float(metrics["mean_lat_violation"]),
            "P95_Latency_Violation": float(metrics["p95_lat_violation"]),
            "Latency_Critical_Violation": float(metrics["mean_lat_critical_violation"]),
            "Error_Violation": float(metrics["mean_err_violation"]),
            "CPU_Violation": float(metrics["mean_cpu_violation"]),
            "CPU_Pressure": float(metrics["mean_cpu_pressure"]),
            "CPU_Gain": float(metrics["mean_cpu_gain"]),
            "LAT_Gain": float(metrics["mean_lat_gain"]),
            "ERR_Gain": float(metrics["mean_err_gain"]),
            "Capacity_Source_ID": float(metrics["capacity_source_id"]),
        })

        cap_label = "learned" if use_learned_capacity else "handcrafted"
        print(
            f"[{policy_name:<23}] "
            f"Cap={cap_label:<11} | "
            f"Score={score:>8.4f} | "
            f"Reward={metrics['mean_reward']:>8.4f} | "
            f"Return={metrics['return']:>8.3f} | "
            f"SLA={metrics['mean_sla']:.4f} | "
            f"P95_SLA={metrics['p95_sla']:.4f} | "
            f"Cost={metrics['mean_cost']:.4f} | "
            f"Reps={metrics['mean_replicas']:.2f} | "
            f"Churn={metrics['mean_churn']:.4f}"
        )


df_eval_episodes = pd.concat(all_episode_rows, ignore_index=True)
df_eval_summary = pd.DataFrame(summary_rows)

eval_episode_path = OUTPUT_DIR / "evaluation_episode_details.csv"
eval_summary_path = OUTPUT_DIR / "evaluation_summary.csv"

df_eval_episodes.to_csv(eval_episode_path, index=False)
df_eval_summary.to_csv(eval_summary_path, index=False)


# ============================================================
# LOG FULL SUMMARY
# ============================================================

pd.set_option("display.max_columns", None)
pd.set_option("display.width", 260)

print("\n" + "=" * 100)
print(" EVALUATION SUMMARY ")
print("=" * 100)
print(df_eval_summary.round(4).to_string(index=False))

print("\nSaved episode details:", eval_episode_path)
print("Saved summary        :", eval_summary_path)


# ============================================================
# RANKING
# ============================================================

rank_df = df_eval_summary.copy()
rank_df["Rank"] = rank_df.groupby("Scenario")["Evaluation_Score"].rank(
    ascending=False,
    method="min",
)

print("\n" + "=" * 100)
print(" RANKING PER SCENARIO ")
print("=" * 100)

for scenario in df_eval_summary["Scenario"].unique():
    sub = (
        rank_df[rank_df["Scenario"] == scenario]
        .sort_values("Rank")
        .reset_index(drop=True)
    )

    print(f"\nScenario: {scenario}")

    for _, row in sub.iterrows():
        print(
            f"{int(row['Rank'])}. "
            f"{row['Policy']:<25} "
            f"Score={row['Evaluation_Score']:.4f} | "
            f"Reward={row['Mean_Reward']:.4f} | "
            f"SLA={row['SLA_Violation']:.4f} | "
            f"P95_SLA={row['P95_SLA']:.4f} | "
            f"Cost={row['Cost']:.4f} | "
            f"Replicas={row['Replicas']:.2f} | "
            f"Churn={row['Churn']:.4f} | "
            f"Cap={'learned' if row['Uses_Learned_Capacity'] else 'handcrafted'}"
        )


# ============================================================
# PPO + GNN COMPARISON AGAINST BEST REFERENCE POLICY
# ============================================================

print("\n" + "=" * 100)
print(" PPO + GNN COMPARISON AGAINST BEST REFERENCE POLICY ")
print("=" * 100)

advantage_rows = []

for scenario in df_eval_summary["Scenario"].unique():
    sub = df_eval_summary[df_eval_summary["Scenario"] == scenario]

    ppo_row = sub[sub["Policy"] == "PPO + GNN"].iloc[0]

    reference_row = (
        sub[sub["Policy"] != "PPO + GNN"]
        .sort_values("Evaluation_Score", ascending=False)
        .iloc[0]
    )

    diff = ppo_row["Evaluation_Score"] - reference_row["Evaluation_Score"]
    reward_diff = ppo_row["Mean_Reward"] - reference_row["Mean_Reward"]

    denom = abs(float(reference_row["Evaluation_Score"])) + 1e-8
    relative_improvement = diff / denom * 100.0

    advantage_rows.append({
        "Scenario": scenario,
        "PPO_Evaluation_Score": float(ppo_row["Evaluation_Score"]),
        "Best_Reference_Policy": reference_row["Policy"],
        "Best_Reference_Evaluation_Score": float(reference_row["Evaluation_Score"]),
        "Delta": float(diff),
        "Relative_Improvement_Percent": float(relative_improvement),
        "PPO_Mean_Reward": float(ppo_row["Mean_Reward"]),
        "Best_Reference_Mean_Reward": float(reference_row["Mean_Reward"]),
        "Reward_Delta": float(reward_diff),
        "PPO_Wins": bool(diff > 0),
    })

    print(
        f"{scenario:<20} | "
        f"PPO={ppo_row['Evaluation_Score']:.4f} | "
        f"BestReference={reference_row['Policy']} ({reference_row['Evaluation_Score']:.4f}) | "
        f"Delta={diff:+.4f} | "
        f"RelImprove={relative_improvement:+.2f}% | "
        f"RewardDelta={reward_diff:+.4f}"
    )

df_advantage = pd.DataFrame(advantage_rows)
advantage_path = OUTPUT_DIR / "evaluation_ppo_advantage.csv"
df_advantage.to_csv(advantage_path, index=False)

print("\nSaved PPO advantage:", advantage_path)


# ============================================================
# CAPACITY MODEL COMPARISON: PPO learned capacity vs PPO handcrafted capacity
# ============================================================

print("\n" + "=" * 100)
print(" CAPACITY MODEL COMPARISON: PPO learned capacity vs PPO handcrafted capacity ")
print("=" * 100)

capacity_effect_rows = []

for scenario in EVALUATION_SCENARIOS:
    for cap_name, use_learned_capacity in [
        ("PPO + Physics-Informed Hybrid Capacity", True),
        ("PPO + Handcrafted Capacity", False),
    ]:
        env_eval = make_weighted_eval_env(
            trace_weights=scenario["weights"],
            episode_len=scenario["episode_len"],
            use_learned_capacity=use_learned_capacity,
        )

        df_ep = evaluate_policy(
            eval_ppo_model,
            env_eval,
            n_episodes=scenario["episodes"],
            base_seed=SEED + 1000,
        )

        metrics = df_ep[
            [
                "return",
                "mean_reward",
                "mean_sla",
                "p95_sla",
                "mean_cost",
                "mean_churn",
                "mean_replicas",
                "mean_lat_violation",
                "mean_cpu_violation",
                "mean_cpu_gain",
                "mean_lat_gain",
            ]
        ].mean()

        score = evaluation_score(metrics)

        capacity_effect_rows.append({
            "Scenario": scenario["name"],
            "Variant": cap_name,
            "Uses_Learned_Capacity": bool(use_learned_capacity),
            "Evaluation_Score": float(score),
            "Return": float(metrics["return"]),
            "Mean_Reward": float(metrics["mean_reward"]),
            "SLA_Violation": float(metrics["mean_sla"]),
            "P95_SLA": float(metrics["p95_sla"]),
            "Cost": float(metrics["mean_cost"]),
            "Churn": float(metrics["mean_churn"]),
            "Replicas": float(metrics["mean_replicas"]),
            "Latency_Violation": float(metrics["mean_lat_violation"]),
            "CPU_Violation": float(metrics["mean_cpu_violation"]),
            "CPU_Gain": float(metrics["mean_cpu_gain"]),
            "LAT_Gain": float(metrics["mean_lat_gain"]),
        })

capacity_effect_df = pd.DataFrame(capacity_effect_rows)
capacity_effect_path = OUTPUT_DIR / "capacity_model_effect_summary.csv"
capacity_effect_df.to_csv(capacity_effect_path, index=False)

print(capacity_effect_df.round(4).to_string(index=False))
print("\nSaved capacity model comparison:", capacity_effect_path)


# ============================================================
# FINAL RESULT
# ============================================================

ppo_wins = int(df_advantage["PPO_Wins"].sum())
total_scenarios = int(df_advantage["Scenario"].nunique())

print("\n" + "=" * 100)
print(" FINAL EVALUATION RESULT ")
print("=" * 100)
print(f"PPO + GNN ranks first in {ppo_wins}/{total_scenarios} evaluation scenarios")

if ppo_wins == total_scenarios:
    print("✓ PPO + GNN ranks first in all evaluation scenarios.")
elif ppo_wins > 0:
    print("✓ PPO + GNN ranks first in some evaluation scenarios. Inspect scenario-level trade-offs.")
else:
    print("⚠ PPO does not rank first under the current evaluation score. Check reward/curriculum/capacity model.")


# ============================================================
# PLOTS
# ============================================================

def plot_grouped_bar(df, metric, title, ylabel):
    if df.empty:
        return

    order = [
        "PPO + GNN",
        "Pure PPO",
        "DL-Assisted Rule-Based",
        "Reactive Rule-Based",
        "HPA-like",
    ]

    pivot_df = df.pivot(index="Scenario", columns="Policy", values=metric)
    pivot_df = pivot_df[[p for p in order if p in pivot_df.columns]]

    x = np.arange(len(pivot_df))
    width = 0.16

    fig, ax = plt.subplots(figsize=(16, 7))

    offsets = np.linspace(
        -width * (len(pivot_df.columns) - 1) / 2,
        width * (len(pivot_df.columns) - 1) / 2,
        len(pivot_df.columns),
    )

    for offset, policy in zip(offsets, pivot_df.columns):
        rects = ax.bar(
            x + offset,
            pivot_df[policy],
            width,
            label=policy,
        )

        for rect in rects:
            height = rect.get_height()
            ax.annotate(
                f"{height:.3f}",
                xy=(rect.get_x() + rect.get_width() / 2, height),
                xytext=(0, 3 if height >= 0 else -15),
                textcoords="offset points",
                ha="center",
                va="bottom" if height >= 0 else "top",
                fontsize=8,
            )

    ax.set_ylabel(ylabel, fontsize=12, fontweight="bold")
    ax.set_title(title, fontsize=14, fontweight="bold")
    ax.set_xticks(x)
    ax.set_xticklabels(
        [name.replace("_", "\n") for name in pivot_df.index],
        fontsize=11,
    )
    ax.legend(fontsize=10)
    ax.grid(axis="y", linestyle="--", alpha=0.7)

    fig.tight_layout()

    save_name = f"evaluation_{metric}.png"
    save_path = OUTPUT_DIR / save_name
    plt.savefig(save_path, dpi=150)
    plt.show()

    print(f"Saved plot: {save_path}")


def plot_advantage_bar(df):
    if df.empty:
        return

    fig, ax = plt.subplots(figsize=(12, 5))
    x = np.arange(len(df))

    bars = ax.bar(x, df["Delta"])

    for rect, val in zip(bars, df["Delta"]):
        ax.annotate(
            f"{val:+.4f}",
            xy=(rect.get_x() + rect.get_width() / 2, val),
            xytext=(0, 3 if val >= 0 else -15),
            textcoords="offset points",
            ha="center",
            va="bottom" if val >= 0 else "top",
            fontsize=9,
        )

    ax.axhline(0, linewidth=1)
    ax.set_title("PPO + GNN Comparison Against Best Reference Policy", fontsize=14, fontweight="bold")
    ax.set_ylabel("Evaluation Score Delta", fontsize=12, fontweight="bold")
    ax.set_xticks(x)
    ax.set_xticklabels([s.replace("_", "\n") for s in df["Scenario"]], fontsize=11)
    ax.grid(axis="y", linestyle="--", alpha=0.7)

    fig.tight_layout()

    save_path = OUTPUT_DIR / "evaluation_ppo_advantage.png"
    plt.savefig(save_path, dpi=150)
    plt.show()

    print(f"Saved plot: {save_path}")


def plot_relative_improvement(df):
    if df.empty:
        return

    fig, ax = plt.subplots(figsize=(12, 5))
    x = np.arange(len(df))

    bars = ax.bar(x, df["Relative_Improvement_Percent"])

    for rect, val in zip(bars, df["Relative_Improvement_Percent"]):
        ax.annotate(
            f"{val:+.1f}%",
            xy=(rect.get_x() + rect.get_width() / 2, val),
            xytext=(0, 3 if val >= 0 else -15),
            textcoords="offset points",
            ha="center",
            va="bottom" if val >= 0 else "top",
            fontsize=9,
        )

    ax.axhline(0, linewidth=1)
    ax.set_title("PPO + GNN Relative Improvement Over Best Reference Policy", fontsize=14, fontweight="bold")
    ax.set_ylabel("Relative Improvement (%)", fontsize=12, fontweight="bold")
    ax.set_xticks(x)
    ax.set_xticklabels([s.replace("_", "\n") for s in df["Scenario"]], fontsize=11)
    ax.grid(axis="y", linestyle="--", alpha=0.7)

    fig.tight_layout()

    save_path = OUTPUT_DIR / "evaluation_relative_improvement.png"
    plt.savefig(save_path, dpi=150)
    plt.show()

    print(f"Saved plot: {save_path}")


def plot_capacity_effect(df, metric, title, ylabel):
    if df.empty:
        return

    pivot_df = df.pivot(index="Scenario", columns="Variant", values=metric)
    order = ["PPO + Learned Capacity", "PPO + Handcrafted Capacity"]
    pivot_df = pivot_df[[c for c in order if c in pivot_df.columns]]

    x = np.arange(len(pivot_df))
    width = 0.28

    fig, ax = plt.subplots(figsize=(14, 6))
    offsets = np.linspace(
        -width * (len(pivot_df.columns) - 1) / 2,
        width * (len(pivot_df.columns) - 1) / 2,
        len(pivot_df.columns),
    )

    for offset, variant in zip(offsets, pivot_df.columns):
        rects = ax.bar(x + offset, pivot_df[variant], width, label=variant)

        for rect in rects:
            height = rect.get_height()
            ax.annotate(
                f"{height:.3f}",
                xy=(rect.get_x() + rect.get_width() / 2, height),
                xytext=(0, 3 if height >= 0 else -15),
                textcoords="offset points",
                ha="center",
                va="bottom" if height >= 0 else "top",
                fontsize=8,
            )

    ax.set_ylabel(ylabel, fontsize=12, fontweight="bold")
    ax.set_title(title, fontsize=14, fontweight="bold")
    ax.set_xticks(x)
    ax.set_xticklabels([s.replace("_", "\n") for s in pivot_df.index], fontsize=11)
    ax.legend(fontsize=10)
    ax.grid(axis="y", linestyle="--", alpha=0.7)

    fig.tight_layout()

    save_path = OUTPUT_DIR / f"capacity_model_effect_{metric}.png"
    plt.savefig(save_path, dpi=150)
    plt.show()

    print(f"Saved plot: {save_path}")


# Main comparison plots: PPO vs baselines in each scenario
plot_grouped_bar(df_eval_summary, "Evaluation_Score", "Evaluation Score Across Scaling Policies", "Evaluation Score")
plot_grouped_bar(df_eval_summary, "Mean_Reward", "Mean Reward Across Scaling Policies", "Mean Reward")
plot_grouped_bar(df_eval_summary, "Return", "Episode Return Across Scaling Policies", "Episode Return")
plot_grouped_bar(df_eval_summary, "SLA_Violation", "Mean SLA Violation Under Evaluation", "Mean SLA Violation")
plot_grouped_bar(df_eval_summary, "P95_SLA", "P95 SLA Violation Under Evaluation", "P95 SLA Violation")
plot_grouped_bar(df_eval_summary, "Cost", "Resource Cost Under Evaluation", "Resource Cost")
plot_grouped_bar(df_eval_summary, "Replicas", "Average Replicas Under Evaluation", "Average Replicas")
plot_grouped_bar(df_eval_summary, "Churn", "Scaling Churn Under Evaluation", "Churn")
plot_grouped_bar(df_eval_summary, "Latency_Violation", "Latency Violation Under Evaluation", "Latency Violation")
plot_grouped_bar(df_eval_summary, "P95_Latency_Violation", "P95 Latency Violation Under Evaluation", "P95 Latency Violation")
plot_grouped_bar(df_eval_summary, "CPU_Violation", "CPU Violation Under Evaluation", "CPU Violation")
plot_grouped_bar(df_eval_summary, "CPU_Gain", "Effective CPU Capacity Gain", "CPU Gain")
plot_grouped_bar(df_eval_summary, "LAT_Gain", "Effective Latency Capacity Gain", "Latency Gain")

# Advantage plots
plot_advantage_bar(df_advantage)
plot_relative_improvement(df_advantage)

# Capacity model effect plots
plot_capacity_effect(capacity_effect_df, "Evaluation_Score", "Capacity Model Comparison on PPO Evaluation Score", "Evaluation Score")
plot_capacity_effect(capacity_effect_df, "Mean_Reward", "Capacity Model Comparison on PPO Mean Reward", "Mean Reward")
plot_capacity_effect(capacity_effect_df, "SLA_Violation", "Capacity Model Comparison on PPO SLA Violation", "SLA Violation")
plot_capacity_effect(capacity_effect_df, "Cost", "Capacity Model Comparison on PPO Cost", "Cost")



# ============================================================
# REPORTING AND ANALYSIS VISUALIZATIONS
# ============================================================
# These plots are designed for thesis/reporting outputs:
#   1. Cost-SLA Pareto tradeoff
#   2. Return / Reward distribution boxplots
#   3. Stability summary across episodes
#   4. Per-scenario policy table exports
#   5. PPO vs best reference policy detailed delta
#
# They reuse df_eval_summary, df_eval_episodes, df_advantage,
# and capacity_effect_df computed above.
# ============================================================

print("\n" + "=" * 100)
print(" REPORTING AND ANALYSIS OUTPUTS ")
print("=" * 100)


def _safe_filename(name: str) -> str:
    return (
        str(name)
        .replace(" ", "_")
        .replace("/", "_")
        .replace("+", "plus")
        .replace("-", "_")
        .replace("__", "_")
    )


# ------------------------------------------------------------
# 1) Cost vs SLA Pareto / tradeoff plot per scenario
# ------------------------------------------------------------

def plot_cost_sla_tradeoff(df):
    """
    Each scenario gets one scatter plot.
    X = Cost
    Y = SLA_Violation
    Better policies are closer to bottom-left.
    """
    if df.empty:
        print("[SKIP] plot_cost_sla_tradeoff: df is empty")
        return

    for scenario in df["Scenario"].unique():
        sub = df[df["Scenario"] == scenario].copy()
        if sub.empty:
            continue

        fig, ax = plt.subplots(figsize=(9, 6))

        for _, row in sub.iterrows():
            ax.scatter(row["Cost"], row["SLA_Violation"], s=110)
            ax.annotate(
                row["Policy"],
                (row["Cost"], row["SLA_Violation"]),
                textcoords="offset points",
                xytext=(7, 5),
                fontsize=9,
            )

        ax.set_title(
            f"Cost-SLA Tradeoff — {scenario}",
            fontsize=14,
            fontweight="bold",
        )
        ax.set_xlabel("Resource Cost", fontsize=12, fontweight="bold")
        ax.set_ylabel("SLA Violation", fontsize=12, fontweight="bold")
        ax.grid(True, linestyle="--", alpha=0.6)

        # Mark desirable direction.
        ax.text(
            0.02,
            0.02,
            "Better ↓ left",
            transform=ax.transAxes,
            fontsize=10,
            bbox=dict(boxstyle="round", alpha=0.15),
        )

        fig.tight_layout()

        save_path = OUTPUT_DIR / f"report_tradeoff_cost_sla_{_safe_filename(scenario)}.png"
        plt.savefig(save_path, dpi=150)
        plt.show()
        print(f"Saved plot: {save_path}")


# ------------------------------------------------------------
# 2) Global Pareto plot over all scenarios
# ------------------------------------------------------------

def plot_global_cost_sla_tradeoff(df):
    """
    One global plot using scenario-averaged policy metrics.
    Useful for report summary.
    """
    if df.empty:
        print("[SKIP] plot_global_cost_sla_tradeoff: df is empty")
        return

    global_df = (
        df.groupby("Policy", as_index=False)
        .agg({
            "Cost": "mean",
            "SLA_Violation": "mean",
            "P95_SLA": "mean",
            "Mean_Reward": "mean",
            "Evaluation_Score": "mean",
            "Replicas": "mean",
            "Churn": "mean",
        })
    )

    global_path = OUTPUT_DIR / "report_global_policy_summary.csv"
    global_df.to_csv(global_path, index=False)
    print(f"Saved global policy summary: {global_path}")

    fig, ax = plt.subplots(figsize=(9, 6))

    for _, row in global_df.iterrows():
        ax.scatter(row["Cost"], row["SLA_Violation"], s=130)
        ax.annotate(
            row["Policy"],
            (row["Cost"], row["SLA_Violation"]),
            textcoords="offset points",
            xytext=(8, 5),
            fontsize=9,
        )

    ax.set_title(
        "Global Cost-SLA Pareto Tradeoff",
        fontsize=14,
        fontweight="bold",
    )
    ax.set_xlabel("Mean Resource Cost", fontsize=12, fontweight="bold")
    ax.set_ylabel("Mean SLA Violation", fontsize=12, fontweight="bold")
    ax.grid(True, linestyle="--", alpha=0.6)
    ax.text(
        0.02,
        0.02,
        "Better ↓ left",
        transform=ax.transAxes,
        fontsize=10,
        bbox=dict(boxstyle="round", alpha=0.15),
    )

    fig.tight_layout()

    save_path = OUTPUT_DIR / "report_global_cost_sla_pareto.png"
    plt.savefig(save_path, dpi=150)
    plt.show()
    print(f"Saved plot: {save_path}")


# ------------------------------------------------------------
# 3) Return / reward distribution boxplots
# ------------------------------------------------------------

def plot_episode_distribution_boxplot(df_ep, metric, title, ylabel):
    """
    Boxplot across all episodes, grouped by policy.
    This shows stability, not only mean performance.
    """
    if df_ep.empty or metric not in df_ep.columns:
        print(f"[SKIP] plot_episode_distribution_boxplot: missing {metric}")
        return

    order = [
        "PPO + GNN",
        "Pure PPO",
        "DL-Assisted Rule-Based",
        "Reactive Rule-Based",
        "HPA-like",
    ]

    policies_present = [p for p in order if p in set(df_ep["Policy"])]
    data = [
        df_ep[df_ep["Policy"] == p][metric].dropna().values
        for p in policies_present
    ]

    fig, ax = plt.subplots(figsize=(13, 6))
    ax.boxplot(data, labels=policies_present, showmeans=True)

    ax.set_title(title, fontsize=14, fontweight="bold")
    ax.set_ylabel(ylabel, fontsize=12, fontweight="bold")
    ax.set_xticklabels(policies_present, rotation=20, ha="right")
    ax.grid(axis="y", linestyle="--", alpha=0.6)

    fig.tight_layout()

    save_path = OUTPUT_DIR / f"report_boxplot_{metric}.png"
    plt.savefig(save_path, dpi=150)
    plt.show()
    print(f"Saved plot: {save_path}")


def plot_episode_distribution_by_scenario(df_ep, metric, title_prefix, ylabel):
    """
    One boxplot per scenario.
    Useful when the report needs to show variance in challenging cases.
    """
    if df_ep.empty or metric not in df_ep.columns:
        print(f"[SKIP] plot_episode_distribution_by_scenario: missing {metric}")
        return

    order = [
        "PPO + GNN",
        "Pure PPO",
        "DL-Assisted Rule-Based",
        "Reactive Rule-Based",
        "HPA-like",
    ]

    for scenario in df_ep["Scenario"].unique():
        sub = df_ep[df_ep["Scenario"] == scenario]
        policies_present = [p for p in order if p in set(sub["Policy"])]
        data = [
            sub[sub["Policy"] == p][metric].dropna().values
            for p in policies_present
        ]

        fig, ax = plt.subplots(figsize=(13, 6))
        ax.boxplot(data, labels=policies_present, showmeans=True)

        ax.set_title(
            f"{title_prefix} — {scenario}",
            fontsize=14,
            fontweight="bold",
        )
        ax.set_ylabel(ylabel, fontsize=12, fontweight="bold")
        ax.set_xticklabels(policies_present, rotation=20, ha="right")
        ax.grid(axis="y", linestyle="--", alpha=0.6)

        fig.tight_layout()

        save_path = OUTPUT_DIR / f"report_boxplot_{metric}_{_safe_filename(scenario)}.png"
        plt.savefig(save_path, dpi=150)
        plt.show()
        print(f"Saved plot: {save_path}")


# ------------------------------------------------------------
# 4) Stability table: mean/std across episodes
# ------------------------------------------------------------

def export_policy_stability_tables(df_ep):
    """
    Export mean/std tables for reporting.
    """
    if df_ep.empty:
        print("[SKIP] export_policy_stability_tables: df_ep is empty")
        return

    metric_cols = [
        "return",
        "mean_reward",
        "mean_sla",
        "p95_sla",
        "mean_cost",
        "mean_churn",
        "mean_replicas",
        "mean_lat_violation",
        "mean_err_violation",
        "mean_cpu_violation",
    ]

    available_cols = [c for c in metric_cols if c in df_ep.columns]

    stability_df = (
        df_ep.groupby(["Scenario", "Policy"])[available_cols]
        .agg(["mean", "std", "min", "max"])
        .reset_index()
    )

    # Flatten multi-index columns.
    stability_df.columns = [
        "_".join([str(x) for x in col if str(x) != ""])
        for col in stability_df.columns.values
    ]

    path = OUTPUT_DIR / "report_policy_stability_summary.csv"
    stability_df.to_csv(path, index=False)
    print(f"Saved stability summary: {path}")

    # Also export compact table with mean±std string.
    compact_rows = []

    for (scenario, policy), g in df_ep.groupby(["Scenario", "Policy"]):
        row = {
            "Scenario": scenario,
            "Policy": policy,
        }
        for metric in available_cols:
            vals = g[metric].dropna()
            row[f"{metric}_mean"] = float(vals.mean())
            row[f"{metric}_std"] = float(vals.std())
            row[f"{metric}_mean_std"] = f"{vals.mean():.4f} ± {vals.std():.4f}"
        compact_rows.append(row)

    compact_df = pd.DataFrame(compact_rows)
    compact_path = OUTPUT_DIR / "report_policy_stability_compact.csv"
    compact_df.to_csv(compact_path, index=False)
    print(f"Saved compact stability summary: {compact_path}")

    print("\nREPORT STABILITY COMPACT SUMMARY")
    print(compact_df.round(4).to_string(index=False))


# ------------------------------------------------------------
# 5) PPO vs best reference policy detailed deltas
# ------------------------------------------------------------

def export_ppo_delta_table(df_summary):
    """
    Export detailed PPO improvement over the best reference policy per scenario.
    """
    if df_summary.empty:
        print("[SKIP] export_ppo_delta_table: df_summary is empty")
        return

    rows = []

    for scenario in df_summary["Scenario"].unique():
        sub = df_summary[df_summary["Scenario"] == scenario]

        if "PPO" not in set(sub["Policy"]):
            continue

        ppo = sub[sub["Policy"] == "PPO + GNN"].iloc[0]
        best_base = (
            sub[sub["Policy"] != "PPO + GNN"]
            .sort_values("Evaluation_Score", ascending=False)
            .iloc[0]
        )

        rows.append({
            "Scenario": scenario,
            "Best_Reference_Policy": best_base["Policy"],

            "PPO_Evaluation_Score": float(ppo["Evaluation_Score"]),
            "Best_Reference_Evaluation_Score": float(best_base["Evaluation_Score"]),
            "Evaluation_Score_Delta": float(ppo["Evaluation_Score"] - best_base["Evaluation_Score"]),

            "PPO_Mean_Reward": float(ppo["Mean_Reward"]),
            "Best_Reference_Mean_Reward": float(best_base["Mean_Reward"]),
            "Mean_Reward_Delta": float(ppo["Mean_Reward"] - best_base["Mean_Reward"]),

            "PPO_SLA": float(ppo["SLA_Violation"]),
            "Best_Reference_SLA": float(best_base["SLA_Violation"]),
            "SLA_Reduction": float(best_base["SLA_Violation"] - ppo["SLA_Violation"]),

            "PPO_P95_SLA": float(ppo["P95_SLA"]),
            "Best_Reference_P95_SLA": float(best_base["P95_SLA"]),
            "P95_SLA_Reduction": float(best_base["P95_SLA"] - ppo["P95_SLA"]),

            "PPO_Cost": float(ppo["Cost"]),
            "Best_Reference_Cost": float(best_base["Cost"]),
            "Cost_Reduction": float(best_base["Cost"] - ppo["Cost"]),

            "PPO_Churn": float(ppo["Churn"]),
            "Best_Reference_Churn": float(best_base["Churn"]),
            "Churn_Reduction": float(best_base["Churn"] - ppo["Churn"]),
        })

    delta_df = pd.DataFrame(rows)
    path = OUTPUT_DIR / "report_ppo_vs_best_baseline_deltas.csv"
    delta_df.to_csv(path, index=False)

    print(f"\nSaved PPO detailed delta table: {path}")
    print(delta_df.round(4).to_string(index=False))

    return delta_df


def plot_ppo_delta_bar(delta_df, metric, title, ylabel):
    if delta_df is None or delta_df.empty or metric not in delta_df.columns:
        print(f"[SKIP] plot_ppo_delta_bar: missing {metric}")
        return

    fig, ax = plt.subplots(figsize=(12, 5))
    x = np.arange(len(delta_df))

    bars = ax.bar(x, delta_df[metric])

    for rect, val in zip(bars, delta_df[metric]):
        ax.annotate(
            f"{val:+.4f}",
            xy=(rect.get_x() + rect.get_width() / 2, val),
            xytext=(0, 3 if val >= 0 else -15),
            textcoords="offset points",
            ha="center",
            va="bottom" if val >= 0 else "top",
            fontsize=9,
        )

    ax.axhline(0, linewidth=1)
    ax.set_title(title, fontsize=14, fontweight="bold")
    ax.set_ylabel(ylabel, fontsize=12, fontweight="bold")
    ax.set_xticks(x)
    ax.set_xticklabels([s.replace("_", "\n") for s in delta_df["Scenario"]], fontsize=11)
    ax.grid(axis="y", linestyle="--", alpha=0.6)

    fig.tight_layout()

    save_path = OUTPUT_DIR / f"report_ppo_delta_{metric}.png"
    plt.savefig(save_path, dpi=150)
    plt.show()
    print(f"Saved plot: {save_path}")


# ------------------------------------------------------------
# 6) Policy win count across scenarios
# ------------------------------------------------------------

def export_policy_win_count(df_summary):
    if df_summary.empty:
        print("[SKIP] export_policy_win_count: df_summary is empty")
        return

    winners = (
        df_summary.sort_values("Evaluation_Score", ascending=False)
        .groupby("Scenario")
        .head(1)
        [["Scenario", "Policy", "Evaluation_Score"]]
        .reset_index(drop=True)
    )

    win_count = winners["Policy"].value_counts().reset_index()
    win_count.columns = ["Policy", "Win_Count"]

    winners_path = OUTPUT_DIR / "report_scenario_winners.csv"
    win_count_path = OUTPUT_DIR / "report_policy_win_count.csv"

    winners.to_csv(winners_path, index=False)
    win_count.to_csv(win_count_path, index=False)

    print(f"Saved scenario winners: {winners_path}")
    print(f"Saved policy win count: {win_count_path}")

    print("\nSCENARIO WINNERS")
    print(winners.round(4).to_string(index=False))

    print("\nPOLICY WIN COUNT")
    print(win_count.to_string(index=False))

    fig, ax = plt.subplots(figsize=(9, 5))
    bars = ax.bar(win_count["Policy"], win_count["Win_Count"])

    for rect, val in zip(bars, win_count["Win_Count"]):
        ax.annotate(
            f"{int(val)}",
            xy=(rect.get_x() + rect.get_width() / 2, val),
            xytext=(0, 3),
            textcoords="offset points",
            ha="center",
            va="bottom",
            fontsize=10,
        )

    ax.set_title("Policy Win Count Across Scenarios", fontsize=14, fontweight="bold")
    ax.set_ylabel("Number of winning scenarios", fontsize=12, fontweight="bold")
    ax.set_xticklabels(win_count["Policy"], rotation=20, ha="right")
    ax.grid(axis="y", linestyle="--", alpha=0.6)

    fig.tight_layout()

    save_path = OUTPUT_DIR / "report_policy_win_count.png"
    plt.savefig(save_path, dpi=150)
    plt.show()
    print(f"Saved plot: {save_path}")


# ------------------------------------------------------------
# 7) Optional training curve from curriculum_training_log.csv
# ------------------------------------------------------------

def plot_training_curve_from_log():
    """
    If curriculum_training_log.csv exists, plot reward/SLA/cost curves.
    This is commonly used in RL experiment reports.
    """
    log_path = OUTPUT_DIR / "curriculum_training_log.csv"

    if not log_path.exists():
        print(f"[SKIP] Training curve: {log_path} not found")
        return

    train_df = pd.read_csv(log_path)

    print(f"Loaded training log: {log_path}")
    print("Training log columns:", list(train_df.columns))

    x_candidates = [
        "global_timesteps",
        "total_timesteps",
        "num_timesteps",
        "timesteps",
        "step",
    ]

    x_col = None
    for c in x_candidates:
        if c in train_df.columns:
            x_col = c
            break

    if x_col is None:
        # fallback: use row index
        train_df = train_df.copy()
        train_df["eval_index"] = np.arange(len(train_df))
        x_col = "eval_index"

    curve_specs = [
        ("mean_reward", "Training Mean Reward", "Mean Reward"),
        ("mean_sla", "Training SLA Violation", "SLA Violation"),
        ("mean_cost", "Training Resource Cost", "Resource Cost"),
        ("mean_churn", "Training Churn", "Churn"),
        ("mean_replicas", "Training Average Replicas", "Average Replicas"),
    ]

    for metric, title, ylabel in curve_specs:
        if metric not in train_df.columns:
            print(f"[SKIP] Training curve missing column: {metric}")
            continue

        fig, ax = plt.subplots(figsize=(12, 5))

        if "stage_name" in train_df.columns:
            for stage_name, sub in train_df.groupby("stage_name"):
                ax.plot(sub[x_col], sub[metric], marker="o", linewidth=1.8, label=stage_name)
            ax.legend(fontsize=9)
        else:
            ax.plot(train_df[x_col], train_df[metric], marker="o", linewidth=1.8)

        ax.set_title(title, fontsize=14, fontweight="bold")
        ax.set_xlabel(x_col, fontsize=12, fontweight="bold")
        ax.set_ylabel(ylabel, fontsize=12, fontweight="bold")
        ax.grid(True, linestyle="--", alpha=0.6)

        fig.tight_layout()

        save_path = OUTPUT_DIR / f"report_training_curve_{metric}.png"
        plt.savefig(save_path, dpi=150)
        plt.show()
        print(f"Saved plot: {save_path}")


# ------------------------------------------------------------
# Execute reporting exports
# ------------------------------------------------------------

plot_cost_sla_tradeoff(df_eval_summary)
plot_global_cost_sla_tradeoff(df_eval_summary)

plot_episode_distribution_boxplot(
    df_eval_episodes,
    "return",
    "Episode Return Distribution Across Policies",
    "Episode Return",
)

plot_episode_distribution_boxplot(
    df_eval_episodes,
    "mean_reward",
    "Mean Reward Distribution Across Policies",
    "Mean Reward",
)

plot_episode_distribution_by_scenario(
    df_eval_episodes,
    "return",
    "Episode Return Distribution",
    "Episode Return",
)

plot_episode_distribution_by_scenario(
    df_eval_episodes,
    "mean_reward",
    "Mean Reward Distribution",
    "Mean Reward",
)

export_policy_stability_tables(df_eval_episodes)

delta_df = export_ppo_delta_table(df_eval_summary)

plot_ppo_delta_bar(
    delta_df,
    "Evaluation_Score_Delta",
    "PPO Evaluation Score Delta Over Best Reference Policy",
    "Evaluation Score Delta",
)

plot_ppo_delta_bar(
    delta_df,
    "Mean_Reward_Delta",
    "PPO Mean Reward Delta Over Best Reference Policy",
    "Mean Reward Delta",
)

plot_ppo_delta_bar(
    delta_df,
    "SLA_Reduction",
    "PPO SLA Reduction Over Best Reference Policy",
    "SLA Reduction",
)

plot_ppo_delta_bar(
    delta_df,
    "Cost_Reduction",
    "PPO Cost Reduction Over Best Reference Policy",
    "Cost Reduction",
)

export_policy_win_count(df_eval_summary)

plot_training_curve_from_log()

print("\n" + "=" * 100)
print(" ALL REPORTING OUTPUTS SAVED ")
print("=" * 100)
print("Extra CSV files:")
for name in [
    "report_global_policy_summary.csv",
    "report_policy_stability_summary.csv",
    "report_policy_stability_compact.csv",
    "report_ppo_vs_best_baseline_deltas.csv",
    "report_scenario_winners.csv",
    "report_policy_win_count.csv",
]:
    print(" -", OUTPUT_DIR / name)

print("\nExtra PNG examples:")
for name in [
    "report_global_cost_sla_pareto.png",
    "report_boxplot_return.png",
    "report_boxplot_mean_reward.png",
    "report_ppo_delta_Evaluation_Score_Delta.png",
    "report_ppo_delta_Mean_Reward_Delta.png",
    "report_ppo_delta_SLA_Reduction.png",
    "report_policy_win_count.png",
]:
    print(" -", OUTPUT_DIR / name)


In [ ]:
# # ============================================================
# # CAPACITY / WORKLOAD PATTERN ANALYSIS FOR ONLINE BOUTIQUE
# # FIXED VERSION: KHÔNG DÙNG pandas .describe()
# # ============================================================

# import os
# import json
# from pathlib import Path

# import numpy as np
# import pandas as pd
# import matplotlib.pyplot as plt
# from sklearn.linear_model import LinearRegression

# # ============================================================
# # CONFIG
# # ============================================================

# DATA_ROOT = Path("/kaggle/input/datasets/kudo123a/dataset-v8")

# OUT_DIR = Path("/kaggle/working/capacity_pattern_analysis")
# OUT_DIR.mkdir(parents=True, exist_ok=True)

# SCENARIOS = {
#     "normal": "data_normal",
#     "anomaly_main": "data_anomaly",
#     "anomaly_edge": "data_deep_anomaly",
#     "anomaly_cascade": "data_final_cascade_anomaly",
# }

# TARGET_SERVICES = sorted([
#     "adservice",
#     "cartservice",
#     "checkoutservice",
#     "currencyservice",
#     "emailservice",
#     "frontend",
#     "paymentservice",
#     "productcatalogservice",
#     "recommendationservice",
#     "shippingservice",
# ])

# R_MAX = 10
# EPS = 1e-8

# NODE_COLS = {
#     "cpu": "cpu_usage_millicores_norm",
#     "memory": "memory_usage_bytes_norm",
#     "replicas": "pod_replicas_count_norm",
#     "alloc_cpu": "allocated_cpu_quota_millicores_norm",
#     "error": "error_rate_ratio_norm",
#     "rps": "request_per_second_norm",
#     "lat_p50": "latency_p50_seconds_norm",
#     "lat_p95": "latency_p95_seconds_norm",
#     "lat_p99": "latency_p99_seconds_norm",
# }

# EDGE_COLS = {
#     "network_latency": "network_latency_seconds_norm",
#     "payload": "payload_size_bytes_norm",
#     "edge_rps": "edge_request_rate_rps_norm",
#     "edge_error": "edge_error_rate_ratio_norm",
# }

# RL_NODE_METRICS = ["cpu", "error", "lat_p99", "rps"]
# RL_EDGE_METRICS = ["network_latency", "edge_rps", "edge_error", "payload"]


# # ============================================================
# # SAFE SUMMARY FUNCTIONS - KHÔNG DÙNG pandas describe()
# # ============================================================

# def safe_numeric_summary(df, cols=None):
#     """
#     Thay thế cho df.describe().
#     Không dùng pandas describe để tránh lỗi Pandas4Warning trên Kaggle.
#     """
#     if df is None or len(df) == 0:
#         return pd.DataFrame()

#     if cols is None:
#         numeric_df = df.select_dtypes(include=[np.number])
#     else:
#         valid_cols = [c for c in cols if c in df.columns]
#         numeric_df = df[valid_cols].select_dtypes(include=[np.number])

#     rows = []

#     for col in numeric_df.columns:
#         vals = pd.to_numeric(numeric_df[col], errors="coerce")
#         vals = vals.replace([np.inf, -np.inf], np.nan).dropna().values

#         if len(vals) == 0:
#             continue

#         rows.append({
#             "column": col,
#             "count": int(len(vals)),
#             "mean": float(np.mean(vals)),
#             "std": float(np.std(vals)),
#             "min": float(np.min(vals)),
#             "p25": float(np.percentile(vals, 25)),
#             "p50": float(np.percentile(vals, 50)),
#             "p75": float(np.percentile(vals, 75)),
#             "p90": float(np.percentile(vals, 90)),
#             "p95": float(np.percentile(vals, 95)),
#             "p99": float(np.percentile(vals, 99)),
#             "max": float(np.max(vals)),
#         })

#     return pd.DataFrame(rows)


# def safe_group_summary(df, group_col, value_cols):
#     """
#     Thay thế cho groupby(...).describe().
#     """
#     if df is None or len(df) == 0:
#         return pd.DataFrame()

#     if group_col not in df.columns:
#         return pd.DataFrame()

#     value_cols = [c for c in value_cols if c in df.columns]
#     if len(value_cols) == 0:
#         return pd.DataFrame()

#     rows = []

#     for group_value, sub in df.groupby(group_col):
#         for col in value_cols:
#             vals = pd.to_numeric(sub[col], errors="coerce")
#             vals = vals.replace([np.inf, -np.inf], np.nan).dropna().values

#             if len(vals) == 0:
#                 continue

#             rows.append({
#                 group_col: group_value,
#                 "column": col,
#                 "count": int(len(vals)),
#                 "mean": float(np.mean(vals)),
#                 "std": float(np.std(vals)),
#                 "min": float(np.min(vals)),
#                 "p25": float(np.percentile(vals, 25)),
#                 "p50": float(np.percentile(vals, 50)),
#                 "p75": float(np.percentile(vals, 75)),
#                 "p90": float(np.percentile(vals, 90)),
#                 "p95": float(np.percentile(vals, 95)),
#                 "p99": float(np.percentile(vals, 99)),
#                 "max": float(np.max(vals)),
#             })

#     return pd.DataFrame(rows)


# # ============================================================
# # LOAD DATA
# # ============================================================

# node_dfs = []
# edge_dfs = []

# for scenario_name, folder in SCENARIOS.items():
#     node_path = DATA_ROOT / folder / "nodes_data.csv"
#     edge_path = DATA_ROOT / folder / "edges_data.csv"

#     if node_path.exists():
#         df = pd.read_csv(node_path)
#         df["scenario"] = scenario_name

#         if "service" in df.columns:
#             df = df[df["service"].isin(TARGET_SERVICES)].copy()

#         node_dfs.append(df)
#         print(f"✓ Loaded node {scenario_name}: {len(df)} rows")
#     else:
#         print(f"⚠ Missing node file: {node_path}")

#     if edge_path.exists():
#         df = pd.read_csv(edge_path)
#         df["scenario"] = scenario_name
#         edge_dfs.append(df)
#         print(f"✓ Loaded edge {scenario_name}: {len(df)} rows")
#     else:
#         print(f"⚠ Missing edge file: {edge_path}")

# if len(node_dfs) == 0:
#     raise RuntimeError("Không tìm thấy nodes_data.csv")

# nodes = pd.concat(node_dfs, ignore_index=True)

# if len(edge_dfs) > 0:
#     edges = pd.concat(edge_dfs, ignore_index=True)
# else:
#     edges = pd.DataFrame()

# for name, col in NODE_COLS.items():
#     if col in nodes.columns:
#         nodes[col] = (
#             pd.to_numeric(nodes[col], errors="coerce")
#             .replace([np.inf, -np.inf], np.nan)
#             .fillna(0.0)
#         )

# if not edges.empty:
#     for name, col in EDGE_COLS.items():
#         if col in edges.columns:
#             edges[col] = (
#                 pd.to_numeric(edges[col], errors="coerce")
#                 .replace([np.inf, -np.inf], np.nan)
#                 .fillna(0.0)
#             )

# nodes["replicas_int"] = np.clip(
#     np.rint(nodes[NODE_COLS["replicas"]] * R_MAX),
#     1,
#     R_MAX,
# ).astype(int)

# print("\nNode rows:", len(nodes))
# print("Edge rows:", len(edges))


# # ============================================================
# # 1. DISTRIBUTION BY SCENARIO
# # ============================================================

# dist_rows = []

# for scenario in sorted(nodes["scenario"].unique()):
#     sub = nodes[nodes["scenario"] == scenario]

#     for metric_name in RL_NODE_METRICS:
#         col = NODE_COLS[metric_name]
#         if col not in sub.columns:
#             continue

#         vals = sub[col].values

#         dist_rows.append({
#             "type": "node",
#             "scenario": scenario,
#             "metric": metric_name,
#             "p50": np.percentile(vals, 50),
#             "p75": np.percentile(vals, 75),
#             "p90": np.percentile(vals, 90),
#             "p95": np.percentile(vals, 95),
#             "p99": np.percentile(vals, 99),
#             "max": np.max(vals),
#             "mean": np.mean(vals),
#             "std": np.std(vals),
#             "nonzero_percent": float((vals > 1e-6).mean() * 100),
#         })

# if not edges.empty:
#     for scenario in sorted(edges["scenario"].unique()):
#         sub = edges[edges["scenario"] == scenario]

#         for metric_name in RL_EDGE_METRICS:
#             col = EDGE_COLS[metric_name]
#             if col not in sub.columns:
#                 continue

#             vals = sub[col].values

#             dist_rows.append({
#                 "type": "edge",
#                 "scenario": scenario,
#                 "metric": metric_name,
#                 "p50": np.percentile(vals, 50),
#                 "p75": np.percentile(vals, 75),
#                 "p90": np.percentile(vals, 90),
#                 "p95": np.percentile(vals, 95),
#                 "p99": np.percentile(vals, 99),
#                 "max": np.max(vals),
#                 "mean": np.mean(vals),
#                 "std": np.std(vals),
#                 "nonzero_percent": float((vals > 1e-6).mean() * 100),
#             })

# dist_df = pd.DataFrame(dist_rows)
# dist_df.to_csv(OUT_DIR / "01_metric_distribution_by_scenario.csv", index=False)

# print("\n=== Metric distribution summary ===")
# print(dist_df.head(20))


# # ============================================================
# # 2. NODE CORRELATION
# # ============================================================

# node_corr_rows = []

# for scenario in sorted(nodes["scenario"].unique()):
#     for svc in TARGET_SERVICES:
#         sub = nodes[
#             (nodes["scenario"] == scenario) &
#             (nodes["service"] == svc)
#         ].copy()

#         if len(sub) < 20:
#             continue

#         row = {
#             "scenario": scenario,
#             "service": svc,
#             "rows": len(sub),
#             "mean_replicas": sub["replicas_int"].mean(),
#             "mean_rps": sub[NODE_COLS["rps"]].mean(),
#         }

#         for metric_name in ["cpu", "memory", "alloc_cpu", "error", "lat_p50", "lat_p95", "lat_p99"]:
#             col = NODE_COLS[metric_name]

#             if col in sub.columns:
#                 row[f"corr_rps_{metric_name}"] = sub[NODE_COLS["rps"]].corr(sub[col])
#                 row[f"corr_replicas_{metric_name}"] = sub["replicas_int"].corr(sub[col])

#         node_corr_rows.append(row)

# node_corr_df = pd.DataFrame(node_corr_rows)
# node_corr_df.to_csv(OUT_DIR / "02_node_correlations_by_service.csv", index=False)

# print("\n=== Node correlation summary ===")
# node_corr_summary = safe_numeric_summary(node_corr_df)
# print(node_corr_summary)
# node_corr_summary.to_csv(OUT_DIR / "02_node_correlations_summary.csv", index=False)


# # ============================================================
# # 3. EDGE CORRELATION
# # ============================================================

# edge_corr_rows = []

# if not edges.empty:
#     possible_src = ["source", "src", "src_service", "source_service", "from_service"]
#     possible_dst = ["destination", "dst", "dst_service", "destination_service", "to_service"]

#     src_col = next((c for c in possible_src if c in edges.columns), None)
#     dst_col = next((c for c in possible_dst if c in edges.columns), None)

#     if src_col is None or dst_col is None:
#         print("⚠ Không tìm thấy cột src/dst trong edges. Sẽ phân tích theo scenario tổng.")
#         group_cols = ["scenario"]
#     else:
#         group_cols = ["scenario", src_col, dst_col]

#     for keys, sub in edges.groupby(group_cols):
#         if len(sub) < 20:
#             continue

#         if not isinstance(keys, tuple):
#             keys = (keys,)

#         row = {"rows": len(sub)}

#         for col_name, key in zip(group_cols, keys):
#             row[col_name] = key

#         edge_rps_col = EDGE_COLS["edge_rps"]

#         for metric_name in ["network_latency", "edge_error", "payload"]:
#             col = EDGE_COLS[metric_name]

#             if col in sub.columns and edge_rps_col in sub.columns:
#                 row[f"corr_edge_rps_{metric_name}"] = sub[edge_rps_col].corr(sub[col])

#         edge_corr_rows.append(row)

# edge_corr_df = pd.DataFrame(edge_corr_rows)
# edge_corr_df.to_csv(OUT_DIR / "03_edge_correlations.csv", index=False)

# print("\n=== Edge correlation summary ===")
# if len(edge_corr_df) > 0:
#     edge_corr_summary = safe_numeric_summary(edge_corr_df)
#     print(edge_corr_summary)
#     edge_corr_summary.to_csv(OUT_DIR / "03_edge_correlations_summary.csv", index=False)
# else:
#     print("No edge correlation rows.")


# # ============================================================
# # 4. FIT CAPACITY EXPONENTS FOR NODE METRICS
# # ============================================================

# def fit_node_capacity_exponent(df, metric_col, min_rows=30):
#     needed = [NODE_COLS["rps"], "replicas_int", metric_col]
#     needed = [c for c in needed if c in df.columns]

#     if len(needed) < 3:
#         return None

#     sub = df[needed].copy()

#     sub = sub[
#         (sub[NODE_COLS["rps"]] > 1e-5) &
#         (sub[metric_col] > 1e-5) &
#         (sub["replicas_int"] >= 1)
#     ].copy()

#     if len(sub) < min_rows:
#         return None

#     X = np.stack([
#         np.log(sub[NODE_COLS["rps"]].values + EPS),
#         np.log(sub["replicas_int"].values + EPS),
#     ], axis=1)

#     y = np.log(sub[metric_col].values + EPS)

#     model = LinearRegression()
#     model.fit(X, y)

#     coef_rps = float(model.coef_[0])
#     coef_rep = float(model.coef_[1])
#     k = -coef_rep
#     r2 = float(model.score(X, y))

#     return {
#         "rows": len(sub),
#         "rps_exponent": coef_rps,
#         "replica_coef_raw": coef_rep,
#         "capacity_exponent_k": k,
#         "r2_log_model": r2,
#         "intercept": float(model.intercept_),
#     }


# fit_rows = []

# for scenario in sorted(nodes["scenario"].unique()):
#     for svc in TARGET_SERVICES:
#         sub = nodes[
#             (nodes["scenario"] == scenario) &
#             (nodes["service"] == svc)
#         ].copy()

#         for metric_name in ["cpu", "lat_p99", "error"]:
#             col = NODE_COLS[metric_name]

#             if col not in sub.columns:
#                 continue

#             res = fit_node_capacity_exponent(sub, col)

#             if res is None:
#                 continue

#             fit_rows.append({
#                 "scenario": scenario,
#                 "service": svc,
#                 "metric": metric_name,
#                 **res,
#             })

# fit_node_df = pd.DataFrame(fit_rows)
# fit_node_df.to_csv(OUT_DIR / "04_node_capacity_exponents.csv", index=False)

# print("\n=== Node capacity exponents ===")
# if len(fit_node_df) > 0:
#     fit_node_summary = safe_group_summary(
#         fit_node_df,
#         group_col="metric",
#         value_cols=["capacity_exponent_k", "rps_exponent", "r2_log_model"]
#     )
#     print(fit_node_summary)
#     fit_node_summary.to_csv(OUT_DIR / "04_node_capacity_exponents_summary.csv", index=False)
# else:
#     print("No fitted node exponents.")


# # ============================================================
# # 5. FIT EDGE BOTTLENECK EXPONENTS
# # ============================================================

# def fit_edge_pressure_model(df, target_col, min_rows=30):
#     needed = [EDGE_COLS["edge_rps"], EDGE_COLS["payload"], target_col]
#     needed = [c for c in needed if c in df.columns]

#     if len(needed) < 3:
#         return None

#     sub = df[needed].copy()

#     sub = sub[
#         (sub[EDGE_COLS["edge_rps"]] > 1e-5) &
#         (sub[EDGE_COLS["payload"]] > 1e-5) &
#         (sub[target_col] > 1e-5)
#     ].copy()

#     if len(sub) < min_rows:
#         return None

#     X = np.stack([
#         np.log(sub[EDGE_COLS["edge_rps"]].values + EPS),
#         np.log(sub[EDGE_COLS["payload"]].values + EPS),
#     ], axis=1)

#     y = np.log(sub[target_col].values + EPS)

#     model = LinearRegression()
#     model.fit(X, y)

#     return {
#         "rows": len(sub),
#         "edge_rps_exponent": float(model.coef_[0]),
#         "payload_exponent": float(model.coef_[1]),
#         "r2_log_model": float(model.score(X, y)),
#         "intercept": float(model.intercept_),
#     }


# edge_fit_rows = []

# if not edges.empty:
#     possible_src = ["source", "src", "src_service", "source_service", "from_service"]
#     possible_dst = ["destination", "dst", "dst_service", "destination_service", "to_service"]

#     src_col = next((c for c in possible_src if c in edges.columns), None)
#     dst_col = next((c for c in possible_dst if c in edges.columns), None)

#     if src_col is not None and dst_col is not None:
#         group_cols = ["scenario", src_col, dst_col]
#     else:
#         group_cols = ["scenario"]

#     for keys, sub in edges.groupby(group_cols):
#         if len(sub) < 20:
#             continue

#         if not isinstance(keys, tuple):
#             keys = (keys,)

#         base_row = {}

#         for col_name, key in zip(group_cols, keys):
#             base_row[col_name] = key

#         for target_name in ["network_latency", "edge_error"]:
#             target_col = EDGE_COLS[target_name]

#             if target_col not in sub.columns:
#                 continue

#             res = fit_edge_pressure_model(sub, target_col)

#             if res is None:
#                 continue

#             edge_fit_rows.append({
#                 **base_row,
#                 "target": target_name,
#                 **res,
#             })

# edge_fit_df = pd.DataFrame(edge_fit_rows)
# edge_fit_df.to_csv(OUT_DIR / "05_edge_pressure_models.csv", index=False)

# print("\n=== Edge pressure model summary ===")
# if len(edge_fit_df) > 0:
#     edge_fit_summary = safe_group_summary(
#         edge_fit_df,
#         group_col="target",
#         value_cols=["edge_rps_exponent", "payload_exponent", "r2_log_model"]
#     )
#     print(edge_fit_summary)
#     edge_fit_summary.to_csv(OUT_DIR / "05_edge_pressure_models_summary.csv", index=False)
# else:
#     print("No fitted edge pressure models.")


# # ============================================================
# # 6. RECOMMENDED CAPACITY MODEL PARAMETERS
# # ============================================================

# def safe_median(df, metric):
#     if df is None or len(df) == 0:
#         return None

#     needed_cols = ["metric", "r2_log_model", "capacity_exponent_k"]

#     for c in needed_cols:
#         if c not in df.columns:
#             return None

#     sub = df[
#         (df["metric"] == metric) &
#         (df["r2_log_model"] > 0.05) &
#         (df["capacity_exponent_k"] > -1.0) &
#         (df["capacity_exponent_k"] < 2.0)
#     ].copy()

#     if len(sub) == 0:
#         return None

#     vals = pd.to_numeric(sub["capacity_exponent_k"], errors="coerce")
#     vals = vals.replace([np.inf, -np.inf], np.nan).dropna().values

#     if len(vals) == 0:
#         return None

#     return float(np.median(vals))


# cpu_alpha = safe_median(fit_node_df, "cpu")
# lat_beta = safe_median(fit_node_df, "lat_p99")
# err_gamma = safe_median(fit_node_df, "error")

# recommended = {
#     "cpu_alpha_from_data": cpu_alpha,
#     "lat_beta_from_data": lat_beta,
#     "err_gamma_from_data": err_gamma,

#     "cpu_alpha_fallback": 0.85,
#     "lat_beta_fallback": 0.60,
#     "err_gamma_fallback": 0.50,

#     "notes": (
#         "Use *_from_data only if signs are reasonable and r2_log_model is meaningful. "
#         "Otherwise use fallback queueing-inspired exponents."
#     )
# }

# with open(OUT_DIR / "06_recommended_capacity_params.json", "w", encoding="utf-8") as f:
#     json.dump(recommended, f, indent=4, ensure_ascii=False)

# print("\n=== Recommended capacity params ===")
# print(json.dumps(recommended, indent=4))


# # ============================================================
# # 7. PLOTS
# # ============================================================

# def save_scatter_by_service(metric_x, metric_y, filename, max_points=3000):
#     x_col = NODE_COLS[metric_x]
#     y_col = NODE_COLS[metric_y]

#     if x_col not in nodes.columns or y_col not in nodes.columns:
#         print(f"Skip plot {metric_x} vs {metric_y}: missing columns.")
#         return

#     plot_df = nodes[["scenario", "service", x_col, y_col]].copy()
#     plot_df = plot_df.dropna()

#     if len(plot_df) == 0:
#         print(f"Skip plot {metric_x} vs {metric_y}: empty data.")
#         return

#     if len(plot_df) > max_points:
#         plot_df = plot_df.sample(max_points, random_state=42)

#     plt.figure(figsize=(8, 6))

#     for scenario in sorted(plot_df["scenario"].unique()):
#         sub = plot_df[plot_df["scenario"] == scenario]
#         plt.scatter(sub[x_col], sub[y_col], s=8, alpha=0.35, label=scenario)

#     plt.xlabel(metric_x)
#     plt.ylabel(metric_y)
#     plt.title(f"{metric_x} vs {metric_y}")
#     plt.legend()
#     plt.grid(True, alpha=0.3)
#     plt.tight_layout()

#     path = OUT_DIR / filename
#     plt.savefig(path, dpi=150)
#     plt.show()
#     print(f"Saved plot: {path}")


# save_scatter_by_service("rps", "cpu", "plot_rps_vs_cpu.png")
# save_scatter_by_service("rps", "lat_p99", "plot_rps_vs_lat_p99.png")
# save_scatter_by_service("rps", "error", "plot_rps_vs_error.png")


# for metric in ["cpu", "lat_p99", "error"]:
#     col = NODE_COLS[metric]

#     if col not in nodes.columns:
#         print(f"Skip replicas vs {metric}: missing column.")
#         continue

#     agg = (
#         nodes
#         .groupby(["scenario", "replicas_int"])[col]
#         .agg(["mean", "median", "count"])
#         .reset_index()
#     )

#     agg.to_csv(OUT_DIR / f"replicas_vs_{metric}.csv", index=False)

#     plt.figure(figsize=(8, 5))

#     for scenario in sorted(agg["scenario"].unique()):
#         sub = agg[agg["scenario"] == scenario]
#         plt.plot(sub["replicas_int"], sub["mean"], marker="o", label=scenario)

#     plt.xlabel("replicas")
#     plt.ylabel(metric)
#     plt.title(f"Replicas vs {metric}")
#     plt.grid(True, alpha=0.3)
#     plt.legend()
#     plt.tight_layout()

#     path = OUT_DIR / f"plot_replicas_vs_{metric}.png"
#     plt.savefig(path, dpi=150)
#     plt.show()
#     print(f"Saved plot: {path}")


# if not edges.empty:
#     def save_edge_scatter(metric_x, metric_y, filename, max_points=3000):
#         x_col = EDGE_COLS[metric_x]
#         y_col = EDGE_COLS[metric_y]

#         if x_col not in edges.columns or y_col not in edges.columns:
#             print(f"Skip edge plot {metric_x} vs {metric_y}: missing columns.")
#             return

#         plot_df = edges[["scenario", x_col, y_col]].copy().dropna()

#         if len(plot_df) == 0:
#             print(f"Skip edge plot {metric_x} vs {metric_y}: empty data.")
#             return

#         if len(plot_df) > max_points:
#             plot_df = plot_df.sample(max_points, random_state=42)

#         plt.figure(figsize=(8, 6))

#         for scenario in sorted(plot_df["scenario"].unique()):
#             sub = plot_df[plot_df["scenario"] == scenario]
#             plt.scatter(sub[x_col], sub[y_col], s=8, alpha=0.35, label=scenario)

#         plt.xlabel(metric_x)
#         plt.ylabel(metric_y)
#         plt.title(f"{metric_x} vs {metric_y}")
#         plt.legend()
#         plt.grid(True, alpha=0.3)
#         plt.tight_layout()

#         path = OUT_DIR / filename
#         plt.savefig(path, dpi=150)
#         plt.show()
#         print(f"Saved plot: {path}")

#     save_edge_scatter("edge_rps", "network_latency", "plot_edge_rps_vs_network_latency.png")
#     save_edge_scatter("edge_rps", "edge_error", "plot_edge_rps_vs_edge_error.png")
#     save_edge_scatter("payload", "network_latency", "plot_payload_vs_network_latency.png")


# # ============================================================
# # 8. FINAL INDEX
# # ============================================================

# print("\n" + "=" * 80)
# print("HOÀN TẤT PHÂN TÍCH")
# print("=" * 80)
# print(f"Output folder: {OUT_DIR.resolve()}")

# print("\nCác file quan trọng cần gửi lại:")
# print("01_metric_distribution_by_scenario.csv")
# print("02_node_correlations_by_service.csv")
# print("02_node_correlations_summary.csv")
# print("03_edge_correlations.csv")
# print("03_edge_correlations_summary.csv")
# print("04_node_capacity_exponents.csv")
# print("04_node_capacity_exponents_summary.csv")
# print("05_edge_pressure_models.csv")
# print("05_edge_pressure_models_summary.csv")
# print("06_recommended_capacity_params.json")
# print("plot_rps_vs_cpu.png")
# print("plot_rps_vs_lat_p99.png")
# print("plot_rps_vs_error.png")
# print("plot_replicas_vs_cpu.png")
# print("plot_replicas_vs_lat_p99.png")
# print("plot_replicas_vs_error.png")
# print("plot_edge_rps_vs_network_latency.png")
# print("plot_edge_rps_vs_edge_error.png")
# print("plot_payload_vs_network_latency.png")

In [ ]:
# pip install --upgrade pandas
